# Audited revision (v5)

Original files are preserved. All saved outputs in this revised notebook are cleared so old results cannot be mistaken for rerun results. The original full notebook and its recorded outputs remain the provenance source.

Fixes include explicit saved-model keys, consistency checks against stored errors, exact calibration bounds, correct residual-score documentation, fail-closed boundary sample selection, and corrected statistical interpretations. The full notebook retains the explicit seed 2026 for the original development split.

The accompanying reproduction package distinguishes reconstructed model artifacts from the unavailable original time-stamped model file. New resolution and mechanism diagnostics are exploratory. Run the original-data cells in Colab with the same source files; never substitute a new model for the original frozen evaluation.


# Complete HIT / 3DMSL / MitoEM-R notebook with frozen-shard validation

Includes the original audit, QC, representation agreement, repeatability, internal correction validation, imaging analysis, MitoEM-R analysis, and frozen-model validation on `1_1_2729.zip`.

**For your current work: open in Colab and Run all with the defaults below.** Sections 1–18 are included for reproducibility but skipped because you already completed them. Section 19 loads your existing model from `MyDrive/HIT_outputs_verified` and evaluates the new ZIP without training or recalibrating. Results go into a separate dated folder.

To rerun the complete development pipeline, set `RUN_ORIGINAL_PIPELINE = True`. Development reruns write to `HIT_outputs_reproduced`, leaving your existing verified model untouched. The final frozen evaluation still uses the existing `HIT_outputs_verified` model.

Archives are expected in `MyDrive`: `10_24553_27272.zip`, `1_1_2729.zip`, `im.zip`, and `EM30-R-mito-train-val-v2.zip`. Only the new shard is required with the default mode, along with your saved original model, splits, repeatability table, and audit scripts.

This consolidated file uses the locally available metadata-root-fixed pipeline plus the new frozen-validation section. Saved outputs are cleared to avoid presenting old results as a new run. Within-dataset replication does not establish independent biological replication.


In [ ]:
RUN_ORIGINAL_PIPELINE = False
print("Mode:", "Rerun development pipeline, then evaluate existing frozen model" if RUN_ORIGINAL_PIPELINE else "Evaluate existing frozen model on new shard only")


## 1. Mount Drive and set paths

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from google.colab import drive
    from pathlib import Path
    import os

    drive.mount('/content/drive')

    ZIP_PATH = '/content/drive/MyDrive/10_24553_27272.zip'
    WORK_DIR = '/content/hit'
    DATA_DIR = '/content/data'
    OUT_DIR = '/content/hit_outputs_reproduced'
    DRIVE_OUT = '/content/drive/MyDrive/HIT_outputs_reproduced'

    for folder in (WORK_DIR, DATA_DIR, OUT_DIR):
        os.makedirs(folder, exist_ok=True)
    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(f'Missing 3DMSL archive: {ZIP_PATH}')
    print('3DMSL ZIP size: %.2f GB' % (os.path.getsize(ZIP_PATH) / 1e9))
    print('Verified results will be written to:', DRIVE_OUT)
    get_ipython().system('df -h /content | tail -1')


## 2. Install dependencies

Colab usually includes NumPy, pandas, Matplotlib, and Pillow. Install the geometry libraries used by the 3DMSL and MitoEM sections.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    get_ipython().system('pip install -q trimesh scikit-image scipy')


## 3. Locate or extract the real 3DMSL shard

The ZIP includes a `__MACOSX` metadata tree. The code below identifies the real shard from `raw_mesh.off` archive entries and excludes Mac metadata. It checks whether all expected real meshes and occupancy archives are already extracted before doing any work.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import time, zipfile
    from pathlib import Path

    base = Path(DATA_DIR)
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH) as archive:
        members = archive.infolist()
        real_mesh_paths = [m.filename for m in members
                           if m.filename.endswith('/raw_mesh.off')
                           and not m.filename.startswith('__MACOSX/')
                           and not Path(m.filename).name.startswith('._')]
        if not real_mesh_paths:
            raise ValueError('No real raw_mesh.off files were found in the ZIP')
        shard_names = {Path(name).parts[0] for name in real_mesh_paths}
        if len(shard_names) != 1:
            raise ValueError(f'Expected one real shard folder; found {sorted(shard_names)}')
        shard_name = shard_names.pop()
        shard_dir = base / shard_name
        expected_instances = len(real_mesh_paths)
        mesh_count = len(list(shard_dir.glob('*/raw_mesh.off')))
        occ_count = len(list(shard_dir.glob('*/occupancies.npz')))
        if mesh_count == expected_instances and occ_count == expected_instances:
            print(f'Reusing complete real extraction: {shard_dir}')
        else:
            print(f'Extracting {expected_instances} real instances; skipping __MACOSX metadata')
            clean_members = []
            for member in members:
                parts = Path(member.filename).parts
                if member.filename.startswith('/') or '..' in parts:
                    raise ValueError(f'Unsafe ZIP member path: {member.filename}')
                if '__MACOSX' in parts or any(p.startswith('._') or p == '.DS_Store' for p in parts):
                    continue
                clean_members.append(member)
            archive.extractall(base, members=clean_members)

    SHARD_ROOT = str(shard_dir)
    real_meshes = list(shard_dir.glob('*/raw_mesh.off'))
    real_occupancies = list(shard_dir.glob('*/occupancies.npz'))
    if len(real_meshes) != expected_instances or len(real_occupancies) != expected_instances:
        raise RuntimeError(f'Incomplete real shard: {len(real_meshes)}/{expected_instances} meshes and '
                           f'{len(real_occupancies)}/{expected_instances} occupancy archives')
    if '__MACOSX' in Path(SHARD_ROOT).parts:
        raise RuntimeError('Refusing to analyze Mac ZIP metadata')
    print('SHARD_ROOT =', SHARD_ROOT)
    print('Verified real instances:', expected_instances)
    print('Setup completed in %.1f min' % ((time.time() - t0) / 60))


## 4. Verify one real 3DMSL instance

The check below requires actual mesh, point-cloud, and occupancy data in the selected folder. If any item is missing, stop here and repair the extraction before generating results.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import glob, numpy as np
    from pathlib import Path

    instance_dirs = sorted(p for p in Path(SHARD_ROOT).iterdir() if p.is_dir())
    if len(instance_dirs) != expected_instances:
        raise RuntimeError(f'Expected {expected_instances} real instance folders, found {len(instance_dirs)}')
    example = instance_dirs[0]
    for name in ('raw_mesh.off', 'pointcloud.ply', 'occupancies.npz'):
        path = example / name
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f'Real 3DMSL file missing or empty: {path}')
    print('Verified example instance:', example)
    print('Instance folders:', len(instance_dirs))
    print('Mesh, point cloud, and occupancy files are present.')
    with np.load(example / 'occupancies.npz') as archive:
        print('Occupancy keys:', archive.files)
        for key in archive.files:
            value = archive[key]
            print(f'  {key}: shape={getattr(value, "shape", None)}, dtype={value.dtype}')


## 5. Write the original 3DMSL pipeline scripts

The following `%%writefile` cells create the loaders, audit, QC, feature extraction, agreement, and descriptive-analysis scripts in the Colab runtime.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    os.chdir(WORK_DIR)
    print('cwd:', os.getcwd())


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('hit_common.py').write_text('"""\nhit_common.py — shared loaders and descriptor math for the 3DMSL reliability pipeline.\n\nCoordinate-frame convention (verified in the Week 1-2 audit):\n    raw_mesh.off       -> REAL units\n    pointcloud.ply     -> NORMALIZED frame\n    occupancies.npz    -> query points in NORMALIZED frame\n    normalized = (raw - loc) / scale      (loc, scale stored per-instance in the npz)\nso anything from the point cloud or the occupancy grid must be un-normalized\n(raw = normalized * scale + loc) before it is compared with the mesh.\n"""\nimport os\nimport glob\nimport numpy as np\n\nIMG_CONFIGS = ("Conf1", "Conf2", "Epi1")\nEXPECTED_VIEWS = 6\n\n# -----------------------------------------------------------------------------\n# instance discovery\n# -----------------------------------------------------------------------------\n\ndef find_instance_dirs(root):\n    """Any directory that directly contains a .off mesh is an instance directory."""\n    out = []\n    for dirpath, dirnames, filenames in os.walk(root):\n        # Mac-created ZIPs include an __MACOSX tree containing AppleDouble\n        # `._raw_mesh.off` metadata. These files are not 3DMSL meshes.\n        if "__MACOSX" in os.path.normpath(dirpath).split(os.sep):\n            dirnames[:] = []\n            continue\n        dirnames[:] = [name for name in dirnames if name != "__MACOSX" and not name.startswith("._")]\n        if any(f.lower().endswith(".off") and not f.startswith("._") for f in filenames):\n            out.append(dirpath)\n            dirnames[:] = []          # do not descend into image/render subfolders\n    return sorted(out, key=lambda p: _id_key(os.path.basename(p)))\n\n\ndef _id_key(name):\n    return (0, int(name)) if name.isdigit() else (1, name)\n\n\ndef instance_id(d):\n    return os.path.basename(os.path.normpath(d))\n\n\ndef _first(d, patterns):\n    for pat in patterns:\n        # Search recursively within directory d for the pattern\n        hits = sorted(glob.glob(os.path.join(d, \'**\', pat), recursive=True))\n        if hits:\n            return hits[0]\n    return None\n\n# -----------------------------------------------------------------------------\n# loaders\n# -----------------------------------------------------------------------------\n\ndef load_mesh(d):\n    import trimesh\n    p = _first(d, ("raw_mesh.off", "model.off", "mesh.off", "*.off"))\n    if p is None:\n        raise FileNotFoundError("no .off mesh in %s" % d)\n    m = trimesh.load(p, process=False, force="mesh")\n    if m is None or len(getattr(m, "faces", [])) == 0:\n        raise ValueError("empty mesh: %s" % p)\n    return m\n\n\ndef load_pointcloud(d):\n    """Returns (N,3) points in whatever frame the file stores (normalized for 3DMSL)."""\n    p = _first(d, ("pointcloud.ply", "*.ply"))\n    if p is not None:\n        import trimesh\n        pc = trimesh.load(p, process=False)\n        pts = np.asarray(pc.vertices, dtype=np.float64)\n        if pts.size:\n            return pts, p\n    p = _first(d, ("pointcloud.npz", "points.npz"))\n    if p is not None:\n        z = np.load(p)\n        for k in ("points", "point", "pointcloud"):\n            if k in z:\n                return np.asarray(z[k], dtype=np.float64), p\n    raise FileNotFoundError("no point cloud in %s" % d)\n\n\ndef load_occupancy(d):\n    """Returns (points, occ_bool, loc, scale). Points stay in the normalized frame."""\n    p = _first(d, ("occupancies.npz", "points.npz", "occupancy.npz", "*.npz"))\n    if p is None:\n        raise FileNotFoundError("no occupancy npz in %s" % d)\n    z = np.load(p)\n    keys = set(z.files)\n    pkey = next((k for k in ("points", "point", "query_points") if k in keys), None)\n    okey = next((k for k in ("occupancies", "occupancy", "occ", "labels") if k in keys), None)\n    if pkey is None or okey is None:\n        raise ValueError("npz %s lacks points/occupancies (has %s)" % (p, sorted(keys)))\n    pts = np.asarray(z[pkey], dtype=np.float64)\n    occ = np.asarray(z[okey])\n    if occ.dtype == np.uint8 and occ.size != pts.shape[0]:\n        occ = np.unpackbits(occ)[: pts.shape[0]]          # Occupancy-Networks bit packing\n    occ = occ.astype(bool).reshape(-1)[: pts.shape[0]]\n    loc = np.asarray(z["loc"], dtype=np.float64).reshape(-1) if "loc" in keys else np.zeros(3)\n    scale = np.asarray(z["scale"], dtype=np.float64).reshape(-1) if "scale" in keys else np.ones(1)\n    if loc.size == 1:\n        loc = np.repeat(loc, 3)\n    return pts, occ, loc, scale\n\n\ndef unnormalize(pts, loc, scale):\n    """normalized -> raw. scale may be a scalar or a per-axis vector."""\n    s = np.asarray(scale, dtype=np.float64).reshape(-1)\n    s = np.repeat(s, 3) if s.size == 1 else s[:3]\n    return pts * s[None, :] + np.asarray(loc, dtype=np.float64).reshape(1, 3)\n\n\ndef count_images(d):\n    """Per-configuration render counts. Config name is matched anywhere in the path."""\n    files = [f for ext in ("png", "jpg", "jpeg", "tif", "tiff")\n             for f in glob.glob(os.path.join(d, "**", "*." + ext), recursive=True)]\n    counts = {c: 0 for c in IMG_CONFIGS}\n    for f in files:\n        rel = os.path.relpath(f, d).lower()\n        for c in IMG_CONFIGS:\n            if c.lower() in rel:\n                counts[c] += 1\n                break\n    return counts, files\n\n\ndef images_for_config(d, config):\n    files = [f for ext in ("png", "jpg", "jpeg", "tif", "tiff")\n             for f in glob.glob(os.path.join(d, "**", "*." + ext), recursive=True)]\n    return sorted(f for f in files\n                  if config.lower() in os.path.relpath(f, d).lower())\n\n# -----------------------------------------------------------------------------\n# descriptor math (pure functions - these are what tests.py exercises)\n# -----------------------------------------------------------------------------\n\ndef sphericity(volume, area):\n    """Wadell sphericity: 1.0 for a perfect sphere, < 1.0 otherwise."""\n    if area <= 0 or volume <= 0:\n        return np.nan\n    return (np.pi ** (1.0 / 3.0)) * ((6.0 * volume) ** (2.0 / 3.0)) / area\n\n\ndef raw_extent(pts):\n    pts = np.asarray(pts, dtype=np.float64)\n    return pts.max(axis=0) - pts.min(axis=0)\n\n\ndef trimmed_extent(pts, lo=1.0, hi=99.0):\n    """Percentile-trimmed bounding-box extent - robust to point-cloud sampling noise."""\n    pts = np.asarray(pts, dtype=np.float64)\n    return np.percentile(pts, hi, axis=0) - np.percentile(pts, lo, axis=0)\n\n\ndef elongation(extent):\n    e = np.asarray(extent, dtype=np.float64)\n    return float(e.max() / e.min()) if e.min() > 0 else np.nan\n\n\ndef otsu_threshold(img):\n    """Otsu\'s method on a uint8 image. Returns the threshold level (0-255)."""\n    img = np.asarray(img)\n    hist = np.bincount(img.reshape(-1).astype(np.uint8), minlength=256).astype(np.float64)\n    total = hist.sum()\n    if total == 0:\n        return 0.0\n    p = hist / total\n    levels = np.arange(256, dtype=np.float64)\n    omega = np.cumsum(p)\n    mu = np.cumsum(p * levels)\n    mu_t = mu[-1]\n    denom = omega * (1.0 - omega)\n    with np.errstate(divide="ignore", invalid="ignore"):\n        sigma_b = np.where(denom > 0, (mu_t * omega - mu) ** 2 / denom, 0.0)\n    return float(np.argmax(sigma_b))\n\n\ndef pct_diff(a, b):\n    """Percent difference of a relative to b."""\n    return np.nan if b == 0 else 100.0 * (a - b) / b')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('audit_3dmsl.py').write_text('#!/usr/bin/env python3\n"""\naudit_3dmsl.py - Weeks 1-2 milestone: load every 3DMSL instance in a shard across all\nfour representations (mesh / point cloud / occupancy / rendered images) and write one\nrow of structural descriptors per instance.\n\n  python3 audit_3dmsl.py --root /path/to/10_24553_27272 --out audit_full_descriptors.csv\n  python3 audit_3dmsl.py --root ... --sample 15 --out audit_sample.csv\n  python3 audit_3dmsl.py --root ... --start 24553 --end 24852 --workers 4\n"""\nimport argparse\nimport os\nimport sys\nimport time\nimport traceback\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\n\nimport numpy as np\nimport pandas as pd\n\nfrom hit_common import (IMG_CONFIGS, count_images, elongation, find_instance_dirs,\n                        instance_id, load_mesh, load_occupancy, load_pointcloud,\n                        pct_diff, raw_extent, sphericity, trimmed_extent, unnormalize)\n\n\ndef audit_instance(d):\n    row = {"instance_id": instance_id(d), "path": d,\n           "mesh_ok": False, "pc_ok": False, "occ_ok": False, "error": ""}\n    loc = np.zeros(3)\n    scale = np.ones(1)\n\n    # ---- mesh (real units, ground truth) ------------------------------------\n    try:\n        m = load_mesh(d)\n        ext = np.asarray(m.extents, dtype=np.float64)\n        vol = float(abs(m.volume))\n        area = float(m.area)\n        row.update(mesh_ok=True,\n                   mesh_watertight=bool(m.is_watertight),\n                   mesh_n_vertices=int(len(m.vertices)),\n                   mesh_n_faces=int(len(m.faces)),\n                   mesh_volume=vol,\n                   mesh_area=area,\n                   mesh_sphericity=sphericity(vol, area),\n                   mesh_extent_x=ext[0], mesh_extent_y=ext[1], mesh_extent_z=ext[2],\n                   mesh_elongation=elongation(ext))\n        mesh_center = np.asarray(m.bounds, dtype=np.float64).mean(axis=0)\n        mesh_extent = ext\n    except Exception as e:\n        row["error"] += "mesh:%s; " % e\n        mesh_center, mesh_extent = None, None\n\n    # ---- occupancy (normalized frame + the loc/scale everything else needs) --\n    try:\n        pts, occ, loc, scale = load_occupancy(d)\n        pts_raw = unnormalize(pts, loc, scale)\n        frac = float(occ.mean())\n        qbox = raw_extent(pts_raw)\n        vol_est = frac * float(np.prod(qbox))\n        inside = pts_raw[occ]\n        occ_ext = raw_extent(inside) if len(inside) >= 2 else np.full(3, np.nan)\n        s = np.asarray(scale).reshape(-1)\n        row.update(occ_ok=True,\n                   occ_n_points=int(pts.shape[0]),\n                   occ_fraction_inside=frac,\n                   occ_volume_est=vol_est,\n                   occ_extent_x=occ_ext[0], occ_extent_y=occ_ext[1], occ_extent_z=occ_ext[2],\n                   occ_loc_x=loc[0], occ_loc_y=loc[1], occ_loc_z=loc[2],\n                   occ_scale=float(s[0]) if s.size == 1 else float(np.max(s)))\n        # the coordinate-frame check: loc/scale should reproduce the mesh\'s own bbox\n        if mesh_center is not None:\n            row["loc_vs_mesh_center_err"] = float(np.max(np.abs(loc - mesh_center)))\n            s_full = np.repeat(s, 3) if s.size == 1 else s[:3]\n            row["scale_vs_mesh_extent_err"] = float(\n                np.min(np.abs(s_full - mesh_extent)) / max(mesh_extent.max(), 1e-12))\n    except Exception as e:\n        row["error"] += "occ:%s; " % e\n\n    # ---- point cloud (normalized frame -> un-normalize before comparing) -----\n    try:\n        pc, _ = load_pointcloud(d)\n        pc_raw = unnormalize(pc, loc, scale)\n        ext_raw = raw_extent(pc_raw)\n        ext_trim = trimmed_extent(pc_raw)\n        row.update(pc_ok=True,\n                   pc_n_points=int(pc.shape[0]),\n                   pc_extent_x=ext_raw[0], pc_extent_y=ext_raw[1], pc_extent_z=ext_raw[2],\n                   pc_trimmed_extent_x=ext_trim[0],\n                   pc_trimmed_extent_y=ext_trim[1],\n                   pc_trimmed_extent_z=ext_trim[2])\n        if mesh_extent is not None:\n            row["pc_extent_pct_diff_max"] = float(np.max(np.abs(\n                [pct_diff(ext_raw[i], mesh_extent[i]) for i in range(3)])))\n            row["pc_trimmed_extent_pct_diff_max"] = float(np.max(np.abs(\n                [pct_diff(ext_trim[i], mesh_extent[i]) for i in range(3)])))\n    except Exception as e:\n        row["error"] += "pc:%s; " % e\n\n    # ---- rendered images (presence only; features come from image_features.py)\n    try:\n        counts, files = count_images(d)\n        for c in IMG_CONFIGS:\n            row["n_images_" + c] = counts[c]\n        row["n_images_total"] = len(files)\n    except Exception as e:\n        row["error"] += "img:%s; " % e\n    return row\n\n\ndef _safe(d):\n    try:\n        return audit_instance(d)\n    except Exception:\n        return {"instance_id": instance_id(d), "path": d, "mesh_ok": False,\n                "pc_ok": False, "occ_ok": False, "error": traceback.format_exc(limit=1)}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="3DMSL structural audit")\n    ap.add_argument("--root", required=True, help="unzipped shard directory")\n    ap.add_argument("--out", default="audit_full_descriptors.csv")\n    ap.add_argument("--sample", type=int, default=0, help="audit only the first N instances")\n    ap.add_argument("--start", type=int, default=None, help="numeric instance id lower bound")\n    ap.add_argument("--end", type=int, default=None, help="numeric instance id upper bound")\n    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1))\n    ap.add_argument("--progress-every", type=int, default=100)\n    ap.add_argument("--resume", action="store_true",\n                    help="skip instances already present in --out and append (survives a "\n                         "Colab disconnect)")\n    args = ap.parse_args()\n\n    dirs = find_instance_dirs(args.root)\n    if args.start is not None or args.end is not None:\n        lo = args.start if args.start is not None else -10 ** 18\n        hi = args.end if args.end is not None else 10 ** 18\n        dirs = [d for d in dirs if instance_id(d).isdigit() and lo <= int(instance_id(d)) <= hi]\n    if args.sample:\n        dirs = dirs[: args.sample]\n    done = set()\n    if args.resume and os.path.exists(args.out):\n        done = set(pd.read_csv(args.out, usecols=["instance_id"])["instance_id"].astype(str))\n        dirs = [d for d in dirs if instance_id(d) not in done]\n        print("Resuming: %d already done, %d remaining" % (len(done), len(dirs)))\n        if not dirs:\n            print("Nothing to do.")\n            return\n    if not dirs:\n        sys.exit("No instance directories found under %s" % args.root)\n    print("Auditing %d instances with %d worker(s)" % (len(dirs), args.workers), flush=True)\n\n    t0 = time.time()\n    rows = []\n    if args.workers <= 1:\n        for i, d in enumerate(dirs, 1):\n            rows.append(_safe(d))\n            if i % args.progress_every == 0 or i == len(dirs):\n                print("  %d/%d  (%.2fs/instance)" % (i, len(dirs), (time.time() - t0) / i), flush=True)\n    else:\n        with ProcessPoolExecutor(max_workers=args.workers) as ex:\n            futs = {ex.submit(_safe, d): d for d in dirs}\n            for i, f in enumerate(as_completed(futs), 1):\n                rows.append(f.result())\n                if i % args.progress_every == 0 or i == len(dirs):\n                    print("  %d/%d  (%.2fs/instance)" % (i, len(dirs), (time.time() - t0) / i), flush=True)\n\n    df = pd.DataFrame(rows)\n    if done:\n        df = pd.concat([pd.read_csv(args.out), df], ignore_index=True)\n    df = df.sort_values("instance_id", key=lambda s: s.astype(str).str.zfill(12))\n    df.to_csv(args.out, index=False)\n\n    n = len(df)\n    print("\\n=== AUDIT SUMMARY (n=%d, %.1fs total) ===" % (n, time.time() - t0))\n    for k in ("mesh_ok", "pc_ok", "occ_ok"):\n        print("  %-9s %d/%d (%.1f%%)" % (k, int(df[k].sum()), n, 100.0 * df[k].mean()))\n    if "mesh_watertight" in df:\n        print("  watertight %d/%d" % (int(df["mesh_watertight"].fillna(False).sum()), n))\n    for c in IMG_CONFIGS:\n        col = "n_images_" + c\n        if col in df:\n            bad = df[df[col] < 6]\n            print("  %s renders: %d instance(s) with <6 views %s"\n                  % (c, len(bad), list(bad["instance_id"])[:10]))\n    if "occ_fraction_inside" in df:\n        print("  occ_fraction_inside < 2%%: %d" % int((df["occ_fraction_inside"] < 0.02).sum()))\n    if "mesh_volume" in df and "occ_volume_est" in df:\n        err = 100.0 * (df["occ_volume_est"] - df["mesh_volume"]).abs() / df["mesh_volume"]\n        print("  occupancy volume error vs mesh: mean %.2f%% / median %.2f%%"\n              % (err.mean(), err.median()))\n    if "loc_vs_mesh_center_err" in df:\n        print("  coordinate-frame check: max |loc - mesh bbox center| = %.3g"\n              % df["loc_vs_mesh_center_err"].max())\n    print("\\nWrote %s" % os.path.abspath(args.out))\n\n\nif __name__ == "__main__":\n    main()')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('qc_filter.py').write_text('#!/usr/bin/env python3\n"""\nqc_filter.py - Weeks 3-4 milestone: apply the documented two-tier QC policy to the\naudit table and emit a QC\'d descriptor table plus a human-readable QC report.\n\n  HARD EXCLUDE  mesh, point cloud or occupancy failed to load -> instance unusable\n  FLAG (kept)   not watertight\n                occ_fraction_inside < 2%            (Monte-Carlo volume unreliable)\n                point-cloud raw extent > 10% off the mesh  (sampling-noise outlier)\n                fewer than 6 renders in any image configuration\n\nEdit the THRESHOLDS dict below to change the policy; the report restates whatever\nvalues were actually used, so it never goes stale.\n\n  python3 qc_filter.py --in audit_full_descriptors.csv --out audit_descriptors_qc.csv\n"""\nimport argparse\nimport os\n\nimport numpy as np\nimport pandas as pd\n\nfrom hit_common import IMG_CONFIGS, EXPECTED_VIEWS\n\nTHRESHOLDS = {\n    "min_occ_fraction_inside": 0.02,   # below this, occupancy volume error ~11% not ~3.5%\n    "max_pc_extent_pct_diff": 10.0,    # raw point-cloud extent vs mesh extent\n    "expected_views_per_config": EXPECTED_VIEWS,\n}\n\n\ndef apply_qc(df, th=THRESHOLDS):\n    df = df.copy()\n    for c in ("mesh_ok", "pc_ok", "occ_ok"):\n        if c not in df:\n            df[c] = False\n        df[c] = df[c].fillna(False).astype(bool)\n\n    df["qc_exclude"] = ~(df["mesh_ok"] & df["pc_ok"] & df["occ_ok"])\n    df["qc_exclude_reason"] = np.where(\n        df["qc_exclude"],\n        df.apply(lambda r: ",".join(\n            [k for k, v in (("mesh_load_failed", r["mesh_ok"]),\n                            ("pointcloud_load_failed", r["pc_ok"]),\n                            ("occupancy_load_failed", r["occ_ok"])) if not v]), axis=1),\n        "")\n\n    flags = [[] for _ in range(len(df))]\n\n    def add(mask, name):\n        for i in np.flatnonzero(np.asarray(mask.fillna(False))):\n            flags[i].append(name)\n\n    if "mesh_watertight" in df:\n        add(~df["mesh_watertight"].fillna(False).astype(bool) & df["mesh_ok"], "not_watertight")\n    if "occ_fraction_inside" in df:\n        add(df["occ_fraction_inside"] < th["min_occ_fraction_inside"], "low_occupancy_sampling")\n    if "pc_extent_pct_diff_max" in df:\n        add(df["pc_extent_pct_diff_max"].abs() > th["max_pc_extent_pct_diff"], "pc_extent_outlier")\n    img_cols = [c for c in ("n_images_" + c for c in IMG_CONFIGS) if c in df]\n    if img_cols:\n        add((df[img_cols] < th["expected_views_per_config"]).any(axis=1), "incomplete_images")\n\n    df["qc_flags"] = [";".join(f) for f in flags]\n    df["qc_n_flags"] = [len(f) for f in flags]\n    df["qc_pass"] = ~df["qc_exclude"]\n    return df\n\n\ndef write_report(df, path, th=THRESHOLDS, src=""):\n    n = len(df)\n    kept = int(df["qc_pass"].sum())\n    lines = ["# QC Report (Weeks 3-4)", ""]\n    if src:\n        lines += ["Input: `%s`" % src, ""]\n    lines += ["Instances audited: **%d**" % n,\n              "", "## Policy applied", "",\n              "| Tier | Rule | Effect |", "|---|---|---|",\n              "| HARD EXCLUDE | mesh / point cloud / occupancy failed to load | row dropped from analysis |",\n              "| FLAG | mesh not watertight | kept, caveat |",\n              "| FLAG | `occ_fraction_inside` < %g | kept, caveat |" % th["min_occ_fraction_inside"],\n              "| FLAG | raw point-cloud extent > %g%% from mesh extent | kept, use trimmed extent |" % th["max_pc_extent_pct_diff"],\n              "| FLAG | fewer than %d renders in a configuration | kept, image features partial |" % th["expected_views_per_config"],\n              "", "## Outcome", "",\n              "- Hard exclusions: **%d**" % (n - kept),\n              "- Retained for analysis: **%d** (%.1f%%)" % (kept, 100.0 * kept / max(n, 1)), ""]\n    if n - kept:\n        lines.append("### Excluded instances")\n        lines.append("")\n        for _, r in df[df["qc_exclude"]].iterrows():\n            lines.append("- `%s` - %s" % (r["instance_id"], r["qc_exclude_reason"]))\n        lines.append("")\n\n    lines += ["### Flag counts", "", "| Flag | n | % of retained |", "|---|---|---|"]\n    kept_df = df[df["qc_pass"]]\n    allflags = sorted({f for s in kept_df["qc_flags"] for f in s.split(";") if f})\n    for f in allflags:\n        c = int(kept_df["qc_flags"].str.split(";").apply(lambda L: f in L).sum())\n        lines.append("| `%s` | %d | %.2f%% |" % (f, c, 100.0 * c / max(kept, 1)))\n    if not allflags:\n        lines.append("| (none) | 0 | 0.00%% |")\n    lines += ["", "Flagged instances are **kept**. Flags travel with the row into",\n              "`paired_feature_table.csv` so downstream analysis can condition on them",\n              "(e.g. the sphericity-conditioned equivalence margin uses",\n              "`low_occupancy_sampling`).", ""]\n    with open(path, "w") as fh:\n        fh.write("\\n".join(lines))\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="QC filter for the 3DMSL audit table")\n    ap.add_argument("--in", dest="inp", default="audit_full_descriptors.csv")\n    ap.add_argument("--out", default="audit_descriptors_qc.csv")\n    ap.add_argument("--report", default="qc_report.md")\n    ap.add_argument("--keep-excluded", action="store_true",\n                    help="write excluded rows too (marked qc_pass=False)")\n    args = ap.parse_args()\n\n    df = apply_qc(pd.read_csv(args.inp))\n    write_report(df, args.report, src=args.inp)\n    out = df if args.keep_excluded else df[df["qc_pass"]].copy()\n    out.to_csv(args.out, index=False)\n\n    print("QC: %d audited, %d hard-excluded, %d retained"\n          % (len(df), int(df["qc_exclude"].sum()), int(df["qc_pass"].sum())))\n    kept = df[df["qc_pass"]]\n    for f in sorted({x for s in kept["qc_flags"] for x in s.split(";") if x}):\n        c = int(kept["qc_flags"].str.split(";").apply(lambda L: f in L).sum())\n        print("  flag %-24s %d" % (f, c))\n    print("Wrote %s and %s" % (os.path.abspath(args.out), os.path.abspath(args.report)))\n\n\nif __name__ == "__main__":\n    main()')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('image_features.py').write_text('#!/usr/bin/env python3\n"""\nimage_features.py - Weeks 5-6: descriptors from the simulated-microscopy renders.\n\nFor each instance and each configuration (Conf1, Conf2, Epi1) it reads every view,\ncomputes intensity statistics and an Otsu-threshold foreground segmentation, then\naggregates across views (mean and max).\n\nColumns produced per configuration C:\n    img_C_n_views\n    img_C_intensity_mean / img_C_intensity_std\n    img_C_fg_fraction        / img_C_fg_fraction_max\n    img_C_fg_bbox_area_frac  / img_C_fg_bbox_area_frac_max\n    img_C_otsu_threshold\n\nNOTE (full-shard finding): img_Epi1_fg_bbox_area_frac_max is exactly 1.0 for every\ninstance - Otsu never finds a clean split on the blurred epifluorescence renders.\nIt is a zero-variance column and is dropped automatically by agreement_stats.py.\n\n  python3 image_features.py --root /path/to/shard --out image_features.csv\n"""\nimport argparse\nimport os\nimport time\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nfrom hit_common import IMG_CONFIGS, find_instance_dirs, images_for_config, instance_id, otsu_threshold\n\n\ndef _view_features(path):\n    im = Image.open(path)\n    if im.mode not in ("L", "I;16", "I"):\n        im = im.convert("L")\n    a = np.asarray(im).astype(np.float64)\n    if a.ndim == 3:\n        a = a.mean(axis=2)\n    if a.max() > 255 or a.dtype != np.uint8:\n        rng = a.max() - a.min()\n        a8 = np.zeros_like(a, dtype=np.uint8) if rng == 0 else \\\n            ((a - a.min()) / rng * 255.0).astype(np.uint8)\n    else:\n        a8 = a.astype(np.uint8)\n\n    thr = otsu_threshold(a8)\n    fg = a8 > thr\n    frac = float(fg.mean())\n    if fg.any():\n        rs, cs = np.where(fg)\n        bbox = (rs.max() - rs.min() + 1) * (cs.max() - cs.min() + 1)\n        bbox_frac = float(bbox) / float(fg.size)\n    else:\n        bbox_frac = 0.0\n    return dict(mean=float(a8.mean()), std=float(a8.std()),\n                fg_fraction=frac, fg_bbox_area_frac=bbox_frac, otsu=thr)\n\n\ndef instance_image_features(d):\n    row = {"instance_id": instance_id(d)}\n    for c in IMG_CONFIGS:\n        files = images_for_config(d, c)\n        per = []\n        for f in files:\n            try:\n                per.append(_view_features(f))\n            except Exception:\n                pass\n        row["img_%s_n_views" % c] = len(per)\n        if not per:\n            for k in ("intensity_mean", "intensity_std", "fg_fraction", "fg_fraction_max",\n                      "fg_bbox_area_frac", "fg_bbox_area_frac_max", "otsu_threshold"):\n                row["img_%s_%s" % (c, k)] = np.nan\n            continue\n        g = lambda k: np.array([p[k] for p in per], dtype=np.float64)\n        row["img_%s_intensity_mean" % c] = float(g("mean").mean())\n        row["img_%s_intensity_std" % c] = float(g("std").mean())\n        row["img_%s_fg_fraction" % c] = float(g("fg_fraction").mean())\n        row["img_%s_fg_fraction_max" % c] = float(g("fg_fraction").max())\n        row["img_%s_fg_bbox_area_frac" % c] = float(g("fg_bbox_area_frac").mean())\n        row["img_%s_fg_bbox_area_frac_max" % c] = float(g("fg_bbox_area_frac").max())\n        row["img_%s_otsu_threshold" % c] = float(g("otsu").mean())\n    return row\n\n\ndef _safe(d):\n    try:\n        return instance_image_features(d)\n    except Exception as e:\n        return {"instance_id": instance_id(d), "img_error": str(e)}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="Image-derived descriptors for 3DMSL renders")\n    ap.add_argument("--root", required=True)\n    ap.add_argument("--out", default="image_features.csv")\n    ap.add_argument("--sample", type=int, default=0)\n    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1))\n    ap.add_argument("--progress-every", type=int, default=100)\n    ap.add_argument("--resume", action="store_true")\n    args = ap.parse_args()\n\n    dirs = find_instance_dirs(args.root)\n    if args.sample:\n        dirs = dirs[: args.sample]\n    done = set()\n    if args.resume and os.path.exists(args.out):\n        done = set(pd.read_csv(args.out, usecols=["instance_id"])["instance_id"].astype(str))\n        dirs = [d for d in dirs if instance_id(d) not in done]\n        print("Resuming: %d already done, %d remaining" % (len(done), len(dirs)))\n        if not dirs:\n            return\n    print("Image features for %d instances" % len(dirs), flush=True)\n\n    t0, rows = time.time(), []\n    if args.workers <= 1:\n        for i, d in enumerate(dirs, 1):\n            rows.append(_safe(d))\n            if i % args.progress_every == 0 or i == len(dirs):\n                print("  %d/%d" % (i, len(dirs)), flush=True)\n    else:\n        with ProcessPoolExecutor(max_workers=args.workers) as ex:\n            futs = [ex.submit(_safe, d) for d in dirs]\n            for i, f in enumerate(as_completed(futs), 1):\n                rows.append(f.result())\n                if i % args.progress_every == 0 or i == len(dirs):\n                    print("  %d/%d" % (i, len(dirs)), flush=True)\n\n    df = pd.DataFrame(rows)\n    if done:\n        df = pd.concat([pd.read_csv(args.out), df], ignore_index=True)\n    df = df.sort_values("instance_id", key=lambda s: s.astype(str).str.zfill(12))\n    df.to_csv(args.out, index=False)\n    ok = int((df[[c for c in df.columns if c.endswith("_n_views")]].sum(axis=1) > 0).sum())\n    print("Extracted features for %d/%d instances (%.1fs)" % (ok, len(df), time.time() - t0))\n    for c in IMG_CONFIGS:\n        col = "img_%s_fg_bbox_area_frac_max" % c\n        if col in df and df[col].notna().any() and df[col].nunique(dropna=True) == 1:\n            print("  NOTE: %s is constant (%.3f) - zero-variance column"\n                  % (col, df[col].dropna().iloc[0]))\n    print("Wrote %s" % os.path.abspath(args.out))\n\n\nif __name__ == "__main__":\n    main()')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('build_paired_features.py').write_text('#!/usr/bin/env python3\n"""\nbuild_paired_features.py - Weeks 5-6 deliverable: merge the QC\'d structural table with\nthe image-derived descriptors into one paired feature table, one row per instance with\nevery representation\'s descriptors side by side. This is the table the Weeks 7-8\nagreement statistics run on.\n\n  python3 build_paired_features.py --structural audit_descriptors_qc.csv \\\n                                   --images image_features.csv \\\n                                   --out paired_feature_table.csv\n"""\nimport argparse\nimport os\n\nimport numpy as np\nimport pandas as pd\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--structural", default="audit_descriptors_qc.csv")\n    ap.add_argument("--images", default="image_features.csv")\n    ap.add_argument("--out", default="paired_feature_table.csv")\n    args = ap.parse_args()\n\n    s = pd.read_csv(args.structural)\n    i = pd.read_csv(args.images)\n    s["instance_id"] = s["instance_id"].astype(str)\n    i["instance_id"] = i["instance_id"].astype(str)\n\n    df = s.merge(i, on="instance_id", how="left", validate="one_to_one")\n\n    # derived cross-representation comparison columns\n    if {"occ_volume_est", "mesh_volume"} <= set(df.columns):\n        df["vol_pct_err_occ_vs_mesh"] = 100.0 * (df["occ_volume_est"] - df["mesh_volume"]) / df["mesh_volume"]\n    for ax in "xyz":\n        a, b = "pc_trimmed_extent_%s" % ax, "mesh_extent_%s" % ax\n        if {a, b} <= set(df.columns):\n            df["extent_%s_pct_err_pc_vs_mesh" % ax] = 100.0 * (df[a] - df[b]) / df[b]\n        a2 = "occ_extent_%s" % ax\n        if {a2, b} <= set(df.columns):\n            df["extent_%s_pct_err_occ_vs_mesh" % ax] = 100.0 * (df[a2] - df[b]) / df[b]\n\n    df.to_csv(args.out, index=False)\n\n    nimg = int(df[[c for c in df.columns if c.endswith("_n_views")]].sum(axis=1).gt(0).sum()) \\\n        if any(c.endswith("_n_views") for c in df.columns) else 0\n    print("Paired table: %d rows x %d columns" % df.shape)\n    print("  with image features: %d" % nimg)\n    const = [c for c in df.columns\n             if df[c].dtype.kind in "fi" and df[c].notna().any() and df[c].nunique(dropna=True) <= 1]\n    if const:\n        print("  zero-variance numeric columns (dropped downstream): %s" % ", ".join(const))\n    print("Wrote %s" % os.path.abspath(args.out))\n\n\nif __name__ == "__main__":\n    main()')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('agreement_stats.py').write_text('#!/usr/bin/env python3\n"""\nagreement_stats.py - Weeks 7-8: representation-agreement statistics on the paired table.\n\n  ICC(2,1)  two-way random effects, absolute agreement, single measurement\n            (Shrout & Fleiss 1979; McGraw & Wong 1996 ICC(A,1)), computed from the\n            ANOVA variance components directly.\n  Spearman  rank correlation between each pair of representations (rank-transform\n            + Pearson, so no scipy dependency).\n  Bootstrap 95% CI for each ICC, resampling INSTANCES (not measurements), because\n            representations are paired within an instance.\n  Equivalence-margin check: sphericity-conditioned margin taken from the Week 1-2\n            measured error (15% when occ_fraction_inside < 2%, else 6%).\n            *** Starting point for pre-registration - needs adviser sign-off. ***\n\n  python3 agreement_stats.py --in paired_feature_table.csv --outdir .\n"""\nimport argparse\nimport os\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n# Descriptor families compared across representations. Add rows to extend the analysis.\nPAIRS = [\n    {"name": "extent_x",\n     "label": "extent_x (mesh vs. point-cloud[trimmed] vs. occupancy)",\n     "cols": ["mesh_extent_x", "pc_trimmed_extent_x", "occ_extent_x"],\n     "methods": ["mesh", "pointcloud_trimmed", "occupancy"]},\n    {"name": "volume",\n     "label": "volume (mesh vs. occupancy-derived estimate)",\n     "cols": ["mesh_volume", "occ_volume_est"],\n     "methods": ["mesh", "occupancy_est"]},\n]\n\nMARGIN_LOW_OCC = 15.0    # percent, occ_fraction_inside < 2%\nMARGIN_OTHER = 6.0       # percent, everything else\nLOW_OCC_CUT = 0.02\n\n\ndef icc21(M):\n    """ICC(2,1): M is (n subjects x k methods), no missing values."""\n    M = np.asarray(M, dtype=np.float64)\n    n, k = M.shape\n    if n < 2 or k < 2:\n        return np.nan\n    grand = M.mean()\n    row_m = M.mean(axis=1)\n    col_m = M.mean(axis=0)\n    SSR = k * ((row_m - grand) ** 2).sum()\n    SSC = n * ((col_m - grand) ** 2).sum()\n    SSE = ((M - row_m[:, None] - col_m[None, :] + grand) ** 2).sum()\n    MSR = SSR / (n - 1)\n    MSC = SSC / (k - 1)\n    MSE = SSE / ((n - 1) * (k - 1))\n    denom = MSR + (k - 1) * MSE + k * (MSC - MSE) / n\n    return np.nan if denom == 0 else float((MSR - MSE) / denom)\n\n\ndef spearman(x, y):\n    x = pd.Series(x).rank().to_numpy()\n    y = pd.Series(y).rank().to_numpy()\n    if x.std() == 0 or y.std() == 0:\n        return np.nan\n    return float(np.corrcoef(x, y)[0, 1])\n\n\ndef bootstrap_icc(M, n_boot=1000, seed=0):\n    rng = np.random.default_rng(seed)\n    n = M.shape[0]\n    vals = []\n    for _ in range(n_boot):\n        idx = rng.integers(0, n, n)\n        v = icc21(M[idx])\n        if np.isfinite(v):\n            vals.append(v)\n    if not vals:\n        return (np.nan, np.nan)\n    return tuple(np.percentile(vals, [2.5, 97.5]))\n\n\ndef drop_zero_variance(df):\n    dead = [c for c in df.columns\n            if df[c].dtype.kind in "fi" and df[c].notna().any() and df[c].nunique(dropna=True) <= 1]\n    if dead:\n        print("Dropping zero-variance columns: %s" % ", ".join(dead))\n    return df.drop(columns=dead), dead\n\n\ndef fig_icc(results, path):\n    rows = [r for r in results if np.isfinite(r["icc"])]\n    if not rows:\n        return\n    fig, ax = plt.subplots(figsize=(7, 3.2 + 0.4 * len(rows)))\n    y = np.arange(len(rows))\n    icc = [r["icc"] for r in rows]\n    lo = [max(r["icc"] - r["ci"][0], 0) for r in rows]\n    hi = [max(r["ci"][1] - r["icc"], 0) for r in rows]\n    ax.barh(y, icc, xerr=[lo, hi], color="#4C72B0", ecolor="#333333", capsize=4, height=0.5)\n    ax.set_yticks(y)\n    ax.set_yticklabels(["%s\\n(n=%d)" % (r["name"], r["n"]) for r in rows])\n    ax.set_xlim(0, 1.05)\n    ax.axvline(0.90, ls="--", c="#888888", lw=1)\n    ax.text(0.90, -0.7, "0.90 \'excellent\'", fontsize=8, color="#666666", ha="center")\n    for yi, r in zip(y, rows):\n        ax.text(min(r["icc"], 1.0) + 0.01, yi, "%.3f" % r["icc"], va="center", fontsize=9)\n    ax.set_xlabel("ICC(2,1), absolute agreement (bars = 95% bootstrap CI)")\n    ax.set_title("Cross-representation agreement")\n    ax.invert_yaxis()\n    fig.tight_layout()\n    fig.savefig(path, dpi=150)\n    plt.close(fig)\n\n\ndef fig_bland_altman(df, path):\n    panels = []\n    if {"mesh_extent_x", "pc_trimmed_extent_x"} <= set(df.columns):\n        panels.append(("extent_x: point-cloud[trimmed] vs. mesh", df["mesh_extent_x"], df["pc_trimmed_extent_x"]))\n    if {"mesh_volume", "occ_volume_est"} <= set(df.columns):\n        panels.append(("volume: occupancy estimate vs. mesh", df["mesh_volume"], df["occ_volume_est"]))\n    if not panels:\n        return\n    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4.5))\n    axes = np.atleast_1d(axes)\n    for ax, (title, a, b) in zip(axes, panels):\n        m = (a + b) / 2.0\n        d = b - a\n        ok = np.isfinite(m) & np.isfinite(d)\n        m, d = m[ok], d[ok]\n        ax.scatter(m, d, s=6, alpha=0.25, color="#4C72B0", edgecolors="none")\n        mu, sd = d.mean(), d.std(ddof=1)\n        ax.axhline(mu, color="#C44E52", lw=1.4, label="bias = %.3g" % mu)\n        ax.axhline(mu + 1.96 * sd, color="#C44E52", ls="--", lw=1, label="+/-1.96 SD")\n        ax.axhline(mu - 1.96 * sd, color="#C44E52", ls="--", lw=1)\n        ax.set_xlabel("mean of the two methods")\n        ax.set_ylabel("difference (second - first)")\n        ax.set_title(title, fontsize=10)\n        ax.legend(fontsize=8, frameon=False)\n    fig.suptitle("Bland-Altman style agreement plots", y=1.01)\n    fig.tight_layout()\n    fig.savefig(path, dpi=150, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--in", dest="inp", default="paired_feature_table.csv")\n    ap.add_argument("--outdir", default=".")\n    ap.add_argument("--n-boot", type=int, default=1000)\n    ap.add_argument("--seed", type=int, default=0)\n    args = ap.parse_args()\n    os.makedirs(args.outdir, exist_ok=True)\n\n    df = pd.read_csv(args.inp)\n    df, dead = drop_zero_variance(df)\n\n    results = []\n    for spec in PAIRS:\n        cols = [c for c in spec["cols"] if c in df.columns]\n        if len(cols) < 2:\n            print("skipping %s (missing columns)" % spec["name"])\n            continue\n        sub = df[cols].apply(pd.to_numeric, errors="coerce").dropna()\n        M = sub.to_numpy()\n        icc = icc21(M)\n        ci = bootstrap_icc(M, n_boot=args.n_boot, seed=args.seed)\n        sp = {}\n        for a in range(len(cols)):\n            for b in range(a + 1, len(cols)):\n                sp["%s vs %s" % (spec["methods"][a], spec["methods"][b])] = spearman(M[:, a], M[:, b])\n        results.append(dict(name=spec["name"], label=spec["label"], n=len(sub),\n                            icc=icc, ci=ci, spearman=sp, cols=cols))\n\n    # equivalence-margin check (occupancy vs mesh volume)\n    eq = None\n    if {"mesh_volume", "occ_volume_est", "occ_fraction_inside"} <= set(df.columns):\n        d = df[["mesh_volume", "occ_volume_est", "occ_fraction_inside"]].dropna()\n        err = (100.0 * (d["occ_volume_est"] - d["mesh_volume"]) / d["mesh_volume"]).abs()\n        low = d["occ_fraction_inside"] < LOW_OCC_CUT\n        margin = np.where(low, MARGIN_LOW_OCC, MARGIN_OTHER)\n        within = err.to_numpy() <= margin\n        eq = dict(n=len(d), overall=100.0 * within.mean(),\n                  n_low=int(low.sum()),\n                  low=100.0 * within[low.to_numpy()].mean() if low.any() else np.nan,\n                  n_high=int((~low).sum()),\n                  high=100.0 * within[(~low).to_numpy()].mean() if (~low).any() else np.nan)\n\n    icc_png = os.path.join(args.outdir, "icc_summary.png")\n    ba_png = os.path.join(args.outdir, "agreement_scatter.png")\n    fig_icc(results, icc_png)\n    fig_bland_altman(df, ba_png)\n\n    lines = ["# Representation-Agreement Statistics (Weeks 7-8)", "",\n             "Input: `%s` (n=%d rows)" % (os.path.basename(args.inp), len(df)), ""]\n    if dead:\n        lines += ["Zero-variance columns dropped before analysis: %s"\n                  % ", ".join("`%s`" % c for c in dead), ""]\n    lines += ["## ICC(2,1) and rank agreement", "",\n              "| Pair | n | ICC(2,1) | 95% bootstrap CI |", "|---|---|---|---|"]\n    for r in results:\n        lines.append("| %s | %d | %.3f | [%.3f, %.3f] |"\n                     % (r["label"], r["n"], r["icc"], r["ci"][0], r["ci"][1]))\n    lines += ["", "Spearman rank correlations:", ""]\n    for r in results:\n        for k, v in r["spearman"].items():\n            lines.append("- %s - %s: **%.4f**" % (r["name"], k, v))\n    if eq:\n        lines += ["", "## Equivalence-margin check (occupancy vs. mesh volume)", "",\n                  "Margins: **%.0f%%** when `occ_fraction_inside` < %g, **%.0f%%** otherwise."\n                  % (MARGIN_LOW_OCC, LOW_OCC_CUT, MARGIN_OTHER),\n                  "These come from the Week 1-2 measured error and are a *starting point for",\n                  "pre-registration*, not a finalized value - they need adviser/mentor sign-off.", "",\n                  "- Overall: **%.1f%%** of %d instances within margin" % (eq["overall"], eq["n"]),\n                  "- Low `occ_fraction_inside` (<%g, n=%d, %.0f%% margin): **%.1f%%** within"\n                  % (LOW_OCC_CUT, eq["n_low"], MARGIN_LOW_OCC, eq["low"]),\n                  "- Higher `occ_fraction_inside` (>=%g, n=%d, %.0f%% margin): **%.1f%%** within"\n                  % (LOW_OCC_CUT, eq["n_high"], MARGIN_OTHER, eq["high"])]\n    lines += ["", "## Figures", "", "- `icc_summary.png`", "- `agreement_scatter.png`", ""]\n    rep = os.path.join(args.outdir, "agreement_report.md")\n    with open(rep, "w") as fh:\n        fh.write("\\n".join(lines))\n\n    print("\\n".join(lines[:40]))\n    print("\\nWrote %s, %s, %s" % (rep, icc_png, ba_png))\n\n\nif __name__ == "__main__":\n    main()')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('analyze_results.py').write_text('#!/usr/bin/env python3\n"""\nanalyze_results.py - descriptive pass over the paired feature table: morphology-space\nsummary, the size/shape confound, and how image-derived features track shape vs. size.\nReproduces the three full-shard findings recorded in the Weeks 5-6 milestone.\n\n  python3 analyze_results.py --in paired_feature_table.csv --outdir .\n"""\nimport argparse\nimport os\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\ndef r(df, a, b):\n    if a not in df or b not in df:\n        return np.nan\n    d = df[[a, b]].apply(pd.to_numeric, errors="coerce").dropna()\n    if len(d) < 3 or d[a].std() == 0 or d[b].std() == 0:\n        return np.nan\n    return float(np.corrcoef(d[a], d[b])[0, 1])\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--in", dest="inp", default="paired_feature_table.csv")\n    ap.add_argument("--outdir", default=".")\n    args = ap.parse_args()\n    os.makedirs(args.outdir, exist_ok=True)\n    df = pd.read_csv(args.inp)\n    n = len(df)\n    L = ["# Descriptive Analysis (n=%d)" % n, ""]\n\n    L += ["## Morphology space", "", "| Descriptor | median | min | max |", "|---|---|---|---|"]\n    for c in ("mesh_sphericity", "mesh_elongation", "mesh_volume", "mesh_area", "occ_fraction_inside"):\n        if c in df:\n            s = pd.to_numeric(df[c], errors="coerce").dropna()\n            if len(s):\n                L.append("| `%s` | %.4g | %.4g | %.4g |" % (c, s.median(), s.min(), s.max()))\n\n    L += ["", "## Size / shape confound", ""]\n    rv = r(df, "mesh_volume", "mesh_sphericity")\n    L.append("- `mesh_volume` vs `mesh_sphericity`: **r = %.3f** - size and shape are not "\n             "independent axes in this dataset, so a naive volume-vs-shape comparison is "\n             "partly confounded." % rv)\n\n    L += ["", "## Occupancy volume error structure", ""]\n    if {"occ_volume_est", "mesh_volume"} <= set(df.columns):\n        err = (100.0 * (df["occ_volume_est"] - df["mesh_volume"]) / df["mesh_volume"]).abs()\n        df = df.assign(_abs_vol_err=err)\n        L += ["- absolute error vs mesh volume: mean **%.2f%%**, median **%.2f%%**"\n              % (err.mean(), err.median()),\n              "- error vs `mesh_sphericity`: r = **%.3f**" % r(df, "_abs_vol_err", "mesh_sphericity"),\n              "- error vs `occ_fraction_inside`: r = **%.3f**" % r(df, "_abs_vol_err", "occ_fraction_inside")]\n        if "occ_fraction_inside" in df:\n            low = df["occ_fraction_inside"] < 0.02\n            if low.any():\n                L.append("- `occ_fraction_inside` < 2%% (n=%d): mean error **%.2f%%** vs **%.2f%%** "\n                         "for the rest" % (int(low.sum()), err[low].mean(), err[~low].mean()))\n\n    L += ["", "## Image-derived features: shape vs. size", "",\n          "| Image feature | r with `mesh_sphericity` | r with `mesh_extent_x` |", "|---|---|---|"]\n    for c in [c for c in df.columns if c.startswith("img_") and\n              ("fg_fraction" in c or "fg_bbox_area_frac" in c)]:\n        L.append("| `%s` | %.3f | %.3f |" % (c, r(df, c, "mesh_sphericity"), r(df, c, "mesh_extent_x")))\n    L += ["", "Threshold-based image morphometry recovers **shape** better than absolute "\n          "**size** - a representation-invariance result worth carrying into the manuscript.", ""]\n\n    # figure: sphericity vs volume error + error distribution\n    if "_abs_vol_err" in df:\n        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))\n        d = df[["mesh_sphericity", "_abs_vol_err"]].dropna()\n        axes[0].scatter(d["mesh_sphericity"], d["_abs_vol_err"], s=6, alpha=0.3,\n                        color="#4C72B0", edgecolors="none")\n        axes[0].set_xlabel("mesh sphericity")\n        axes[0].set_ylabel("|occupancy volume error| (%)")\n        axes[0].set_title("Volume error is sphericity-conditioned", fontsize=10)\n        axes[1].hist(df["_abs_vol_err"].dropna(), bins=60, color="#4C72B0")\n        axes[1].set_xlabel("|occupancy volume error| (%)")\n        axes[1].set_ylabel("instances")\n        axes[1].set_title("Error distribution", fontsize=10)\n        fig.tight_layout()\n        p = os.path.join(args.outdir, "volume_consistency.png")\n        fig.savefig(p, dpi=150)\n        plt.close(fig)\n        L += ["Figure: `volume_consistency.png`", ""]\n\n    out = os.path.join(args.outdir, "analysis_report.md")\n    with open(out, "w") as fh:\n        fh.write("\\n".join(L))\n    print("\\n".join(L))\n    print("\\nWrote %s" % out)\n\n\nif __name__ == "__main__":\n    main()')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('tests.py').write_text('#!/usr/bin/env python3\n"""\ntests.py - unit tests for the 3DMSL reliability pipeline.\n\n  python3 tests.py                                  # 7 pure-function tests, no data needed\n  python3 tests.py --shard-root /path/to/shard      # + the coordinate-frame test on real data\n\nThe data-dependent test re-verifies the single most load-bearing correctness claim in\nthe pipeline: that the occupancy npz\'s loc/scale reproduce the mesh\'s own bounding-box\ncentre and extent, i.e. that normalized = (raw - loc)/scale.\n"""\nimport argparse\nimport sys\n\nimport numpy as np\n\nfrom hit_common import (elongation, find_instance_dirs, instance_id, load_mesh,\n                        load_occupancy, otsu_threshold, pct_diff, raw_extent,\n                        sphericity, trimmed_extent, unnormalize)\n\nFAILS = []\n\n\ndef check(name, cond, detail=""):\n    print(("  PASS  " if cond else "  FAIL  ") + name + (("  " + detail) if detail else ""))\n    if not cond:\n        FAILS.append(name)\n\n\ndef test_sphericity_sphere():\n    R = 3.0\n    V = 4.0 / 3.0 * np.pi * R ** 3\n    A = 4.0 * np.pi * R ** 2\n    check("sphericity of a perfect sphere == 1", abs(sphericity(V, A) - 1.0) < 1e-12)\n\n\ndef test_sphericity_bounded():\n    # an elongated box must score below a sphere of the same volume\n    V, A = 8.0, 2 * (1 * 8 + 1 * 1 + 8 * 1)\n    s = sphericity(V, A)\n    check("sphericity of a non-sphere is in (0,1)", 0 < s < 1, "s=%.4f" % s)\n\n\ndef test_sphericity_degenerate():\n    check("sphericity handles zero area/volume",\n          np.isnan(sphericity(0, 1)) and np.isnan(sphericity(1, 0)))\n\n\ndef test_trimmed_extent_robust():\n    rng = np.random.default_rng(0)\n    pts = rng.uniform(-1, 1, size=(5000, 3))\n    pts[0] = [50.0, 0.0, 0.0]                      # one sampling-noise outlier\n    raw = raw_extent(pts)\n    trim = trimmed_extent(pts)\n    check("raw extent is corrupted by a single outlier", raw[0] > 20, "raw_x=%.1f" % raw[0])\n    check("trimmed (1-99 pct) extent rejects it", trim[0] < 2.2, "trim_x=%.3f" % trim[0])\n\n\ndef test_elongation():\n    check("elongation = longest/shortest axis", abs(elongation([4.0, 2.0, 1.0]) - 4.0) < 1e-12)\n\n\ndef test_otsu_separates():\n    img = np.full((64, 64), 40, dtype=np.uint8)\n    img[:, 32:] = 200\n    t = otsu_threshold(img)\n    fg = (img > t).mean()\n    check("Otsu splits a bimodal image at ~50% foreground",\n          40 <= t < 200 and abs(fg - 0.5) < 0.02, "thr=%.0f fg=%.3f" % (t, fg))\n\n\ndef test_unnormalize_roundtrip():\n    rng = np.random.default_rng(1)\n    raw = rng.normal(size=(100, 3)) * 7 + 30\n    loc = raw.mean(axis=0)\n    scale = np.array([5.0])\n    norm = (raw - loc) / scale\n    check("unnormalize inverts (raw-loc)/scale",\n          np.allclose(unnormalize(norm, loc, scale), raw))\n\n\ndef test_pct_diff():\n    check("pct_diff basic + zero guard",\n          abs(pct_diff(110, 100) - 10.0) < 1e-12 and np.isnan(pct_diff(1, 0)))\n\n\ndef test_coordinate_frame(root):\n    dirs = find_instance_dirs(root)\n    if not dirs:\n        check("coordinate-frame check (real data)", False, "no instances under " + root)\n        return\n    d = dirs[0]\n    m = load_mesh(d)\n    pts, occ, loc, scale = load_occupancy(d)\n    center = np.asarray(m.bounds, dtype=np.float64).mean(axis=0)\n    ext = np.asarray(m.extents, dtype=np.float64)\n    s = np.asarray(scale).reshape(-1)\n    s_full = np.repeat(s, 3) if s.size == 1 else s[:3]\n    center_err = float(np.max(np.abs(loc - center)) / max(ext.max(), 1e-12))\n    scale_err = float(np.min(np.abs(s_full - ext)) / max(ext.max(), 1e-12))\n    check("occupancy `loc` == mesh bbox centre (instance %s)" % instance_id(d),\n          center_err < 0.01, "rel err %.2e" % center_err)\n    check("occupancy `scale` matches a mesh bbox extent (instance %s)" % instance_id(d),\n          scale_err < 0.01, "rel err %.2e" % scale_err)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--shard-root", default=None,\n                    help="run the data-dependent coordinate-frame test against a real shard")\n    args = ap.parse_args()\n\n    print("Pure-function tests:")\n    for t in (test_sphericity_sphere, test_sphericity_bounded, test_sphericity_degenerate,\n              test_trimmed_extent_robust, test_elongation, test_otsu_separates,\n              test_unnormalize_roundtrip, test_pct_diff):\n        t()\n    if args.shard_root:\n        print("Data-dependent tests:")\n        test_coordinate_frame(args.shard_root)\n    else:\n        print("Data-dependent tests: skipped (pass --shard-root to run them)")\n\n    print("\\n%s" % ("ALL TESTS PASSED" if not FAILS else "FAILURES: " + ", ".join(FAILS)))\n    sys.exit(1 if FAILS else 0)\n\n\nif __name__ == "__main__":\n    main()')


## 6. Unit tests and sample audit

The tests must pass on the real shard before any full run. The sample audit then requires usable meshes and occupancy files; an empty paired table is treated as a failure.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    subprocess.run([sys.executable, 'tests.py', '--shard-root', SHARD_ROOT], check=True)
    print('Unit tests passed on the real shard.')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    import pandas as pd
    import subprocess, sys

    SAMPLE = os.path.join(OUT_DIR, 'sample')
    os.makedirs(SAMPLE, exist_ok=True)
    subprocess.run([sys.executable, 'audit_3dmsl.py', '--root', SHARD_ROOT, '--sample', '15',
                    '--workers', '2', '--progress-every', '5', '--out', f'{SAMPLE}/audit_sample.csv'], check=True)
    sample_audit = pd.read_csv(f'{SAMPLE}/audit_sample.csv')
    if len(sample_audit) != 15 or sample_audit['mesh_ok'].sum() < 14 or sample_audit['occ_ok'].sum() < 14:
        raise RuntimeError('Sample audit did not load real 3DMSL geometry; inspect audit_sample.csv before continuing.')
    subprocess.run([sys.executable, 'qc_filter.py', '--in', f'{SAMPLE}/audit_sample.csv',
                    '--out', f'{SAMPLE}/audit_sample_qc.csv', '--report', f'{SAMPLE}/qc_report.md'], check=True)
    subprocess.run([sys.executable, 'image_features.py', '--root', SHARD_ROOT, '--sample', '15',
                    '--workers', '2', '--progress-every', '5', '--out', f'{SAMPLE}/image_features.csv'], check=True)
    subprocess.run([sys.executable, 'build_paired_features.py', '--structural', f'{SAMPLE}/audit_sample_qc.csv',
                    '--images', f'{SAMPLE}/image_features.csv', '--out', f'{SAMPLE}/paired_sample.csv'], check=True)
    sample_paired = pd.read_csv(f'{SAMPLE}/paired_sample.csv')
    if len(sample_paired) < 10:
        raise RuntimeError('The sample paired table is unexpectedly small; inspect sample QC and image reports.')
    print(f'Sample passed: {len(sample_paired)} paired instances.')


## 7. Full-shard structural audit

The `--resume` option is safe here because this version writes to a new verified output directory. The cell checks that geometry loads before continuing.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    import pandas as pd

    AUDIT = os.path.join(OUT_DIR, 'audit_full_descriptors.csv')
    subprocess.run([sys.executable, 'audit_3dmsl.py', '--root', SHARD_ROOT,
                    '--out', AUDIT, '--workers', str(os.cpu_count() or 2),
                    '--progress-every', '100', '--resume'], check=True)
    full_audit = pd.read_csv(AUDIT)
    if len(full_audit) != expected_instances or full_audit['mesh_ok'].sum() < 2700 or full_audit['occ_ok'].sum() < 2700:
        raise RuntimeError('Full audit did not load the expected real 3DMSL instances. Do not continue.')
    print(f'Full audit verified: {len(full_audit)} instances; '
          f'{full_audit["mesh_ok"].sum()} meshes and {full_audit["occ_ok"].sum()} occupancies loaded.')


## 8. QC filter (Weeks 3–4)

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    import pandas as pd
    from pathlib import Path
    from IPython.display import Markdown, display

    QC = os.path.join(OUT_DIR, 'audit_descriptors_qc.csv')
    subprocess.run([sys.executable, 'qc_filter.py', '--in', AUDIT,
                    '--out', QC, '--report', os.path.join(OUT_DIR, 'qc_report.md')], check=True)
    qc_table = pd.read_csv(QC)
    if 'qc_pass' not in qc_table or qc_table['qc_pass'].sum() < 2700:
        raise RuntimeError('QC retained too few real 3DMSL instances. Inspect the QC report before continuing.')
    display(Markdown(Path(OUT_DIR, 'qc_report.md').read_text()))


## 9. Image features + paired feature table (Weeks 5–6)

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    import pandas as pd

    IMG = os.path.join(OUT_DIR, 'image_features.csv')
    PAIRED = os.path.join(OUT_DIR, 'paired_feature_table.csv')
    subprocess.run([sys.executable, 'image_features.py', '--root', SHARD_ROOT,
                    '--out', IMG, '--workers', str(os.cpu_count() or 2),
                    '--progress-every', '100', '--resume'], check=True)
    subprocess.run([sys.executable, 'build_paired_features.py', '--structural', QC,
                    '--images', IMG, '--out', PAIRED], check=True)
    pt = pd.read_csv(PAIRED)
    if len(pt) < 2700:
        raise RuntimeError(f'Paired table has only {len(pt)} rows; expected approximately 2,720.')
    print('Valid paired table:', pt.shape)
    display(pt.head())


## 10. Agreement statistics

These analyses compute ICC, rank correlation, and exploratory equivalence checks. The 6% and 15% margins were informed by the observed 3DMSL errors; choose a scientifically justified primary margin with the adviser before confirmatory use.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    from pathlib import Path
    from IPython.display import Image, Markdown, display

    if len(pd.read_csv(PAIRED)) < 2700:
        raise RuntimeError('Paired table is empty or incomplete; rerun the verified audit first.')
    subprocess.run([sys.executable, 'agreement_stats.py', '--in', PAIRED,
                    '--outdir', OUT_DIR, '--n-boot', '1000'], check=True)
    display(Markdown(Path(OUT_DIR, 'agreement_report.md').read_text()))
    for filename in ('icc_summary.png', 'agreement_scatter.png'):
        figure = Path(OUT_DIR, filename)
        if figure.exists():
            display(Image(filename=str(figure)))


## 11. Descriptive analysis

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    from pathlib import Path
    from IPython.display import Image, Markdown, display

    if len(pd.read_csv(PAIRED)) < 2700:
        raise RuntimeError('Paired table is empty or incomplete; rerun the verified audit first.')
    subprocess.run([sys.executable, 'analyze_results.py', '--in', PAIRED,
                    '--outdir', OUT_DIR], check=True)
    display(Markdown(Path(OUT_DIR, 'analysis_report.md').read_text()))
    figure = Path(OUT_DIR, 'volume_consistency.png')
    if figure.exists():
        display(Image(filename=str(figure)))


## 12. Copy 3DMSL results back to Drive

Files under `/content` disappear when the Colab runtime is recycled. Copy results and scripts to the persistent `HIT_outputs` folder.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import shutil
    import pandas as pd

    if len(pd.read_csv(PAIRED)) < 2700:
        raise RuntimeError('Refusing to save an empty or incomplete paired table to Drive.')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    for filename in sorted(os.listdir(OUT_DIR)):
        source = os.path.join(OUT_DIR, filename)
        if os.path.isfile(source):
            shutil.copy2(source, os.path.join(DRIVE_OUT, filename))
            print('saved', filename)
    scripts_dir = os.path.join(DRIVE_OUT, 'scripts')
    os.makedirs(scripts_dir, exist_ok=True)
    for filename in os.listdir(WORK_DIR):
        if filename.endswith('.py'):
            shutil.copy2(os.path.join(WORK_DIR, filename), os.path.join(scripts_dir, filename))
    print('Verified results saved to', DRIVE_OUT)


---

### Check against the earlier 3DMSL run

Expected checks include 2,720 loaded instances, no hard QC exclusions, approximately +3.66% mean occupancy-volume error relative to mesh, and volume ICC around 0.994. Treat these as checks against the earlier run, not as hard-coded results of the current execution.

## 13. Repeatability and the bias/noise split

This pass estimates numerical point-sampling repeatability and contrasts it with systematic occupancy-versus-mesh bias. Run the earlier 3DMSL setup first so the scripts and shard exist.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import os
    from pathlib import Path
    import pandas as pd
    import shutil

    WORK_DIR = '/content/hit'
    OUT_DIR = '/content/hit_outputs_reproduced'
    DRIVE_OUT = '/content/drive/MyDrive/HIT_outputs_reproduced'
    PAIRED = os.path.join(OUT_DIR, 'paired_feature_table.csv')
    SHARD_ROOT = '/content/data/10_24553_27272'

    if not Path(SHARD_ROOT, '24553', 'raw_mesh.off').is_file():
        raise FileNotFoundError('The real 3DMSL shard is absent. Run Section 3 to locate or extract it.')
    if not os.path.exists(PAIRED):
        source = os.path.join(DRIVE_OUT, 'paired_feature_table.csv')
        if not os.path.exists(source):
            raise FileNotFoundError('No verified paired table is available. Run Sections 6–9 first.')
        os.makedirs(OUT_DIR, exist_ok=True)
        shutil.copy2(source, PAIRED)
    if len(pd.read_csv(PAIRED)) < 2700:
        raise RuntimeError('The paired table is empty or incomplete. Do not use the previous HIT_outputs folder.')
    os.chdir(WORK_DIR)
    print('Verified paired table:', PAIRED, '| real shard:', SHARD_ROOT)


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('repeatability.py').write_text('#!/usr/bin/env python3\n"""\nrepeatability.py - the missing piece of the Weeks 7-8 milestone: an estimate of the\noccupancy volume estimator\'s *own precision*, measured without reference to the mesh.\n\nThe equivalence margins used so far were derived from occupancy-vs-mesh agreement,\nwhich mixes two different things: how noisy the estimator is (precision) and how far\noff-centre it is (bias). This script separates them using only the occupancy archive:\n\n  1. K-fold resampling of each instance\'s query points gives the estimator\'s replicate\n     SD at full N - a genuine repeatability estimate, no mesh involved.\n  2. That empirical SD is compared with the analytic binomial floor sqrt(p(1-p)/N).\n     A ratio near 1 is a descriptive check; random fold assignment cannot establish independent query generation.\n  3. A convergence curve (N = 1k ... 100k) shows whether the estimate is approaching\n     the mesh volume or a persistent offset at the available sampling depth.\n  4. A uniformity check on the query-point marginals tests the one alternative\n     explanation - that the sampling design, not the labels, causes the offset.\n\n  python3 repeatability.py --root /path/to/shard --out repeatability.csv --sample 400\n"""\nimport argparse\nimport os\nimport time\nfrom collections import Counter\n\nimport numpy as np\nimport pandas as pd\ntry:\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\nexcept ImportError:\n    plt = None\n\nfrom hit_common import find_instance_dirs, instance_id, load_mesh, load_occupancy, raw_extent, unnormalize\n\nC_BLUE, C_ORANGE, C_INK, C_MUTED, C_GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#d8d7d2"\nK_FOLDS = 10\nCONVERGENCE_N = (1000, 2500, 5000, 10000, 25000, 50000, 100000)\n\n\ndef instance_repeatability(d, rng, with_mesh=True):\n    pts, occ, loc, scale = load_occupancy(d)\n    pts_raw = unnormalize(pts, loc, scale)\n    N = len(occ)\n    p = float(occ.mean())\n    v_box = float(np.prod(raw_extent(pts_raw)))\n    v_full = p * v_box\n\n    # 1. K-fold replicate SD -> SD of the full-N estimate\n    idx = rng.permutation(N)\n    folds = np.array_split(idx, K_FOLDS)\n    fold_est = np.array([occ[f].mean() * v_box for f in folds])\n    sd_fold = fold_est.std(ddof=1)\n    sd_full_emp = sd_fold / np.sqrt(K_FOLDS)\n\n    # 2. analytic binomial floor at full N\n    sd_full_ana = np.sqrt(p * (1 - p) / N) * v_box\n\n    # 3. convergence curve\n    conv = {}\n    for n in CONVERGENCE_N:\n        if n <= N:\n            conv["v_at_%d" % n] = float(occ[idx[:n]].mean() * v_box)\n\n    # 4. uniformity of the query-point marginals (10 bins per axis, chi-square df=9)\n    chi2 = []\n    for a in range(3):\n        counts, _ = np.histogram(pts_raw[:, a], bins=10)\n        exp = N / 10.0\n        chi2.append(float(((counts - exp) ** 2 / exp).sum()))\n\n    row = dict(instance_id=instance_id(d), n_points=N, occ_fraction_inside=p,\n               v_box=v_box, occ_volume_est=v_full,\n               sd_replicate=sd_full_emp, sd_analytic=sd_full_ana,\n               sd_ratio=sd_full_emp / sd_full_ana if sd_full_ana else np.nan,\n               rel_sd_pct=100 * sd_full_emp / v_full if v_full else np.nan,\n               rc_pct=100 * 2.77 * sd_full_emp / v_full if v_full else np.nan,\n               chi2_x=chi2[0], chi2_y=chi2[1], chi2_z=chi2[2])\n    row.update(conv)\n    if with_mesh:\n        try:\n            m = load_mesh(d)\n            row["mesh_volume"] = float(abs(m.volume))\n            row["mesh_area"] = float(m.area)\n            row["bias_pct"] = 100 * (v_full - row["mesh_volume"]) / row["mesh_volume"]\n        except Exception as e:\n            row["mesh_error"] = str(e)\n    return row\n\n\ndef fig_convergence(df, path):\n    if plt is None:\n        return\n    ns = [n for n in CONVERGENCE_N if ("v_at_%d" % n) in df.columns]\n    if not ns or "mesh_volume" not in df.columns:\n        return\n    rel = np.array([(100 * (df["v_at_%d" % n] / df["mesh_volume"] - 1)).to_numpy() for n in ns])\n    fig, ax = plt.subplots(figsize=(7.4, 4.3))\n    med = np.median(rel, axis=1)\n    lo = np.percentile(rel, 25, axis=1)\n    hi = np.percentile(rel, 75, axis=1)\n    ax.fill_between(ns, lo, hi, color=C_BLUE, alpha=0.18, lw=0)\n    ax.plot(ns, med, color=C_BLUE, lw=2.2, marker="o", ms=6,\n            markeredgecolor="white", markeredgewidth=1)\n    ax.axhline(0, color=C_INK, lw=1.4)\n    ax.text(ns[0], 0.25, "mesh volume", color=C_INK, fontsize=9, va="bottom")\n    ax.annotate("converges to %+.2f%%,\\nnot to zero" % med[-1],\n                xy=(ns[-1], med[-1]), xytext=(ns[-1] * 0.34, med[-1] + 1.9),\n                color=C_ORANGE, fontsize=10,\n                arrowprops=dict(arrowstyle="->", color=C_ORANGE, lw=1.4))\n    ax.set_xscale("log")\n    ax.set_xlabel("query points used in the volume estimate")\n    ax.set_ylabel("occupancy − mesh volume (%)")\n    ax.set_title("More sampling does not close the gap — the estimator is biased",\n                 fontsize=11, loc="left")\n    ax.set_facecolor("white")\n    for s in ("top", "right"):\n        ax.spines[s].set_visible(False)\n    for s in ("left", "bottom"):\n        ax.spines[s].set_color(C_GRID)\n    ax.tick_params(colors=C_MUTED, labelsize=9)\n    ax.xaxis.label.set_color(C_MUTED); ax.yaxis.label.set_color(C_MUTED)\n    ax.title.set_color(C_INK)\n    fig.tight_layout()\n    fig.savefig(path, dpi=200, facecolor="white")\n    plt.close(fig)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--root", required=True)\n    ap.add_argument("--out", default="repeatability.csv")\n    ap.add_argument("--outdir", default=".")\n    ap.add_argument("--sample", type=int, default=400,\n                    help="instances to measure (a precision estimate does not need all 2,720)")\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--progress-every", type=int, default=50)\n    args = ap.parse_args()\n    os.makedirs(args.outdir, exist_ok=True)\n\n    dirs = find_instance_dirs(args.root)\n    if not dirs:\n        raise FileNotFoundError("No 3DMSL instance folders with .off meshes under %s" % args.root)\n    rng = np.random.default_rng(args.seed)\n    if args.sample and args.sample < len(dirs):\n        dirs = [dirs[i] for i in sorted(rng.choice(len(dirs), args.sample, replace=False))]\n    print("Measuring repeatability on %d instances" % len(dirs), flush=True)\n\n    t0, rows = time.time(), []\n    out_csv = os.path.join(args.outdir, os.path.basename(args.out))\n    for i, d in enumerate(dirs, 1):\n        try:\n            rows.append(instance_repeatability(d, rng))\n        except Exception as e:\n            rows.append({"instance_id": instance_id(d), "error": str(e)})\n        if i % args.progress_every == 0 or i == len(dirs):\n            print("  %d/%d (%.2fs/instance)" % (i, len(dirs), (time.time() - t0) / i), flush=True)\n        if i == min(5, len(dirs)) and not any("rel_sd_pct" in row for row in rows):\n            pd.DataFrame(rows).to_csv(out_csv, index=False)\n            counts = Counter(row.get("error", "unknown error") for row in rows)\n            raise RuntimeError("All first %d instances failed. Most common error: %s. "\n                               "Inspect the error column in %s." % (i, counts.most_common(1)[0][0], out_csv))\n\n    df = pd.DataFrame(rows)\n    df.to_csv(out_csv, index=False)\n\n    if "rel_sd_pct" not in df:\n        counts = Counter(df["error"].fillna("unknown error")) if "error" in df else Counter()\n        raise RuntimeError("No repeatability measurements succeeded. Errors: %s" % counts.most_common(3))\n    ok = df[df["rel_sd_pct"].notna()].copy()\n    if ok.empty:\n        raise RuntimeError("No finite repeatability measurements succeeded; inspect %s" % out_csv)\n    failures = len(df) - len(ok)\n    if failures:\n        print("WARNING: %d/%d instances failed; inspect the error column in %s" % (failures, len(df), out_csv))\n    L = ["# Repeatability of the Occupancy Volume Estimator (n=%d instances)" % len(ok), "",\n         "%d of %d sampled instances failed and were excluded; inspect the error column in `%s`." %\n         (failures, len(df), os.path.basename(out_csv)) if failures else "All sampled instances produced finite repeatability estimates.", "",\n         "Measured from the occupancy archive alone: each instance\'s %d query points were" % int(ok["n_points"].median()),\n         "split into %d disjoint folds, a volume computed from each, and the replicate SD" % K_FOLDS,\n         "rescaled to the full-N estimate. No mesh is used anywhere in steps 1-2, so this is",\n         "precision, not agreement.", "",\n         "## 1. Precision", "", "| Quantity | Median | IQR |", "|---|---|---|",\n         "| replicate SD, as %% of the estimate | **%.3f%%** | %.3f – %.3f |"\n         % (ok["rel_sd_pct"].median(), ok["rel_sd_pct"].quantile(.25), ok["rel_sd_pct"].quantile(.75)),\n         "| repeatability coefficient (2.77 × SD) | **%.3f%%** | %.3f – %.3f |"\n         % (ok["rc_pct"].median(), ok["rc_pct"].quantile(.25), ok["rc_pct"].quantile(.75)),\n         "| empirical SD ÷ analytic binomial SD | **%.3f** | %.3f – %.3f |"\n         % (ok["sd_ratio"].median(), ok["sd_ratio"].quantile(.25), ok["sd_ratio"].quantile(.75)), "",\n         "Agreement of fold and binomial SDs is descriptive; random partitioning cannot prove independent query generation.",\n         "The repeatability coefficient describes pairwise repeat differences, not single-estimate accuracy.", ""]\n\n    mesh_ok = ok.dropna(subset=["mesh_volume"]) if "mesh_volume" in ok else ok.iloc[:0]\n    if len(mesh_ok) and "bias_pct" in mesh_ok:\n        b = mesh_ok["bias_pct"]\n        L += ["## 2. Precision vs. bias", "",\n              "| | Value |", "|---|---|",\n              "| repeatability (replicate SD) | **%.3f%%** |" % ok["rel_sd_pct"].median(),\n              "| bias (occupancy − mesh) | **%+.3f%%** |" % b.median(),\n              "| bias ÷ repeatability SD | **%.1f×** |" % (b.median() / ok["rel_sd_pct"].median()), "",\n              "The offset is roughly %.0f replicate standard deviations away from the mesh" % (b.median() / ok["rel_sd_pct"].median()),\n              "volume. An equivalence margin built on repeatability alone would be about",\n              "**±%.2f%%** (2.77 × SD) — far tighter than the 6%%/15%% margins used so far, and" % ok["rc_pct"].median(),\n              "the right basis for pre-registration once the bias is corrected rather than",\n              "absorbed into the margin.", ""]\n\n    ns = [n for n in CONVERGENCE_N if ("v_at_%d" % n) in mesh_ok.columns]\n    if ns and len(mesh_ok):\n        L += ["## 3. Convergence", "", "| Query points | median error vs. mesh |", "|---|---|"]\n        for n in ns:\n            L.append("| %d | %+.3f%% |" % (n, (100 * (mesh_ok["v_at_%d" % n] / mesh_ok["mesh_volume"] - 1)).median()))\n        final = float((100 * (mesh_ok["v_at_%d" % ns[-1]] / mesh_ok["mesh_volume"] - 1)).median())\n        sd = float(ok["rel_sd_pct"].median())\n        if abs(final) > 3 * sd:\n            L += ["", "At the largest sample the estimate still sits %+.3f%% from the mesh volume — %.1f×"\n                  % (final, abs(final) / sd),\n                  "the replicate SD at that sample size. This is consistent with a systematic",\n                  "offset at the available sampling depth. Confirm the mechanism before",\n                  "assuming that additional query points cannot reduce it.", ""]\n        else:\n            L += ["", "At the largest sample the residual error (%+.3f%%) is within %.1f× the replicate"\n                  % (final, abs(final) / sd if sd else float("nan")),\n                  "SD, so on this evidence the discrepancy is consistent with sampling noise rather",\n                  "than a fixed offset. Re-check against the full shard before concluding either way.", ""]\n\n    crit = 27.88   # chi-square 0.999 critical value, df = 9\n    frac = float(((ok[["chi2_x", "chi2_y", "chi2_z"]] > crit).any(axis=1)).mean())\n    L += ["## 4. Are the query points uniform?", "",\n          "| Axis | median χ² (10 bins, df=9) |", "|---|---|"]\n    for a in "xyz":\n        L.append("| %s | %.2f |" % (a, ok["chi2_%s" % a].median()))\n    L += ["", "Instances failing a χ² uniformity test on any axis at p<0.001 (crit %.2f): **%.1f%%**."\n          % (crit, 100 * frac), ""]\n    if frac < 0.05:\n        L += ["The one-axis marginal tests do not show major nonuniformity. They do not test",\n              "the joint 3D sampling distribution, so they cannot rule out every sampling",\n              "effect. A label-boundary difference remains a plausible mechanism and needs",\n              "direct verification against the 3DMSL generation procedure.", ""]\n    else:\n        L += ["**%.1f%% of instances fail the uniformity test, so the sampling design cannot be" % (100 * frac),\n              "ruled out as a source of the offset.** Inspect the query-point distribution",\n              "directly before attributing the bias to the occupancy labels — a non-uniform",\n              "sampling box biases `mean(occupancy) × box volume` on its own.", ""]\n    L += ["(χ² on 10 bins is mildly inflated by the float16 storage of the query points; treat",\n          "a modest exceedance rate as uniform rather than as evidence of structure.)", ""]\n\n    fig = os.path.join(args.outdir, "estimator_convergence.png")\n    fig_convergence(mesh_ok, fig)\n    if os.path.exists(fig):\n        L += ["## Figure", "", "- `estimator_convergence.png`", ""]\n\n    rep = os.path.join(args.outdir, "repeatability_report.md")\n    with open(rep, "w") as fh:\n        fh.write("\\n".join(L))\n    print("\\n".join(L))\n    written = [out_csv, rep] + ([fig] if os.path.exists(fig) else [])\n    print("\\nWrote %s" % ", ".join(written))\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('agreement_v2.py').write_text('#!/usr/bin/env python3\n"""\nagreement_v2.py - Weeks 7-8, second pass: separate systematic BIAS from measurement\nNOISE, and report agreement statistics that do not depend on how heterogeneous the\nsample happens to be.\n\nWhy this exists. The first pass reported ICC(2,1) = 0.994 for volume alongside "88.8%\nof instances inside the equivalence margin". Those two statements are in tension, and\nthis script resolves it:\n\n  1. The occupancy-vs-mesh volume discrepancy is almost perfectly one-signed, so it is\n     bias, not noise.\n  2. Its size is several times the Monte-Carlo sampling floor the estimator is entitled\n     to, so sampling noise is a minor component.\n  3. It scales with the shape\'s surface-to-volume ratio - the signature of a constant\n     boundary offset, i.e. the occupancy labels describe a slightly dilated shape.\n  4. Because it is bias, it is correctable, and correcting it moves the margin pass\n     rate from 88.8% to ~99%.\n  5. ICC is inflated here by the 68x spread of mitochondrial volumes in the shard.\n     Recomputed inside narrow size strata it collapses - direct evidence that the\n     "excellent agreement" reading is a property of the sample, not the method.\n\n  python3 agreement_v2.py --in paired_feature_table.csv --outdir .\n"""\nimport argparse\nimport os\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n# categorical slots 1-3 of the validated palette (all-pairs safe, light surface)\nC_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"\nC_INK, C_MUTED, C_GRID = "#0b0b0b", "#52514e", "#d8d7d2"\n\nMARGIN_LOW_OCC, MARGIN_OTHER, LOW_OCC_CUT = 15.0, 6.0, 0.02\n\n\n# ----------------------------------------------------------------------------- stats\ndef icc21(M):\n    M = np.asarray(M, dtype=np.float64)\n    n, k = M.shape\n    if n < 2 or k < 2:\n        return np.nan\n    g = M.mean(); rm = M.mean(1); cm = M.mean(0)\n    MSR = k * ((rm - g) ** 2).sum() / (n - 1)\n    MSC = n * ((cm - g) ** 2).sum() / (k - 1)\n    MSE = ((M - rm[:, None] - cm[None, :] + g) ** 2).sum() / ((n - 1) * (k - 1))\n    d = MSR + (k - 1) * MSE + k * (MSC - MSE) / n\n    return float((MSR - MSE) / d) if d else np.nan\n\n\ndef ols(y, *cols):\n    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in cols])\n    b, *_ = np.linalg.lstsq(X, np.asarray(y, float), rcond=None)\n    pred = X @ b\n    ss = ((np.asarray(y, float) - np.asarray(y, float).mean()) ** 2).sum()\n    return b, pred, float(1 - ((y - pred) ** 2).sum() / ss)\n\n\ndef boot_ci(x, fn, n_boot=2000, seed=0):\n    rng = np.random.default_rng(seed)\n    x = np.asarray(x, float)\n    vals = [fn(x[rng.integers(0, len(x), len(x))]) for _ in range(n_boot)]\n    return tuple(np.percentile(vals, [2.5, 97.5]))\n\n\ndef agreement_block(a, b):\n    """Difference-based agreement summary for method b against reference a."""\n    a = np.asarray(a, float); b = np.asarray(b, float)\n    d = b - a; m = (a + b) / 2.0\n    sd = d.std(ddof=1)\n    return dict(bias=d.mean(), bias_pct=100 * d.mean() / a.mean(),\n                loa_lo=d.mean() - 1.96 * sd, loa_hi=d.mean() + 1.96 * sd,\n                percent_error=100 * 1.96 * sd / m.mean(),\n                percent_error_denominator="mean of both methods",\n                median_abs_pct=float(np.median(np.abs(100 * (b / a - 1)))),\n                prop_bias_r=float(np.corrcoef(m, d)[0, 1]),\n                one_signed=float(max((d > 0).mean(), (d < 0).mean())))\n\n\n# --------------------------------------------------------------------------- figures\ndef _style(ax):\n    ax.set_facecolor("white")\n    for s in ("top", "right"):\n        ax.spines[s].set_visible(False)\n    for s in ("left", "bottom"):\n        ax.spines[s].set_color(C_GRID)\n    ax.tick_params(colors=C_MUTED, labelsize=9)\n    ax.xaxis.label.set_color(C_MUTED); ax.yaxis.label.set_color(C_MUTED)\n    ax.title.set_color(C_INK)\n\n\ndef fig_bias_not_noise(e, sv, fit, mc_floor, path):\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))\n\n    ax = axes[0]\n    lo, hi = min(e.min(), -1.0), float(np.percentile(e, 99.5))\n    ax.hist(e, bins=60, range=(lo, hi), color=C_BLUE, edgecolor="none")\n    ax.set_xlim(lo - 0.5, hi + 0.5)\n    n_beyond = int((e > hi).sum())\n    if n_beyond:\n        ax.text(hi, ax.get_ylim()[1] * 0.06, "%d instances\\nbeyond →" % n_beyond,\n                color=C_MUTED, fontsize=8.5, ha="right", va="bottom")\n    ax.axvline(0, color=C_INK, lw=1.6)\n    ax.axvline(e.mean(), color=C_ORANGE, lw=2)\n    ax.annotate("mean bias\\n%+.2f%%" % e.mean(),\n                xy=(e.mean(), ax.get_ylim()[1] * 0.80),\n                xytext=(e.mean() + 1.4, ax.get_ylim()[1] * 0.80),\n                color=C_ORANGE, fontsize=9, va="center",\n                arrowprops=dict(arrowstyle="-", color=C_ORANGE, lw=1.2))\n    ax.annotate("zero", xy=(0, ax.get_ylim()[1] * 0.96), xytext=(0.6, ax.get_ylim()[1] * 0.96),\n                color=C_INK, fontsize=9, va="center")\n    ax.set_xlabel("occupancy − mesh volume (%)")\n    ax.set_ylabel("instances")\n    ax.set_title("%.2f%% of instances lie on one side of zero (all but %d)"\n                 % (100 * (e > 0).mean(), int((e <= 0).sum())),\n                 fontsize=10.5, loc="left")\n    _style(ax)\n\n    ax = axes[1]\n    ax.scatter(sv, e, s=5, alpha=0.22, color=C_BLUE, edgecolors="none")\n    xs = np.linspace(np.percentile(sv, 0.2), np.percentile(sv, 99.8), 50)\n    ax.plot(xs, fit[0] + fit[1] * xs, color=C_ORANGE, lw=2)\n    ax.text(0.03, 0.94, "R² = %.2f" % fit[2], transform=ax.transAxes,\n            color=C_ORANGE, fontsize=10, va="top")\n    ax.axhline(0, color=C_INK, lw=1)\n    ax.axhspan(-mc_floor, mc_floor, color=C_MUTED, alpha=0.14, lw=0)\n    ax.text(ax.get_xlim()[1], mc_floor, " Monte-Carlo\\n noise floor (±%.2f%%)" % mc_floor,\n            color=C_MUTED, fontsize=8.5, va="bottom", ha="right")\n    ax.set_xlabel(r"shape complexity  $A\\,/\\,V^{2/3}$")\n    ax.set_ylabel("occupancy − mesh volume (%)")\n    ax.set_title("Bias tracks surface-to-volume — a boundary-offset signature",\n                 fontsize=10.5, loc="left")\n    _style(ax)\n\n    fig.tight_layout()\n    fig.savefig(path, dpi=200, facecolor="white")\n    plt.close(fig)\n\n\ndef fig_icc_collapse(whole_v, dec_v, whole_e, dec_e, path):\n    fig, ax = plt.subplots(figsize=(8.4, 4.6))\n    groups = [("volume\\nmesh vs. occupancy", whole_v, np.asarray(dec_v)),\n              ("extent_x\\n3 representations", whole_e, np.asarray(dec_e))]\n    rng = np.random.default_rng(7)\n    for i, (lbl, whole, dec) in enumerate(groups):\n        xw, xd = i * 1.0 - 0.17, i * 1.0 + 0.17\n        ax.scatter([xw], [whole], s=110, marker="D", color=C_ORANGE, zorder=4,\n                   edgecolors="white", linewidths=1.2,\n                   label="whole sample" if i == 0 else None)\n        ax.annotate("%.3f" % whole, xy=(xw, whole), xytext=(xw - 0.055, whole),\n                    ha="right", va="center", color=C_ORANGE, fontsize=10)\n        ax.scatter(xd + rng.normal(0, 0.022, len(dec)), dec, s=36, color=C_BLUE,\n                   alpha=0.8, zorder=3, edgecolors="white", linewidths=0.8,\n                   label="one size decile" if i == 0 else None)\n        med = float(np.median(dec))\n        ax.plot([xd - 0.10, xd + 0.10], [med, med], color=C_BLUE, lw=2.4, zorder=5)\n        ax.annotate("median %.3f" % med, xy=(xd + 0.10, med), xytext=(xd + 0.15, med),\n                    ha="left", va="center", color=C_BLUE, fontsize=10)\n        ax.plot([xw, xd], [whole, med], color=C_GRID, lw=1.2, ls=":", zorder=1)\n    ax.axhline(0.90, ls="--", lw=1, color=C_MUTED, zorder=0)\n    ax.text(-0.52, 0.902, "0.90  “excellent”", color=C_MUTED, fontsize=8.5,\n            va="bottom", ha="left")\n    ax.set_xticks([0, 1])\n    ax.set_xticklabels([g[0] for g in groups], fontsize=10)\n    ax.set_xlim(-0.55, 1.62)\n    ax.set_ylim(0.70, 1.035)\n    ax.set_ylabel("ICC(2,1)")\n    ax.set_title("ICC falls apart once the sample\'s size spread is removed",\n                 fontsize=11, loc="left")\n    ax.legend(frameon=False, fontsize=9, loc="lower right", labelcolor=C_MUTED,\n              handletextpad=0.4)\n    _style(ax)\n    ax.tick_params(axis="x", length=0, labelsize=10)\n    for t in ax.get_xticklabels():\n        t.set_color(C_INK)\n    fig.tight_layout()\n    fig.savefig(path, dpi=200, facecolor="white")\n    plt.close(fig)\n\n\ndef fig_correction(rates, path):\n    fig, ax = plt.subplots(figsize=(7.6, 3.4))\n    labels = [r[0] for r in rates]\n    vals = [r[1] for r in rates]\n    colors = [C_MUTED if i == 0 else C_BLUE for i in range(len(rates))]\n    y = np.arange(len(rates))\n    ax.barh(y, vals, height=0.52, color=colors)\n    for yi, v in zip(y, vals):\n        ax.text(v - 1.2, yi, "%.1f%%" % v, va="center", ha="right",\n                color="white", fontsize=10, fontweight="bold")\n    ax.set_yticks(y); ax.set_yticklabels(labels, color=C_MUTED, fontsize=9.5)\n    ax.set_xlim(0, 104)\n    ax.set_xlabel("instances inside exploratory candidate margins (%)")\n    ax.set_title("The margin failures are removable bias, not measurement error",\n                 fontsize=11, loc="left")\n    ax.invert_yaxis()\n    _style(ax)\n    fig.tight_layout()\n    fig.savefig(path, dpi=200, facecolor="white")\n    plt.close(fig)\n\n\n# ------------------------------------------------------------------------------ main\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--in", dest="inp", default="paired_feature_table.csv")\n    ap.add_argument("--outdir", default=".")\n    ap.add_argument("--n-boot", type=int, default=2000)\n    args = ap.parse_args()\n    os.makedirs(args.outdir, exist_ok=True)\n    df = pd.read_csv(args.inp)\n    n = len(df)\n    L = ["# Agreement, Second Pass: Bias vs. Noise (n=%d)" % n, "",\n         "Supersedes the framing in `agreement_report.md`. Same data, same pipeline; the",\n         "question changed from *do the representations agree?* to *what kind of",\n         "disagreement is this, and is it removable?*", ""]\n\n    # ---- 1. bias or noise -------------------------------------------------------\n    e = df["vol_pct_err_occ_vs_mesh"].to_numpy()\n    N = float(df["occ_n_points"].iloc[0])\n    p = df["occ_fraction_inside"].to_numpy()\n    mc = np.sqrt((1 - p) / (p * N)) * 100.0            # binomial MC relative SE, percent\n    ci = boot_ci(e, np.mean, args.n_boot)\n    L += ["## 1. This is bias, not noise", "",\n          "| Quantity | Value |", "|---|---|",\n          "| mean signed error (occupancy − mesh) | **%+.3f%%** (95%% CI %+.3f to %+.3f) |" % (e.mean(), ci[0], ci[1]),\n          "| median signed error | %+.3f%% |" % np.median(e),\n          "| SD of signed error | %.3f%% |" % e.std(ddof=1),\n          "| share of instances on one side of zero | **%.2f%%** |" % (100 * max((e > 0).mean(), (e < 0).mean())),\n          "| Monte-Carlo sampling floor (median, N=%d) | %.3f%% |" % (N, np.median(mc)),\n          "| observed bias ÷ sampling floor | **%.1f×** |" % np.median(e / mc), "",\n          "A Monte-Carlo volume estimator with %d query points is entitled to about" % N,\n          "%.2f%% relative error at the median occupancy fraction. The observed" % np.median(mc),\n          "discrepancy is %.1f× that and points the same direction for %.2f%% of instances." % (np.median(e / mc), 100 * (e > 0).mean()),\n          "Sampling noise is therefore a minor component: the occupancy representation",\n          "reports a *systematically larger* mitochondrion than the mesh does.", ""]\n\n    # ---- 2. mechanism -----------------------------------------------------------\n    sv = (df["mesh_area"] / df["mesh_volume"] ** (2.0 / 3.0)).to_numpy()\n    b_sv, pred_sv, r2_sv = ols(e, sv)\n    t = ((df["occ_volume_est"] - df["mesh_volume"]) / df["mesh_area"]).to_numpy()\n    tn = t / df["occ_scale"].to_numpy()\n    L += ["## 2. The bias has the signature of a constant boundary offset", "",\n          "| Predictor of the signed error | Pearson r |", "|---|---|",\n          "| surface-to-volume ratio `A / V^(2/3)` | **%+.3f** |" % np.corrcoef(sv, e)[0, 1],\n          "| `mesh_sphericity` | %+.3f |" % np.corrcoef(df["mesh_sphericity"], e)[0, 1],\n          "| `occ_fraction_inside` | %+.3f |" % np.corrcoef(p, e)[0, 1],\n          "| `mesh_elongation` | %+.3f |" % np.corrcoef(df["mesh_elongation"], e)[0, 1], "",\n          "If one representation describes a shape uniformly offset outward by a small",\n          "distance *t*, the excess volume is ≈ *A·t*, so the percent error scales with",\n          "surface-to-volume. That is exactly the observed pattern (R² = %.2f against" % r2_sv,\n          "`A / V^(2/3)` alone). Solving for the implied offset:", "",\n          "- `t = ΔV / A` in real units: median **%.4f**, CV %.2f" % (np.median(t), t.std() / t.mean()),\n          "- the same offset divided by each instance\'s `scale`: median **%.5f**, CV %.2f" % (np.median(tn), tn.std() / tn.mean()),\n          "",\n          "The offset is far more consistent once expressed relative to `scale` (CV %.2f vs %.2f),"\n          % (tn.std() / tn.mean(), t.std() / t.mean()),\n          "i.e. it is constant in the *normalized* frame where the occupancy labels were",\n          "generated, not in real units. An offset of %.5f of the normalized extent is about" % np.median(tn),\n          "one cell of a %d³ grid (or half a cell of a %d³ grid) — consistent with the"\n          % (round(1 / np.median(tn)), round(0.5 / np.median(tn))),\n          "occupancy labels having been produced by a voxelization that counts boundary",\n          "cells as interior. **This is a dataset property worth reporting in its own right",\n          "and worth confirming against the 3DMSL generation code.**", ""]\n\n    # ---- 3. correction ----------------------------------------------------------\n    bbox = (df["occ_extent_x"] * df["occ_extent_y"] * df["occ_extent_z"]).to_numpy()\n    fill = df["occ_volume_est"].to_numpy() / bbox\n    _, pred_occ, r2_occ = ols(e, fill, p, np.log(df["occ_volume_est"]))\n    margin = np.where(p < LOW_OCC_CUT, MARGIN_LOW_OCC, MARGIN_OTHER)\n    within = lambda err: 100.0 * np.mean(np.abs(err) <= margin)\n    rates = [("no correction", within(e)),\n             ("subtract the global mean bias", within(e - e.mean())),\n             ("occupancy-only model\\n(no mesh needed)", within(e - pred_occ)),\n             ("mesh-informed A/V model\\n(upper bound)", within(e - pred_sv))]\n    L += ["## 3. Because it is bias, it is correctable", "",\n          "| Correction | R² | inside margin |", "|---|---|---|",\n          "| none | — | **%.1f%%** |" % within(e),\n          "| subtract the global mean (%+.2f%%) | — | **%.1f%%** |" % (e.mean(), within(e - e.mean())),\n          "| occupancy-only regression (bbox fill, `occ_fraction_inside`, log V) | %.2f | **%.1f%%** |" % (r2_occ, within(e - pred_occ)),\n          "| mesh-informed `A / V^(2/3)` regression | %.2f | **%.1f%%** |" % (r2_sv, within(e - pred_sv)), "",\n          "The occupancy-only model matters most: it uses nothing but quantities available",\n          "from the occupancy representation itself, so it is a usable estimator correction",\n          "rather than a post-hoc fit against the answer. Subtracting even a single global",\n          "constant recovers most of the gap, which is the clearest possible statement that",\n          "the %.1f%% of instances outside the margin were never a precision problem." % (100 - within(e)), ""]\n\n    # ---- 4. ICC and heterogeneity ----------------------------------------------\n    V = df[["mesh_volume", "occ_volume_est"]].to_numpy()\n    E = df[["mesh_extent_x", "pc_trimmed_extent_x", "occ_extent_x"]].to_numpy()\n    qv = pd.qcut(df["mesh_volume"], 10, labels=False).to_numpy()\n    qe = pd.qcut(df["mesh_extent_x"], 10, labels=False).to_numpy()\n    dec_v = [icc21(V[qv == i]) for i in range(10)]\n    dec_e = [icc21(E[qe == i]) for i in range(10)]\n    whole_v, whole_e = icc21(V), icc21(E)\n    L += ["## 4. The ICC is a property of this sample, not of the method", "",\n          "The shard spans a %.0f× range of mesh volumes. ICC(2,1) is a ratio of"\n          % (df.mesh_volume.max() / df.mesh_volume.min()),\n          "between-instance variance to total variance, so that spread inflates it toward 1",\n          "relative to the error variance. Recomputing ICC inside volume",\n          "deciles — where instances are near-identical in size — removes that inflation:", "",\n          "| Descriptor | whole sample | median within decile | decile range |", "|---|---|---|---|",\n          "| volume (mesh vs. occupancy) | **%.4f** | **%.4f** | %.3f – %.3f |"\n          % (whole_v, np.median(dec_v), min(dec_v), max(dec_v)),\n          "| extent_x (3 representations) | **%.4f** | **%.4f** | %.3f – %.3f |"\n          % (whole_e, np.median(dec_e), min(dec_e), max(dec_e)), "",\n          "Volume agreement drops from “excellent” (%.3f) to the low end of “good”"\n          % whole_v,\n          "(%.3f) on the same measurements. Koo & Li (2016) warn about exactly this."\n          % np.median(dec_v),\n          "**Recommendation: report ICC with the sample\'s size range stated alongside it,",\n          "and lead the results with the heterogeneity-free statistics below.**", ""]\n\n    # ---- 5. heterogeneity-free agreement ---------------------------------------\n    comps = [("volume: occupancy vs. mesh", df["mesh_volume"], df["occ_volume_est"]),\n             ("extent_x: point-cloud (trimmed) vs. mesh", df["mesh_extent_x"], df["pc_trimmed_extent_x"]),\n             ("extent_x: occupancy vs. mesh", df["mesh_extent_x"], df["occ_extent_x"])]\n    L += ["## 5. Difference-based agreement statistics", "",\n          "Bias here is the mean *absolute-unit* difference (and that difference as a share",\n          "of the reference mean), so it differs slightly from the mean *per-instance* percent",\n          "error in §1 — the latter weights small mitochondria equally with large ones.", "",\n          "| Comparison | bias | 95% limits of agreement | percent error | median &#124;% diff&#124; | proportional bias r |",\n          "|---|---|---|---|---|---|"]\n    for lbl, a, b in comps:\n        s = agreement_block(a, b)\n        L.append("| %s | %+.3f (%+.2f%%) | %+.3f to %+.3f | %.2f%% | %.2f%% | %+.3f |"\n                 % (lbl, s["bias"], s["bias_pct"], s["loa_lo"], s["loa_hi"],\n                    s["percent_error"], s["median_abs_pct"], s["prop_bias_r"]))\n    L += ["", "Percent error (Critchley & Critchley 1999) is the scale-free companion to the",\n          "limits of agreement and is the number to quote when readers ask how good the",\n          "method is. Volume comes out at %.1f%% — a very different message from ICC 0.994."\n          % agreement_block(df["mesh_volume"], df["occ_volume_est"])["percent_error"], ""]\n\n    # ---- 6. the point cloud has its own bias -----------------------------------\n    L += ["## 6. The trimmed point-cloud extent carries its own (smaller) bias", "",\n          "| Axis | mean error vs. mesh | SD | share one-signed |", "|---|---|---|---|"]\n    for ax in "xyz":\n        c = df["extent_%s_pct_err_pc_vs_mesh" % ax]\n        L.append("| %s | **%+.3f%%** | %.3f | %.1f%% |"\n                 % (ax, c.mean(), c.std(ddof=1), 100 * max((c > 0).mean(), (c < 0).mean())))\n    L += ["", "This is the cost of the 1st–99th-percentile trim adopted in Week 1–2: it buys",\n          "immunity to the sampling-noise outlier at the price of a consistent ~2% shrinkage,",\n          "because trimming discards real extremal surface points along with the noise. The",\n          "trade is worth making and the bias is stable enough to state as a known offset —",\n          "but it must be declared in the methods, not left implicit. The occupancy extents,",\n          "by contrast, are near-unbiased (%+.3f%% to %+.3f%%)."\n          % (min(df["extent_%s_pct_err_occ_vs_mesh" % a].mean() for a in "xyz"),\n             max(df["extent_%s_pct_err_occ_vs_mesh" % a].mean() for a in "xyz")), ""]\n\n    # ---- figures ---------------------------------------------------------------\n    f1 = os.path.join(args.outdir, "bias_not_noise.png")\n    f2 = os.path.join(args.outdir, "icc_heterogeneity.png")\n    f3 = os.path.join(args.outdir, "margin_after_correction.png")\n    fig_bias_not_noise(e, sv, (b_sv[0], b_sv[1], r2_sv), float(np.median(mc)), f1)\n    fig_icc_collapse(whole_v, dec_v, whole_e, dec_e, f2)\n    fig_correction(rates, f3)\n    L += ["## Figures", "",\n          "- `bias_not_noise.png` — one-signed error distribution; error vs. surface-to-volume",\n          "- `icc_heterogeneity.png` — whole-sample ICC vs. within-size-decile ICC",\n          "- `margin_after_correction.png` — margin pass rate before and after bias correction", ""]\n\n    out = os.path.join(args.outdir, "agreement_v2_report.md")\n    with open(out, "w") as fh:\n        fh.write("\\n".join(L))\n    print("\\n".join(L))\n    print("\\nWrote %s, %s, %s, %s" % (out, f1, f2, f3))\n\n\nif __name__ == "__main__":\n    main()')


### 13.1 Occupancy estimator repeatability

The previous `KeyError: n_points` came from running on `__MACOSX` metadata. This section now uses only the verified real shard and a new results folder. Run the corrected helper-script cell and Section 13 setup before this analysis.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import os, sys, subprocess
    from pathlib import Path
    import pandas as pd
    from IPython.display import Markdown, Image, display

    if '__MACOSX' in Path(SHARD_ROOT).parts:
        raise RuntimeError('Refusing to run repeatability on Mac ZIP metadata.')
    if not Path(SHARD_ROOT, '24553', 'occupancies.npz').is_file():
        raise FileNotFoundError('Real occupancy archives are missing; rerun Section 3.')
    script_path = Path(WORK_DIR) / 'repeatability.py'
    if not script_path.exists():
        raise FileNotFoundError('Run the %%writefile repeatability.py cell first.')
    subprocess.run([sys.executable, str(script_path), '--root', SHARD_ROOT,
                    '--outdir', OUT_DIR, '--sample', '400'], check=True)
    report = Path(OUT_DIR, 'repeatability_report.md')
    display(Markdown(report.read_text()))
    figure = Path(OUT_DIR, 'estimator_convergence.png')
    if figure.exists():
        display(Image(filename=str(figure)))


### 13.2 Extended agreement analysis

This stage reads the paired feature table and reports additional bias, heterogeneity, and correction diagnostics. In-sample correction rates remain exploratory until Section 15 evaluates held-out instances.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    get_ipython().system('python3 agreement_v2.py --in "{PAIRED}" --outdir "{OUT_DIR}"')

    display(Markdown(open(os.path.join(OUT_DIR, 'agreement_v2_report.md')).read()))
    for f in ('bias_not_noise.png', 'icc_heterogeneity.png', 'margin_after_correction.png'):
        display(Image(os.path.join(OUT_DIR, f)))


### 13.3 Save repeatability and agreement outputs to Drive

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import shutil

    os.makedirs(DRIVE_OUT, exist_ok=True)
    new = ['repeatability.csv', 'repeatability_report.md', 'estimator_convergence.png',
           'agreement_v2_report.md', 'bias_not_noise.png', 'icc_heterogeneity.png',
           'margin_after_correction.png']
    for f in new:
        src = os.path.join(OUT_DIR, f)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_OUT, f))
            print('saved %-30s %7.1f KB' % (f, os.path.getsize(src)/1024))

    for f in ('repeatability.py', 'agreement_v2.py'):
        shutil.copy2(os.path.join(WORK_DIR, f), os.path.join(DRIVE_OUT, 'scripts', f))
    print('\nSaved to', DRIVE_OUT)


## 14. Interpretation checkpoint

The earlier 3DMSL run processed 2,720 paired instances. Occupancy volume was systematically higher than mesh volume (about +3.67% mean and +3.05% median), while point-sampling repeatability was smaller. The high whole-sample ICC should be considered alongside the lower within-volume-decile ICC, because between-instance heterogeneity inflates aggregate agreement.

These results support a morphometric measurement-reliability claim. They do not establish a functional-state or cardiovascular-disease boundary.

## 15. Held-out correction validation on 3DMSL

The earlier correction analysis fitted and evaluated models on the same instances. This version separates model fitting, uncertainty calibration, and final test evaluation with a stratified 60/20/20 train/calibration/test split. The resulting uncertainty bound is an in-domain **measurement-error** bound. A second 3DMSL shard is needed for external confirmation of the original correction.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('validation_v3.py').write_text('#!/usr/bin/env python3\n"""Internal validation after exploratory development of occupancy-volume correction.\n\nFits correction parameters on training instances, selects a 95% residual bound on\ncalibration instances, and reports performance once on held-out test instances.\nThis supersedes in-sample correction rates from agreement_v2.py.\n"""\nimport argparse, json, os\nimport numpy as np\nimport pandas as pd\ntry:\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\nexcept ImportError:\n    plt=None\n\nREQ = [\'instance_id\',\'mesh_volume\',\'mesh_area\',\'occ_volume_est\',\'occ_fraction_inside\',\n       \'occ_extent_x\',\'occ_extent_y\',\'occ_extent_z\']\nCANDIDATE_MARGINS = (2.17, 5.0, 6.0, 10.0)\n\ndef stratified_split(df, seed=790):\n    """60/20/20 split within mesh-volume deciles."""\n    rng=np.random.default_rng(seed)\n    bins=pd.qcut(df[\'mesh_volume\'],10,labels=False,duplicates=\'drop\')\n    split=np.empty(len(df),dtype=object)\n    for b in sorted(pd.unique(bins)):\n        idx=np.flatnonzero(np.asarray(bins)==b); rng.shuffle(idx)\n        n=len(idx); a=int(round(.60*n)); c=int(round(.20*n))\n        split[idx[:a]]=\'train\'; split[idx[a:a+c]]=\'calibration\'; split[idx[a+c:]]=\'test\'\n    return split\n\ndef feature_matrix(df, means=None, scales=None):\n    bbox=df[\'occ_extent_x\']*df[\'occ_extent_y\']*df[\'occ_extent_z\']\n    raw=np.column_stack([\n        df[\'occ_volume_est\']/bbox.clip(lower=1e-12),\n        df[\'occ_fraction_inside\'],\n        np.log(df[\'occ_volume_est\'].clip(lower=1e-12)),\n        np.log(bbox.clip(lower=1e-12)),\n        df[[\'occ_extent_x\',\'occ_extent_y\',\'occ_extent_z\']].max(axis=1) /\n          df[[\'occ_extent_x\',\'occ_extent_y\',\'occ_extent_z\']].min(axis=1).clip(lower=1e-12),\n    ]).astype(float)\n    if means is None: means=np.nanmean(raw,axis=0)\n    if scales is None: scales=np.nanstd(raw,axis=0,ddof=1)\n    scales=np.where((scales==0)|~np.isfinite(scales),1.,scales)\n    return np.column_stack([np.ones(len(raw)),(raw-means)/scales]),means,scales\n\ndef fit_ols(X,y): return np.linalg.lstsq(X,np.asarray(y,float),rcond=None)[0]\ndef predict_pct(X,b): return np.clip(X@b,-50,100)\ndef signed_error(df, corrected=None):\n    value=df[\'occ_volume_est\'].to_numpy() if corrected is None else np.asarray(corrected)\n    return 100*(value/df[\'mesh_volume\'].to_numpy()-1)\ndef corrected_volume(df,pred_pct): return df[\'occ_volume_est\'].to_numpy()/(1+np.asarray(pred_pct)/100)\ndef conformal_q(abs_resid,coverage=.95):\n    x=np.sort(np.asarray(abs_resid,float)); n=len(x)\n    if n == 0 or not np.isfinite(x).all() or not 0 < coverage < 1:\n        raise ValueError("Finite nonempty scores and coverage in (0,1) are required")\n    k=int(np.ceil((n+1)*coverage))\n    return float(x[k-1]) if k <= n else float("inf")\ndef boot_ci(x,fn,n_boot=1000,seed=790):\n    x=np.asarray(x,float);rng=np.random.default_rng(seed)\n    z=np.array([fn(x[rng.integers(0,len(x),len(x))]) for _ in range(n_boot)])\n    return tuple(np.percentile(z,[2.5,97.5]))\ndef summarize(e,n_boot,seed):\n    e=np.asarray(e,float)\n    return dict(n=len(e),mean_bias=float(e.mean()),mean_bias_ci=boot_ci(e,np.mean,n_boot,seed),\n                median_abs=float(np.median(abs(e))),median_abs_ci=boot_ci(e,lambda x:np.median(abs(x)),n_boot,seed+1),\n                rmse=float(np.sqrt(np.mean(e**2))),loa_lo=float(e.mean()-1.96*e.std(ddof=1)),\n                loa_hi=float(e.mean()+1.96*e.std(ddof=1)))\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'--in\',dest=\'inp\',required=True);ap.add_argument(\'--outdir\',required=True)\n    ap.add_argument(\'--seed\',type=int,default=790);ap.add_argument(\'--n-boot\',type=int,default=1000)\n    a=ap.parse_args();os.makedirs(a.outdir,exist_ok=True)\n    df=pd.read_csv(a.inp);missing=[c for c in REQ if c not in df]\n    if missing: raise ValueError(\'missing columns: \'+\', \'.join(missing))\n    df=df[REQ].replace([np.inf,-np.inf],np.nan).dropna().copy()\n    df=df[(df[[\'mesh_volume\',\'occ_volume_est\',\'occ_extent_x\',\'occ_extent_y\',\'occ_extent_z\']]>0).all(axis=1)].reset_index(drop=True)\n    df[\'split\']=stratified_split(df,a.seed)\n    tr=df.split==\'train\';cal=df.split==\'calibration\';te=df.split==\'test\'\n    y=signed_error(df)\n    Xtr,mu,sd=feature_matrix(df[tr]);Xall,_,_=feature_matrix(df,mu,sd)\n    beta=fit_ols(Xtr,y[tr]);global_bias=float(y[tr].mean())\n    preds={\'uncorrected\':np.zeros(len(df)),\'global_train_mean\':np.full(len(df),global_bias),\n           \'occupancy_only_ols\':predict_pct(Xall,beta)}\n    errors={k:signed_error(df,corrected_volume(df,v)) if k!=\'uncorrected\' else y for k,v in preds.items()}\n    summaries={};bounds={}\n    for j,(name,e) in enumerate(errors.items()):\n        q=conformal_q(abs(e[cal]),.95);bounds[name]=q\n        s=summarize(e[te],a.n_boot,a.seed+10*j);s[\'calibration_q95_abs_error\']=q\n        s[\'test_coverage_at_q95\']=float(np.mean(abs(e[te])<=q))\n        s[\'within_candidate_margins\']={str(m):float(np.mean(abs(e[te])<=m)) for m in CANDIDATE_MARGINS}\n        summaries[name]=s\n    out=df[[\'instance_id\',\'split\',\'mesh_volume\',\'occ_volume_est\']].copy()\n    for name in preds:\n        out[\'predicted_bias_\'+name]=preds[name];out[\'corrected_error_\'+name]=errors[name]\n    out.to_csv(os.path.join(a.outdir,\'validation_v3_predictions.csv\'),index=False)\n    df[[\'instance_id\',\'split\']].to_csv(os.path.join(a.outdir,\'validation_v3_splits.csv\'),index=False)\n    model={\'calibration_score\':\'abs(100*(corrected_volume/mesh_volume-1))\', \'calibration_index_1based\':int(np.ceil((int(cal.sum())+1)*.95)), \'seed\':a.seed,\'features\':[\'bbox_fill\',\'occ_fraction_inside\',\'log_occ_volume\',\'log_bbox_volume\',\'occ_extent_elongation\'],\n           \'feature_means\':mu.tolist(),\'feature_scales\':sd.tolist(),\'coefficients\':beta.tolist(),\n           \'global_train_mean_bias_pct\':global_bias,\'calibration_bounds_pct\':bounds,\'summaries\':summaries}\n    with open(os.path.join(a.outdir,\'validation_v3_model.json\'),\'w\') as f:json.dump(model,f,indent=2)\n    # Test-only plot\n    names=list(errors);vals=[errors[k][te] for k in names]\n    if plt is not None:\n        fig,ax=plt.subplots(figsize=(8,4.5));ax.boxplot(vals,showfliers=False); ax.set_xticks([1,2,3]); ax.set_xticklabels([\'uncorrected\',\'global mean\',\'occupancy-only\'])\n        ax.axhline(0,color=\'black\',lw=1);ax.set_ylabel(\'held-out occupancy volume error (%)\')\n        ax.set_title(\'Internal test after exploratory development\');fig.tight_layout()\n        fig.savefig(os.path.join(a.outdir,\'validation_v3_test_errors.png\'),dpi=180);plt.close(fig)\n    L=[\'# Internal Validation Following Exploratory Development\',\'\',\n       \'The earlier correction percentages were computed on the same instances used to fit the correction. This analysis uses a stratified 60/20/20 train/calibration/test split. Model parameters are fitted on training rows; earlier full-shard exploration may have informed model design.\',\'\',\n       \'| Split | n |\',\'|---|---:|\']\n    for z in (\'train\',\'calibration\',\'test\'):L.append(\'| %s | %d |\'%(z,int((df.split==z).sum())))\n    L += [\'\', \'## Held-out test performance\',\'\',\n          \'| Method | mean bias (95% CI) | median absolute error (95% CI) | RMSE | 95% limits | calibration q95 | test coverage |\',\n          \'|---|---:|---:|---:|---:|---:|---:|\']\n    for name in names:\n        s=summaries[name];L.append(\'| %s | %+.3f [%+.3f, %+.3f] | %.3f [%.3f, %.3f] | %.3f | %+.3f to %+.3f | %.3f | %.1f%% |\'%\n        (name,s[\'mean_bias\'],*s[\'mean_bias_ci\'],s[\'median_abs\'],*s[\'median_abs_ci\'],s[\'rmse\'],s[\'loa_lo\'],s[\'loa_hi\'],s[\'calibration_q95_abs_error\'],100*s[\'test_coverage_at_q95\']))\n    L += [\'\', \'## Sensitivity to candidate error margins\',\'\', \'| Method | <=2.17% | <=5% | <=6% | <=10% |\',\'|---|---:|---:|---:|---:|\']\n    for name in names:\n        w=summaries[name][\'within_candidate_margins\'];L.append(\'| %s | %.1f%% | %.1f%% | %.1f%% | %.1f%% |\'%(name,100*w[\'2.17\'],100*w[\'5.0\'],100*w[\'6.0\'],100*w[\'10.0\']))\n    L += [\'\', \'## Interpretation guardrails\',\'\',\n          \'- The calibrated q95 is an **in-domain measurement-error bound**, not a biological or functional threshold.\',\n          \'- The 2.17% value is a Monte-Carlo repeatability coefficient estimated from stored query points; it is shown only as a sensitivity threshold.\',\n          \'- The occupancy-only correction may be deployed without mesh measurements, but it still requires validation on another 3DMSL shard.\',\n          \'- The proposed boundary-dilation mechanism remains a hypothesis until verified against the 3DMSL generation code.\',\n          \'- MitoEM can test descriptor portability, but it cannot validate this occupancy correction because it lacks paired 3DMSL occupancy labels.\',\'\']\n    with open(os.path.join(a.outdir,\'validation_v3_report.md\'),\'w\') as f:f.write(\'\\n\'.join(L))\n    print(\'\\n\'.join(L));print(\'\\nWrote validation_v3 outputs to\',a.outdir)\nif __name__==\'__main__\':main()\n')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    from pathlib import Path
    from IPython.display import display, Markdown, Image

    if len(pd.read_csv(PAIRED)) < 2700:
        raise RuntimeError('Run the verified 3DMSL audit before held-out validation.')
    subprocess.run([sys.executable, 'validation_v3.py', '--in', PAIRED,
                    '--outdir', OUT_DIR, '--n-boot', '2000', '--seed', '2026'], check=True)
    validation_report = Path(OUT_DIR) / 'validation_v3_report.md'
    display(Markdown(validation_report.read_text()))
    validation_figure = Path(OUT_DIR) / 'validation_v3_test_errors.png'
    if validation_figure.exists():
        display(Image(filename=str(validation_figure)))


## 16. Reliability across 3DMSL imaging orientations and configurations

Analyze each rendered view separately, then estimate within-configuration orientation variability and agreement across Conf1, Conf2, and Epi1. Run the 400-instance sample before an optional full-shard analysis. Per-image normalized intensity does not represent calibrated fluorescence.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('imaging_condition_reliability.py').write_text('#!/usr/bin/env python3\n"""Quantify orientation and microscope-configuration sensitivity in 3DMSL renders."""\nimport argparse, os, re, time\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nimport numpy as np\nimport pandas as pd\nfrom hit_common import IMG_CONFIGS, find_instance_dirs, images_for_config, instance_id\nfrom image_features import _view_features\n\nFEATURES=(\'fg_fraction\',\'fg_bbox_area_frac\',\'mean\',\'std\',\'otsu\')\n\ndef view_name(path):\n    s=os.path.splitext(os.path.basename(path))[0]\n    m=re.search(r\'_(0|x90|x180|y90|y180|z90)$\',s)\n    return m.group(1) if m else s\n\ndef one_instance(d):\n    rows=[]\n    for c in IMG_CONFIGS:\n        for f in images_for_config(d,c):\n            z=_view_features(f);z.update(instance_id=instance_id(d),config=c,view=view_name(f),file=os.path.basename(f));rows.append(z)\n    return rows\n\ndef safe(d):\n    try:return one_instance(d)\n    except Exception as e:return [dict(instance_id=instance_id(d),error=str(e))]\n\ndef icc21(M):\n    M=np.asarray(M,float);n,k=M.shape\n    if n<2 or k<2:return np.nan\n    g=M.mean();rm=M.mean(1);cm=M.mean(0)\n    MSR=k*((rm-g)**2).sum()/(n-1);MSC=n*((cm-g)**2).sum()/(k-1)\n    MSE=((M-rm[:,None]-cm[None,:]+g)**2).sum()/((n-1)*(k-1))\n    den=MSR+(k-1)*MSE+k*(MSC-MSE)/n\n    return float((MSR-MSE)/den) if den else np.nan\n\ndef boot_icc(M,n=1000,seed=790):\n    rng=np.random.default_rng(seed);vals=[]\n    for _ in range(n):\n        v=icc21(M[rng.integers(0,len(M),len(M))])\n        if np.isfinite(v):vals.append(v)\n    return tuple(np.percentile(vals,[2.5,97.5]))\ndef rankcorr(x,y):\n    a=pd.Series(x).rank().to_numpy();b=pd.Series(y).rank().to_numpy()\n    return float(np.corrcoef(a,b)[0,1]) if a.std() and b.std() else np.nan\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'--root\',required=True);ap.add_argument(\'--outdir\',required=True)\n    ap.add_argument(\'--sample\',type=int,default=400);ap.add_argument(\'--workers\',type=int,default=2);ap.add_argument(\'--n-boot\',type=int,default=1000)\n    a=ap.parse_args();os.makedirs(a.outdir,exist_ok=True)\n    dirs=find_instance_dirs(a.root);dirs=dirs[:a.sample] if a.sample else dirs\n    rows=[];t=time.time()\n    if a.workers<=1:\n        for d in dirs:rows.extend(safe(d))\n    else:\n        with ProcessPoolExecutor(max_workers=a.workers) as ex:\n            fs=[ex.submit(safe,d) for d in dirs]\n            for i,f in enumerate(as_completed(fs),1):\n                rows.extend(f.result())\n                if i%100==0 or i==len(fs):print(\'%d/%d\'%(i,len(fs)),flush=True)\n    raw=pd.DataFrame(rows);raw.to_csv(os.path.join(a.outdir,\'imaging_view_features.csv\'),index=False)\n    ok=raw.dropna(subset=[\'config\',\'view\']).copy()\n    agg=ok.groupby([\'instance_id\',\'config\'])[list(FEATURES)].agg([\'mean\',\'std\'])\n    flat=pd.DataFrame(index=agg.index)\n    for f in FEATURES:\n        flat[f+\'_mean\']=agg[(f,\'mean\')];flat[f+\'_orientation_cv\']=agg[(f,\'std\')]/agg[(f,\'mean\')].abs().clip(lower=1e-12)\n    flat=flat.reset_index();flat.to_csv(os.path.join(a.outdir,\'imaging_condition_summary.csv\'),index=False)\n    L=[\'# Imaging-Condition Reliability\',\'\',\n       \'This stage measures two distinct effects: variation across the six orientations within a configuration, and agreement across Conf1, Conf2, and Epi1 after averaging orientations.\',\'\',\n       \'## Orientation sensitivity\',\'\', \'| Configuration | feature | median CV | 95th-percentile CV |\',\'|---|---|---:|---:|\']\n    for c in IMG_CONFIGS:\n        d=flat[flat.config==c]\n        for f in (\'fg_fraction\',\'fg_bbox_area_frac\'):\n            x=d[f+\'_orientation_cv\'].replace([np.inf,-np.inf],np.nan).dropna()\n            L.append(\'| %s | %s | %.3f | %.3f |\'%(c,f,x.median(),x.quantile(.95)))\n    L += [\'\', \'## Cross-configuration agreement\',\'\',\n          \'| Feature | complete n | ICC(2,1) | 95% bootstrap CI | lowest pairwise Spearman |\',\'|---|---:|---:|---:|---:|\']\n    for j,f in enumerate((\'fg_fraction\',\'fg_bbox_area_frac\')):\n        w=flat.pivot(index=\'instance_id\',columns=\'config\',values=f+\'_mean\').dropna()\n        cols=[c for c in IMG_CONFIGS if c in w];M=w[cols].to_numpy();ic=icc21(M);ci=boot_icc(M,a.n_boot,790+j)\n        rs=[rankcorr(w[x],w[y]) for q,x in enumerate(cols) for y in cols[q+1:]]\n        L.append(\'| %s | %d | %.3f | [%.3f, %.3f] | %.3f |\'%(f,len(w),ic,ci[0],ci[1],min(rs)))\n    L += [\'\', \'## Interpretation guardrails\',\'\',\n          \'- These are measurement-reliability results. They do not establish mitochondrial function.\',\n          \'- Per-image min-max intensity normalization means intensity means and thresholds should not be compared as calibrated physical intensities.\',\n          \'- A constant or saturated Epi1 bounding-box feature is an assay failure for that descriptor, not evidence of invariance.\',\n          \'- Run the sample first. If the results are stable and QC is acceptable, rerun with `--sample 0` for the full shard.\',\'\']\n    with open(os.path.join(a.outdir,\'imaging_condition_report.md\'),\'w\') as f:f.write(\'\\n\'.join(L))\n    print(\'\\n\'.join(L));print(\'Completed in %.1fs\'%(time.time()-t))\nif __name__==\'__main__\':main()\n')


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    import subprocess, sys
    from pathlib import Path
    from IPython.display import display, Markdown

    if '__MACOSX' in Path(SHARD_ROOT).parts:
        raise RuntimeError('Refusing to analyze Mac ZIP metadata.')
    subprocess.run([sys.executable, 'imaging_condition_reliability.py', '--root', SHARD_ROOT,
                    '--outdir', OUT_DIR, '--sample', '400', '--workers', '2',
                    '--n-boot', '1000'], check=True)
    imaging_report = Path(OUT_DIR) / 'imaging_condition_report.md'
    display(Markdown(imaging_report.read_text()))


## 17. MitoEM-R external-domain analysis

**Input ZIPs in your Google Drive**

- Images: `/content/drive/MyDrive/im.zip`, containing `im/im0000.png` to `im/im0999.png`.
- Revised v2 instance labels: `/content/drive/MyDrive/EM30-R-mito-train-val-v2.zip`, containing `seg0000.tif` to `seg0499.tif` in training (0–399) and validation (400–499) folders.

The pipeline verifies archive indexes and image–label dimensions, uses native voxel spacing **30 × 8 × 8 nm (z × y × x)**, excludes instances cut by analysis or image boundaries, calculates instance-level voxel morphometry, and compares a size-balanced sample of voxel masks with meshes and sampled surface points. It reads ZIP members directly, so the roughly 30 GB of uncompressed images and labels do not need to be extracted.

Run a **pilot on training slices 0–29** to check conversion and outputs. Then run the **prespecified validation section 400–499** without changing the analysis based on its results. All three representations are generated from the same MitoEM segmentation; this tests processing portability and representation sensitivity, not the numerical 3DMSL occupancy correction.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    # Self-contained MitoEM setup; run this even if the earlier 3DMSL cells were skipped.
    from google.colab import drive
    from pathlib import Path
    import os, shutil, zipfile
    from IPython.display import display, Markdown, Image

    get_ipython().system('pip install -q trimesh scikit-image scipy')
    drive.mount('/content/drive', force_remount=False)
    MITOEM_IMAGE_ZIP = Path('/content/drive/MyDrive/im.zip')
    MITOEM_LABEL_ZIP_DRIVE = Path('/content/drive/MyDrive/EM30-R-mito-train-val-v2.zip')
    MITOEM_WORK = Path('/content/hit')
    MITOEM_OUT = Path('/content/hit_outputs_reproduced/mitoem_R_external')
    MITOEM_DRIVE_OUT = Path('/content/drive/MyDrive/HIT_outputs_reproduced/mitoem_R_external')
    MITOEM_WORK.mkdir(parents=True, exist_ok=True)
    MITOEM_OUT.mkdir(parents=True, exist_ok=True)

    for path in (MITOEM_IMAGE_ZIP, MITOEM_LABEL_ZIP_DRIVE):
        if not path.is_file():
            raise FileNotFoundError(f'Missing Drive archive: {path}')
        print(f'{path.name}: {path.stat().st_size / 1e9:.2f} GB')

    # The much smaller label ZIP is copied to Colab's local disk for faster repeated reads.
    MITOEM_LABEL_ZIP = MITOEM_WORK / MITOEM_LABEL_ZIP_DRIVE.name
    if not MITOEM_LABEL_ZIP.exists() or MITOEM_LABEL_ZIP.stat().st_size != MITOEM_LABEL_ZIP_DRIVE.stat().st_size:
        shutil.copy2(MITOEM_LABEL_ZIP_DRIVE, MITOEM_LABEL_ZIP)

    with zipfile.ZipFile(MITOEM_IMAGE_ZIP) as images, zipfile.ZipFile(MITOEM_LABEL_ZIP) as labels:
        print('ZIP members:', len(images.namelist()), 'image archive;', len(labels.namelist()), 'label archive')
    print('Local label ZIP ready:', MITOEM_LABEL_ZIP)
    os.chdir(MITOEM_WORK)


### 17.1 Write the MitoEM analysis script

The script performs paired ZIP and label checks, computes physical-size descriptors, creates an overlay for visual QC, and reports voxel/mesh/point-cloud agreement. The training pilot and validation run save to separate folders.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    from pathlib import Path
    Path('mitoem_external_validation.py').write_text('#!/usr/bin/env python3\n"""MitoEM-R external-domain morphometry audit from the original ZIP archives.\n\nThe labeled validation section (z=400..499) is read without extracting either\narchive. Voxel descriptors are calculated for all complete instances, then a\nsize-balanced sample is converted to meshes and surface point clouds. These\nrepresentations all derive from the same annotation, so the experiment measures\npipeline portability and representation sensitivity, not independent biological\nground truth or the specific 3DMSL occupancy-generation error.\n"""\n\nimport argparse\nimport json\nimport re\nfrom collections import Counter\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\n\nSPACING_UM = (0.030, 0.008, 0.008)  # z, y, x; MitoEM-R native resolution\nVOXEL_UM3 = float(np.prod(SPACING_UM))\nIMAGE_RE = re.compile(r"(?:^|/)im(\\d{4})\\.png$")\nLABEL_RE = re.compile(r"(?:^|/)seg(\\d{4})\\.tif$")\n\n\ndef catalog(archive, pattern):\n    found = {}\n    for name in archive.namelist():\n        if name.startswith("__MACOSX/") or "/._" in name:\n            continue\n        match = pattern.search(name)\n        if match:\n            index = int(match.group(1))\n            if index in found:\n                raise ValueError(f"Duplicate slice {index} in {archive.filename}")\n            found[index] = name\n    return found\n\n\ndef read_slice(archive, name):\n    with archive.open(name) as stream:\n        image = Image.open(stream)\n        image.load()\n        return np.asarray(image).copy()\n\n\ndef pixel_bounds(labels, active_ids):\n    """Return per-label XY bounds; scipy is fast, NumPy remains a fallback."""\n    try:\n        from scipy import ndimage as ndi\n\n        objects = ndi.find_objects(labels)\n        for instance_id in active_ids:\n            box = objects[instance_id - 1]\n            if box is not None:\n                yield int(instance_id), box[0].start, box[0].stop, box[1].start, box[1].stop\n    except ImportError:\n        y, x = np.nonzero(labels)\n        ids = labels[y, x]\n        size = int(ids.max()) + 1\n        min_y = np.full(size, labels.shape[0], dtype=np.int32)\n        min_x = np.full(size, labels.shape[1], dtype=np.int32)\n        max_y = np.zeros(size, dtype=np.int32)\n        max_x = np.zeros(size, dtype=np.int32)\n        np.minimum.at(min_y, ids, y)\n        np.minimum.at(min_x, ids, x)\n        np.maximum.at(max_y, ids, y + 1)\n        np.maximum.at(max_x, ids, x + 1)\n        for instance_id in active_ids:\n            yield int(instance_id), int(min_y[instance_id]), int(max_y[instance_id]), int(min_x[instance_id]), int(max_x[instance_id])\n\n\ndef audit_pair(image_zip, label_zip, image_names, label_names, indices, outdir):\n    samples = sorted(set((indices[0], indices[len(indices) // 2], indices[-1])))\n    audit = []\n    for z in samples:\n        image = read_slice(image_zip, image_names[z])\n        label = read_slice(label_zip, label_names[z])\n        if image.shape != label.shape:\n            raise ValueError(f"Image/label shape mismatch at z={z}: {image.shape} vs {label.shape}")\n        if image.shape != (4096, 4096):\n            raise ValueError(f"Unexpected slice shape at z={z}: {image.shape}")\n        if not np.issubdtype(label.dtype, np.integer):\n            raise ValueError(f"Labels are not integer instance IDs at z={z}: {label.dtype}")\n        values = np.unique(label)\n        audit.append({"z": z, "shape": list(label.shape), "image_dtype": str(image.dtype),\n                      "label_dtype": str(label.dtype), "nonzero_labels": int(np.count_nonzero(values)),\n                      "labeled_fraction": float(np.mean(label > 0))})\n        if z == samples[1]:\n            scale = 8\n            gray = image[::scale, ::scale]\n            binary = label[::scale, ::scale] > 0\n            rgb = np.repeat(gray[..., None], 3, axis=2)\n            rgb[binary, 0] = np.maximum(rgb[binary, 0], 210)\n            rgb[binary, 1] = (rgb[binary, 1] // 2)\n            rgb[binary, 2] = (rgb[binary, 2] // 2)\n            Image.fromarray(rgb.astype(np.uint8)).save(outdir / f"mitoem_R_overlay_z{z:04d}.png")\n    return audit\n\n\ndef scan_instances(label_zip, label_names, indices, min_voxels, min_z_planes, max_roi_voxels):\n    records = {}\n    for n, z in enumerate(indices, 1):\n        label = read_slice(label_zip, label_names[z])\n        counts = np.bincount(label.ravel())\n        active = np.flatnonzero(counts[1:]) + 1\n        for instance_id, y0, y1, x0, x1 in pixel_bounds(label, active):\n            count = int(counts[instance_id])\n            if instance_id not in records:\n                records[instance_id] = dict(instance_id=instance_id, voxel_count=0,\n                                            z0=z, z1=z + 1, y0=y0, y1=y1, x0=x0, x1=x1)\n            row = records[instance_id]\n            row["voxel_count"] += count\n            row["z0"] = min(row["z0"], z)\n            row["z1"] = max(row["z1"], z + 1)\n            row["y0"] = min(row["y0"], y0)\n            row["y1"] = max(row["y1"], y1)\n            row["x0"] = min(row["x0"], x0)\n            row["x1"] = max(row["x1"], x1)\n        if n % 10 == 0 or n == len(indices):\n            print(f"Scanned label slices: {n}/{len(indices)}", flush=True)\n    rows = pd.DataFrame.from_records(list(records.values())).sort_values("instance_id")\n    if rows.empty:\n        raise ValueError("No labeled mitochondria found in the requested slice range")\n    rows["volume_um3"] = rows.voxel_count * VOXEL_UM3\n    rows["z_extent_um"] = (rows.z1 - rows.z0) * SPACING_UM[0]\n    rows["y_extent_um"] = (rows.y1 - rows.y0) * SPACING_UM[1]\n    rows["x_extent_um"] = (rows.x1 - rows.x0) * SPACING_UM[2]\n    rows["bbox_fill"] = rows.volume_um3 / (rows.z_extent_um * rows.y_extent_um * rows.x_extent_um)\n    rows["touch_z_boundary"] = (rows.z0 == indices[0]) | (rows.z1 == indices[-1] + 1)\n    rows["touch_xy_boundary"] = (rows.y0 == 0) | (rows.x0 == 0) | (rows.y1 == 4096) | (rows.x1 == 4096)\n    rows["roi_voxels"] = (rows.z1 - rows.z0 + 2) * (rows.y1 - rows.y0 + 2) * (rows.x1 - rows.x0 + 2)\n    rows["complete_for_morphometry"] = (\n        ~rows.touch_z_boundary & ~rows.touch_xy_boundary &\n        (rows.voxel_count >= min_voxels) &\n        (rows.z1 - rows.z0 >= min_z_planes) &\n        (rows.roi_voxels <= max_roi_voxels)\n    )\n    return rows\n\n\ndef balanced_sample(rows, n, seed):\n    eligible = rows[rows.complete_for_morphometry].copy()\n    if eligible.empty or n <= 0:\n        return eligible.iloc[:0]\n    if len(eligible) <= n:\n        return eligible\n    eligible["size_quartile"] = pd.qcut(eligible.voxel_count, 4, labels=False, duplicates="drop")\n    rng = np.random.default_rng(seed)\n    pools = {int(q): rng.permutation(group.index.to_numpy()).tolist()\n             for q, group in eligible.groupby("size_quartile")}\n    chosen = []\n    while len(chosen) < n and any(pools.values()):\n        for q in sorted(pools):\n            if pools[q] and len(chosen) < n:\n                chosen.append(pools[q].pop())\n    return eligible.loc[chosen].sort_values("instance_id")\n\n\ndef build_masks(label_zip, label_names, selected):\n    objects = {}\n    for row in selected.itertuples():\n        shape = (row.z1 - row.z0 + 2, row.y1 - row.y0 + 2, row.x1 - row.x0 + 2)\n        objects[int(row.instance_id)] = (row, np.zeros(shape, dtype=bool))\n    if not objects:\n        return objects\n    first = min(row.z0 for row, _ in objects.values())\n    last = max(row.z1 for row, _ in objects.values())\n    for z in range(first, last):\n        current = [(row, mask) for row, mask in objects.values() if row.z0 <= z < row.z1]\n        if not current:\n            continue\n        label = read_slice(label_zip, label_names[z])\n        for row, mask in current:\n            mask[z - row.z0 + 1, 1:-1, 1:-1] = (label[row.y0:row.y1, row.x0:row.x1] == row.instance_id)\n    return objects\n\n\ndef voxel_surface_um2(mask):\n    z_area = SPACING_UM[1] * SPACING_UM[2]\n    y_area = SPACING_UM[0] * SPACING_UM[2]\n    x_area = SPACING_UM[0] * SPACING_UM[1]\n    return float(np.count_nonzero(mask[1:] != mask[:-1]) * z_area +\n                 np.count_nonzero(mask[:, 1:] != mask[:, :-1]) * y_area +\n                 np.count_nonzero(mask[:, :, 1:] != mask[:, :, :-1]) * x_area)\n\n\ndef point_extent_error(vertices, faces, mesh_extent, rng, count, repeats):\n    triangles = vertices[faces]\n    cross = np.cross(triangles[:, 1] - triangles[:, 0], triangles[:, 2] - triangles[:, 0])\n    area = np.linalg.norm(cross, axis=1)\n    valid = area > 0\n    if not np.any(valid):\n        return np.nan, np.nan\n    triangles = triangles[valid]\n    probabilities = area[valid] / area[valid].sum()\n    raw, trimmed = [], []\n    for _ in range(repeats):\n        t = triangles[rng.choice(len(triangles), size=count, p=probabilities)]\n        u = np.sqrt(rng.random(count))[:, None]\n        v = rng.random(count)[:, None]\n        points = (1 - u) * t[:, 0] + u * (1 - v) * t[:, 1] + u * v * t[:, 2]\n        raw_extent = np.ptp(points, axis=0)\n        low, high = np.quantile(points, [0.025, 0.975], axis=0)\n        raw.append(np.mean(100 * (raw_extent / mesh_extent - 1)))\n        trimmed.append(np.mean(100 * ((high - low) / mesh_extent - 1)))\n    return float(np.mean(raw)), float(np.mean(trimmed))\n\n\ndef mesh_comparison(objects, seed, point_count, point_repeats):\n    if not objects:\n        return pd.DataFrame()\n    try:\n        from skimage.measure import marching_cubes\n        import trimesh\n    except ImportError as exc:\n        raise RuntimeError("Mesh comparison needs scikit-image and trimesh; run the notebook\'s dependency cell") from exc\n    results = []\n    rng = np.random.default_rng(seed)\n    for index, (instance_id, (row, mask)) in enumerate(objects.items(), 1):\n        record = {"instance_id": instance_id, "voxel_count": int(row.voxel_count),\n                  "voxel_volume_um3": float(row.volume_um3),\n                  "voxel_surface_um2": voxel_surface_um2(mask)}\n        try:\n            vertices, faces, _, _ = marching_cubes(mask.astype(np.uint8), level=0.5, spacing=SPACING_UM)\n            mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)\n            record["mesh_watertight"] = bool(mesh.is_watertight)\n            record["mesh_faces"] = int(len(faces))\n            if not mesh.is_watertight:\n                raise ValueError("Mesh was not watertight")\n            record["mesh_volume_um3"] = float(abs(mesh.volume))\n            record["mesh_surface_um2"] = float(mesh.area)\n            record["mesh_vs_voxel_volume_pct"] = 100 * (record["mesh_volume_um3"] / row.volume_um3 - 1)\n            record["mesh_vs_voxel_surface_pct"] = 100 * (record["mesh_surface_um2"] / record["voxel_surface_um2"] - 1)\n            extent = np.ptp(vertices, axis=0)\n            raw, trimmed = point_extent_error(vertices, faces, extent, rng, point_count, point_repeats)\n            record["point_raw_extent_error_pct"] = raw\n            record["point_trimmed_extent_error_pct"] = trimmed\n            record["error"] = ""\n        except Exception as exc:\n            record["error"] = f"{type(exc).__name__}: {exc}"\n        results.append(record)\n        print(f"Meshed instances: {index}/{len(objects)}", flush=True)\n    return pd.DataFrame(results)\n\n\ndef metric_line(values):\n    x = pd.Series(values, dtype=float).dropna().to_numpy()\n    if not len(x):\n        return "unavailable"\n    return f"median {np.median(x):+.2f}%; 5th–95th percentile {np.percentile(x, 5):+.2f}% to {np.percentile(x, 95):+.2f}%"\n\n\ndef make_report(audit, rows, selected, agreement, args, outdir):\n    success = agreement[agreement.error.eq("")] if not agreement.empty else agreement\n    phase = "validation" if args.start >= 400 else "training-slice pilot"\n    lines = [f"# MitoEM-R {phase} morphometry audit", "",\n             f"Analyzed labeled slices **{args.start}–{args.stop - 1}** at 30 × 8 × 8 nm voxel spacing (z × y × x).",\n             "Image and label ZIPs were read directly; no full-volume extraction was needed.", "",\n             "## Image–label audit", "",\n             "| Slice | Shape | Image type | Label type | Instance IDs | Labeled fraction |",\n             "|---:|---:|---|---|---:|---:|"]\n    for row in audit:\n        lines.append(f"| {row[\'z\']} | {row[\'shape\'][0]} × {row[\'shape\'][1]} | {row[\'image_dtype\']} | "\n                     f"{row[\'label_dtype\']} | {row[\'nonzero_labels\']} | {row[\'labeled_fraction\']:.3f} |")\n    lines += ["", "## Instance selection", "",\n              f"- Unique instance IDs intersecting this slab: **{len(rows)}**.",\n              f"- Complete instances passing ≥{args.min_voxels:,} voxels, ≥{args.min_z_planes} z planes, "\n              f"no slab/image-edge contact, and ROI ≤{args.max_roi_voxels:,} voxels: **{int(rows.complete_for_morphometry.sum())}**.",\n              f"- Size-balanced sample requested: **{args.n_mesh}**; selected: **{len(selected)}**; successful mesh comparisons: **{len(success)}**.", ""]\n    if len(success):\n        lines += ["## Representation agreement in the selected instances", "",\n                  f"- Mesh versus source voxel volume: {metric_line(success.mesh_vs_voxel_volume_pct)}.",\n                  f"- Mesh versus voxel-face surface area: {metric_line(success.mesh_vs_voxel_surface_pct)}.",\n                  f"- Point-cloud raw extent versus mesh extent: {metric_line(success.point_raw_extent_error_pct)}.",\n                  f"- Point-cloud 2.5–97.5% trimmed extent versus mesh extent: {metric_line(success.point_trimmed_extent_error_pct)}.", "",\n                  "| Candidate absolute volume margin | Fraction of successful mesh comparisons within margin |",\n                  "|---:|---:|"]\n        for margin in (2.17, 5.0, 6.0, 10.0):\n            fraction = np.mean(np.abs(success.mesh_vs_voxel_volume_pct) <= margin)\n            lines.append(f"| {margin:.2f}% | {fraction:.1%} |")\n    lines += ["", "## Interpretation", "",\n              "The voxel mask, mesh, and sampled surface points all derive from the same MitoEM annotation. "\n              "Their agreement measures representation sensitivity of this processing pipeline on real EM data. "\n              "It does not test independent acquisition agreement or reproduce the 3DMSL occupancy query process.", "",\n              "Candidate margins above are exploratory sensitivity checks. Choose and freeze a primary margin "\n              "with the adviser before a confirmatory external analysis. The 2.17% value originates in the "\n              "3DMSL sampling repeatability calculation and is not a biological threshold.", "",\n              "This is one rat cortex tissue volume; instances within it are spatially dependent. "\n              "No mitochondrial function, cardiovascular outcome, or independent animal replicate is present. "\n              "Annotation errors and the challenge\'s minimum annotated size can alter the observed size distribution.", ""]\n    (outdir / "mitoem_R_report.md").write_text("\\n".join(lines))\n    print("\\n".join(lines))\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--image-zip", type=Path, required=True)\n    parser.add_argument("--label-zip", type=Path, required=True)\n    parser.add_argument("--outdir", type=Path, required=True)\n    parser.add_argument("--start", type=int, default=400)\n    parser.add_argument("--stop", type=int, default=500, help="exclusive z index")\n    parser.add_argument("--n-mesh", type=int, default=40)\n    parser.add_argument("--min-voxels", type=int, default=2000)\n    parser.add_argument("--min-z-planes", type=int, default=3)\n    parser.add_argument("--max-roi-voxels", type=int, default=2_000_000)\n    parser.add_argument("--point-count", type=int, default=10_000)\n    parser.add_argument("--point-repeats", type=int, default=5)\n    parser.add_argument("--seed", type=int, default=2026)\n    args = parser.parse_args()\n    if not (0 <= args.start < args.stop <= 500):\n        raise ValueError("Labeled slices must be inside 0–499")\n    args.outdir.mkdir(parents=True, exist_ok=True)\n    with ZipFile(args.image_zip) as image_zip, ZipFile(args.label_zip) as label_zip:\n        image_names = catalog(image_zip, IMAGE_RE)\n        label_names = catalog(label_zip, LABEL_RE)\n        if set(image_names) != set(range(1000)):\n            raise ValueError(f"Expected image slices 0–999; found {len(image_names)}")\n        if set(label_names) != set(range(500)):\n            raise ValueError(f"Expected label slices 0–499; found {len(label_names)}")\n        for z in range(400):\n            if "mito-train-v2" not in label_names[z]:\n                raise ValueError(f"Incorrect training label path at z={z}")\n        for z in range(400, 500):\n            if "mito-val-v2" not in label_names[z]:\n                raise ValueError(f"Incorrect validation label path at z={z}")\n        indices = list(range(args.start, args.stop))\n        audit = audit_pair(image_zip, label_zip, image_names, label_names, indices, args.outdir)\n        rows = scan_instances(label_zip, label_names, indices, args.min_voxels,\n                              args.min_z_planes, args.max_roi_voxels)\n        rows.to_csv(args.outdir / "mitoem_R_instance_features.csv", index=False)\n        selected = balanced_sample(rows, args.n_mesh, args.seed)\n        if len(selected):\n            objects = build_masks(label_zip, label_names, selected)\n            agreement = mesh_comparison(objects, args.seed, args.point_count, args.point_repeats)\n        else:\n            agreement = pd.DataFrame(columns=["instance_id", "error"])\n        agreement.to_csv(args.outdir / "mitoem_R_representation_agreement.csv", index=False)\n        metadata = {"image_zip": str(args.image_zip), "label_zip": str(args.label_zip),\n                    "image_slice_count": len(image_names), "label_slice_count": len(label_names),\n                    "analysis_z_start": args.start, "analysis_z_stop_exclusive": args.stop,\n                    "voxel_spacing_um_zyx": SPACING_UM, "seed": args.seed,\n                    "n_instances_intersecting_slab": len(rows),\n                    "n_complete_eligible": int(rows.complete_for_morphometry.sum()),\n                    "n_mesh_selected": len(selected), "audit_samples": audit,\n                    "settings": {key: value for key, value in vars(args).items() if key not in ("image_zip", "label_zip", "outdir")}}\n        (args.outdir / "mitoem_R_audit.json").write_text(json.dumps(metadata, indent=2))\n        make_report(audit, rows, selected, agreement, args, args.outdir)\n\n\nif __name__ == "__main__":\n    main()')


### 17.2 Training-slice pilot

This checks the code and alignment on slices 0–29. The pilot output is for debugging and should not be reported as independent validation.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    PILOT_OUT = MITOEM_OUT / 'pilot_train_000_029'
    PILOT_OUT.mkdir(parents=True, exist_ok=True)
    get_ipython().system('python3 mitoem_external_validation.py      --image-zip "{MITOEM_IMAGE_ZIP}"      --label-zip "{MITOEM_LABEL_ZIP}"      --outdir "{PILOT_OUT}"      --start 0 --stop 30 --n-mesh 4 --seed 2026')

    display(Markdown((PILOT_OUT / 'mitoem_R_report.md').read_text()))


### 17.3 Held-out MitoEM-R validation section

Run on the labeled validation slices 400–499 after the pilot succeeds. The script samples 40 eligible complete instances across size quartiles for the mesh comparison, while calculating voxel descriptors for every detected instance. It reports descriptive performance within this one volume; the candidate margins are sensitivity checks until a primary margin is frozen with the adviser.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    VALIDATION_OUT = MITOEM_OUT / 'validation_400_499'
    VALIDATION_OUT.mkdir(parents=True, exist_ok=True)
    get_ipython().system('python3 mitoem_external_validation.py      --image-zip "{MITOEM_IMAGE_ZIP}"      --label-zip "{MITOEM_LABEL_ZIP}"      --outdir "{VALIDATION_OUT}"      --start 400 --stop 500 --n-mesh 40 --seed 2026')

    display(Markdown((VALIDATION_OUT / 'mitoem_R_report.md').read_text()))
    for figure in sorted(VALIDATION_OUT.glob('mitoem_R_overlay_*.png')):
        display(Image(filename=str(figure)))


In [ ]:
if RUN_ORIGINAL_PIPELINE:
    # Keep the external-domain results and exact analysis script in Drive.
    MITOEM_DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    for folder in (PILOT_OUT, VALIDATION_OUT):
        if folder.exists():
            shutil.copytree(folder, MITOEM_DRIVE_OUT / folder.name, dirs_exist_ok=True)
    shutil.copy2(MITOEM_WORK / 'mitoem_external_validation.py', MITOEM_DRIVE_OUT / 'mitoem_external_validation.py')
    print('Saved MitoEM results to', MITOEM_DRIVE_OUT)


## 18. Manuscript endpoint and claim boundaries

The paper should present the 3DMSL integrity audit, uncorrected representation agreement, repeatability, held-out correction performance, and imaging-condition sensitivity. MitoEM-R adds an **external-domain processing test** on real 3D electron microscopy labels; a second 3DMSL shard remains the direct replication test for the fitted 3DMSL occupancy correction.

The MitoEM labels cover one rat cortex volume. Individual mitochondria within it are spatially related and do not represent independent animals. This dataset has no direct mitochondrial functional readouts or cardiovascular disease outcomes. Frame cardiovascular relevance as motivation for a later disease-linked study.

For a stronger follow-up external-domain comparison, process the matching labeled human MitoEM volume with the same frozen code and physical-unit rules.

In [ ]:
if RUN_ORIGINAL_PIPELINE:
    # Save the corrected 3DMSL outputs and scripts alongside earlier results.
    from pathlib import Path
    from shutil import copy2

    DRIVE_OUT = '/content/drive/MyDrive/HIT_outputs_reproduced'
    Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
    extra_outputs = [
        'validation_v3_predictions.csv',
        'validation_v3_splits.csv',
        'validation_v3_model.json',
        'validation_v3_report.md',
        'validation_v3_test_errors.png',
        'imaging_view_features.csv',
        'imaging_condition_summary.csv',
        'imaging_condition_report.md',
    ]
    for filename in extra_outputs:
        source = Path(OUT_DIR) / filename
        if source.exists():
            destination = Path(DRIVE_OUT) / filename
            copy2(source, destination)
            print('Saved:', destination)
    scripts_out = Path(DRIVE_OUT) / 'scripts'
    scripts_out.mkdir(parents=True, exist_ok=True)
    for filename in ('validation_v3.py', 'imaging_condition_reliability.py'):
        source = Path(WORK_DIR) / filename
        if source.exists():
            copy2(source, scripts_out / filename)
    print('3DMSL outputs saved to', DRIVE_OUT)


## 19. Frozen-model evaluation on `1_1_2729.zip`

The following cells run regardless of the development switch. They use your original saved model and scripts, preserve its coefficients and calibration bounds, verify disjoint IDs, audit the new shard, and save predictions, uncertainty summaries, subgroup results, and a figure. No model training occurs here.

Missing original artifacts stop the run. Restore those artifacts from the completed analysis rather than fitting a new model on the validation shard. Confidence intervals assume object-level independence; source geometry duplication and biological provenance remain unverified.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, shutil, subprocess, sys, zipfile, datetime
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
ZIP_PATH = Path('/content/drive/MyDrive/1_1_2729.zip')
SOURCE = Path('/content/drive/MyDrive/HIT_outputs_verified')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
WORK = Path('/content/hit_frozen_validation') / RUN_ID
RESULTS = SOURCE / 'frozen_validation_1_1_2729' / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
RESULTS.mkdir(parents=True, exist_ok=False)
required = ['validation_v3_model.json', 'validation_v3_splits.csv', 'repeatability.csv',
            'scripts/hit_common.py', 'scripts/audit_3dmsl.py']
for p in [ZIP_PATH] + [SOURCE / f for f in required]:
    if not p.is_file(): raise FileNotFoundError(f'Required existing input missing: {p}. Restore it from the original run; do not refit on this shard.')
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
source_hashes={f:sha256(SOURCE/f) for f in required}
for f in required:
    target=WORK/f; target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SOURCE/f,target)
model=json.loads((WORK/'validation_v3_model.json').read_text())
FEATURES=['bbox_fill','occ_fraction_inside','log_occ_volume','log_bbox_volume','occ_extent_elongation']
assert model['features']==FEATURES, 'Unexpected model feature order'
mu=np.asarray(model['feature_means'],float); sd=np.asarray(model['feature_scales'],float)
beta=np.asarray(model['coefficients'],float)
assert mu.shape==(5,) and sd.shape==(5,) and beta.shape==(6,)
assert np.isfinite(np.r_[mu,sd,beta]).all() and (sd>0).all()
methods=['uncorrected','global_train_mean','occupancy_only_ols']
bounds={k:float(model['calibration_bounds_pct'][k]) for k in methods}
assert all(np.isfinite(v) and v>=0 for v in bounds.values())
assert np.isfinite(model['global_train_mean_bias_pct']) and model['global_train_mean_bias_pct']>-100
# Preserve model and exact source scripts before any new outcomes are examined.
shutil.copytree(WORK,RESULTS/'frozen_inputs')
protocol={'zip_path':str(ZIP_PATH),'run_id':RUN_ID,'source_sha256':source_hashes,
          'analysis':'Apply frozen original model and bounds; no fitting or recalibration',
          'seed':790,'bootstrap_replicates':1000,
          'primary_metrics':['mean signed percent error','median absolute percent error','RMSE percent','coverage at original calibration bound'],
          'independence':'Within-dataset replication; biological provenance unverified'}
(RESULTS/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('Frozen inputs saved. Results:', RESULTS)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','trimesh','scipy','matplotlib','pillow'],check=True)
# Inspect actual instance folders rather than trusting the ZIP endpoint in its filename.
# Isolate this archive from other shards and earlier extraction layouts.
DATA=Path('/content/hit_frozen_data_v2') / ZIP_PATH.stem
DATA.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    members=[]
    for item in z.infolist():
        p=Path(item.filename)
        if p.is_absolute() or '..' in p.parts: raise ValueError('Unsafe archive path')
        if '__MACOSX' in p.parts or any(x.startswith('._') or x=='.DS_Store' for x in p.parts): continue
        members.append(item)
    meshes=[m for m in members if Path(m.filename).name=='raw_mesh.off']
    if not meshes: raise ValueError('No real meshes in archive')
    # Supported layouts: ID/raw_mesh.off or wrapper/.../ID/raw_mesh.off.
    # The shard root is the parent of the instance folders, not the first ZIP component.
    instance_paths=[Path(m.filename).parent for m in meshes]
    shard_parents={p.parent for p in instance_paths}
    if len(shard_parents)!=1:
        raise ValueError(f'Mesh instance folders have multiple parents: {sorted(map(str,shard_parents))[:10]}')
    shard_relative=next(iter(shard_parents))
    ids=[p.name for p in instance_paths]
    if not all(x.isdigit() for x in ids):
        raise ValueError('Expected numeric instance folder names')
    entries={m.filename:m for m in members}
    missing=[str(p / f) for p in instance_paths
             for f in ('raw_mesh.off','pointcloud.ply','occupancies.npz')
             if str(p / f) not in entries or entries[str(p / f)].file_size==0]
    if missing: raise ValueError(f'Missing or empty required instance files: {missing[:10]}')
    assert len(ids)==len(set(ids)), 'Duplicate instance IDs'
    old_ids=set(pd.read_csv(WORK/'validation_v3_splits.csv',dtype={'instance_id':str}).instance_id)
    overlap=old_ids & set(ids)
    if overlap: raise ValueError(f'Instance IDs overlap development data: {sorted(overlap)[:20]}')
    # Match file sizes before reusing an extraction; validate archive CRC during extraction.
    for m in members:
        p=DATA/m.filename
        if m.is_dir(): p.mkdir(parents=True,exist_ok=True); continue
        if not p.exists() or p.stat().st_size!=m.file_size: z.extract(m,DATA)
    SHARD=DATA/shard_relative
    actual_ids={p.parent.name for p in SHARD.glob('*/raw_mesh.off')}
    if actual_ids!=set(ids):
        raise ValueError('Extracted instance IDs differ from this archive; remove stale extraction and rerun.')
    archive_manifest=[{'path':m.filename,'size':m.file_size,'crc':m.CRC} for m in members]
(RESULTS/'archive_manifest.json').write_text(json.dumps(archive_manifest))
print(f'New instances: {len(ids)}; overlapping development IDs: 0; shard: {SHARD}')
subprocess.run([sys.executable,str(WORK/'scripts/audit_3dmsl.py'),
                '--root',str(SHARD),'--out',str(RESULTS/'audit.csv'),'--workers','2'],check=True)

In [ ]:
# Frozen prediction implementation: matches validation_v3 feature order and correction formula.
def frozen_predictions(df, model):
    bbox=df.occ_extent_x*df.occ_extent_y*df.occ_extent_z
    ext=df[['occ_extent_x','occ_extent_y','occ_extent_z']]
    raw=np.column_stack([df.occ_volume_est/bbox.clip(lower=1e-12),df.occ_fraction_inside,
        np.log(df.occ_volume_est.clip(lower=1e-12)),np.log(bbox.clip(lower=1e-12)),
        ext.max(axis=1)/ext.min(axis=1).clip(lower=1e-12)])
    X=np.column_stack([np.ones(len(df)),(raw-np.asarray(model['feature_means']))/np.asarray(model['feature_scales'])])
    pred=np.clip(X@np.asarray(model['coefficients']),-50,100)
    return {'uncorrected':df.occ_volume_est.to_numpy(),
            'global_train_mean':df.occ_volume_est.to_numpy()/(1+model['global_train_mean_bias_pct']/100),
            'occupancy_only_ols':df.occ_volume_est.to_numpy()/(1+pred/100)}
audit=pd.read_csv(RESULTS/'audit.csv',dtype={'instance_id':str})
assert len(audit)==len(ids) and set(audit.instance_id)==set(ids), 'Incomplete or duplicate audit rows'
cols=['mesh_volume','occ_volume_est','occ_fraction_inside','occ_extent_x','occ_extent_y','occ_extent_z']
valid=np.isfinite(audit[cols]).all(axis=1) & (audit[cols]>0).all(axis=1)
for flag in ['mesh_ok','pc_ok','occ_ok']:
    valid &= audit[flag].astype(str).str.lower().eq('true')
audit.assign(included=valid).to_csv(RESULTS/'inclusion_audit.csv',index=False)
df=audit[valid].copy()
if len(df)<2: raise ValueError('Insufficient valid instances')
volumes=frozen_predictions(df,model)
predictions=df[['instance_id','mesh_volume','occ_volume_est','occ_fraction_inside','mesh_watertight']].copy()
for name,v in volumes.items():
    predictions[name+'_volume']=v
    predictions[name+'_error_pct']=100*(v/df.mesh_volume.to_numpy()-1)
predictions.to_csv(RESULTS/'frozen_predictions.csv',index=False)
print(f'Evaluated {len(df)} / {len(audit)} instances; excluded {len(audit)-len(df)}.')

In [ ]:
import matplotlib.pyplot as plt
rng=np.random.default_rng(790)
rc=model.get('repeatability_margin_pct')
if rc is None:
    # Read the earlier run's recorded margin only when present; never estimate from the new shard.
    print('Older model has no recorded repeatability margin; showing fixed 5%, 6%, 10% sensitivity checks only.')
margins=sorted(set([5.,6.,10.]+([] if rc is None else [float(rc)])))
rows=[]
for name in methods:
    e=predictions[name+'_error_pct'].to_numpy()
    bs=np.array([[np.mean(s),np.median(abs(s)),np.mean(abs(s)<=bounds[name])] for s in
                 (e[rng.integers(0,len(e),len(e))] for _ in range(1000))])
    ci=np.percentile(bs,[2.5,97.5],axis=0)
    row={'method':name,'n':len(e),'mean_bias_pct':e.mean(),'bias_ci_low':ci[0,0],'bias_ci_high':ci[1,0],
         'median_absolute_error_pct':np.median(abs(e)),'median_ci_low':ci[0,1],'median_ci_high':ci[1,1],
         'rmse_pct':np.sqrt(np.mean(e**2)),'original_bound_pct':bounds[name],
         'coverage':np.mean(abs(e)<=bounds[name]),'coverage_ci_low':ci[0,2],'coverage_ci_high':ci[1,2],
         'loa_low_pct':e.mean()-1.96*e.std(ddof=1),'loa_high_pct':e.mean()+1.96*e.std(ddof=1)}
    for m in margins: row[f'within_{m:g}_pct']=np.mean(abs(e)<=m)
    rows.append(row)
summary=pd.DataFrame(rows); summary.to_csv(RESULTS/'summary.csv',index=False)
# Descriptive subgroup results only: no subgroup-specific refitting.
subgroups=[]
for label,mask in [('low_occupancy',df.occ_fraction_inside<.02),('other_occupancy',df.occ_fraction_inside>=.02),
                   ('watertight',df.mesh_watertight.astype(str).str.lower().eq('true'))]:
    for name in methods:
        e=predictions.loc[mask,name+'_error_pct'].to_numpy()
        if len(e): subgroups.append(dict(group=label,method=name,n=len(e),bias_pct=e.mean(),median_abs_pct=np.median(abs(e))))
pd.DataFrame(subgroups).to_csv(RESULTS/'subgroup_summary.csv',index=False)
lines=['# Frozen-model validation on a second 3DMSL shard','',
       f'Audited: {len(audit)}; evaluated: {len(df)}; exclusions: {len(audit)-len(df)}.',
       'No coefficients, feature scaling, or calibration bounds were fitted on this shard.','',
       '| Method | Mean bias % | Median absolute error % | RMSE % | Original bound % | Coverage |',
       '|---|---:|---:|---:|---:|---:|']
for r in rows:
    lines.append(f"| {r['method']} | {r['mean_bias_pct']:.3f} | {r['median_absolute_error_pct']:.3f} | {r['rmse_pct']:.3f} | {r['original_bound_pct']:.3f} | {r['coverage']:.1%} |")
lines += ['', 'Confidence intervals and exploratory candidate-margin results are in summary.csv.',
          'Bootstrap intervals resample objects and assume object independence; they do not quantify between-animal uncertainty.',
          'Same-dataset replication does not establish generalization to independent acquisitions. Source-object provenance and geometry duplication remain unverified.',
          'Nominal calibration coverage is not guaranteed after distribution shift. Candidate-margin pass rates are descriptive, not formal equivalence tests.',
          'Nonwatertight meshes remain flagged and retained, following the original policy; a watertight-only sensitivity summary is provided.']
report='\n'.join(lines)
(RESULTS/'report.md').write_text(report)
display(Markdown(report)); display(summary)
fig,ax=plt.subplots(figsize=(9,4))
ax.boxplot([predictions[n+'_error_pct'] for n in methods],showfliers=False)
ax.set_xticks([1,2,3]); ax.set_xticklabels(['Uncorrected','Frozen global','Frozen occupancy-only'])
ax.axhline(0,color='black',linewidth=1); ax.set_ylabel('Volume error relative to mesh (%)')
fig.tight_layout();fig.savefig(RESULTS/'frozen_errors.png',dpi=180);plt.show()
assert source_hashes=={f:sha256(SOURCE/f) for f in required}, 'Original inputs changed during evaluation'
(RESULTS/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Finished. Original inputs unchanged. Results saved to:',RESULTS)

## 20. Remaining analyses before submission

These cells run on their own. Run them top to bottom after Section 19. They read the verified development outputs and the latest frozen-validation run from Drive, and they write every new result to `HIT_outputs_verified/remaining_analyses/<timestamp>/`. They do not refit or change any earlier output.

| Cell | Question it answers | Approx. time on Colab CPU |
|---|---|---|
| 20.0 Setup | mounts Drive, extracts the development shard if needed | ~5 min (first time) |
| 20.1 Mechanism | Where do the occupancy labels put the surface relative to `raw_mesh.off`? Does that explain the whole volume bias? | ~15 min (60 instances) |
| 20.2 Imaging | Rebuilt imaging-condition analysis: physical units, ring-aware segmentation, each image checked against the mesh geometry for its view | ~10 min (300 instances) |
| 20.3 Shard shift | Why is the raw bias larger on the second shard, and where does the frozen correction lose coverage? Includes the `pc_extent_outlier` definition check | < 1 min |
| 20.4 MitoEM v2 | All eligible instances meshed, plus a resolution-sensitivity test (8 → 16 → 24 nm in-plane) | ~20–40 min |
| 20.5 Save | copies the scripts next to the results | < 1 min |

What the 3DMSL paper and code (github.com/bioailab/3DMSL) establish:
- **Mechanism (now documented).** `occupancy_networks/scripts/dataset_mito/build.sh` makes a watertight copy of each mesh with mesh-fusion (`1_scale.py` padding 0.1; `2_fusion.py` at 256³ with `depth_offset_factor` 1.5), then labels 100,000 uniform query points (±0.55 box, float16, bit-packed) against that watertight copy with `sample_mesh.py`. `loc`/`scale` come from the original mesh's bounding box. So the labels describe a slightly inflated copy of `raw_mesh.off`, and the expected outward shift is a fraction of 1.5/256/0.9 = 0.0065 of the normalized extent. The measured 0.00293 is about 0.45 of that. Cell 20.1 measures the shift directly.
- **Imaging (renderer code).** `simulate_img.py` shows:
  - mesh units are 24 nm voxels;
  - emitters are centred on their centroid and rotated by the simulator's own `rotate()` (views `0`, `x90`, `x180`, `y90`, `y180`, `z90`);
  - each image is 128 × 128 and scaled to a maximum of 255;
  - Conf1 files (`N_*`) carry Poisson noise, while Conf2 and Epi1 files are noise-free;
  - the PSF lookup returns zero for emitters below the centroid plane, so **only the upper half of each object is imaged**.

  The old analysis compared foreground *fractions* across different pixel sizes and used Otsu on surface-labelled images. Cell 20.2 fixes both and compares every view with the geometry the simulator actually images.

### 20.0 Setup

In [ ]:
# Section 20 setup: runs on its own (no earlier cells needed). Reads existing outputs and
# writes everything new to a timestamped folder; nothing earlier is modified.
from google.colab import drive
drive.mount('/content/drive')
import datetime, os, shutil, subprocess, sys, zipfile
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'trimesh', 'scipy', 'scikit-image'], check=True)

SRC = Path('/content/drive/MyDrive/HIT_outputs_verified')          # verified development outputs
DEV_ZIP = Path('/content/drive/MyDrive/10_24553_27272.zip')        # development shard
NEW_ZIP = Path('/content/drive/MyDrive/1_1_2729.zip')              # frozen-validation shard
MITO_IMAGE_ZIP = Path('/content/drive/MyDrive/im.zip')
MITO_LABEL_ZIP_DRIVE = Path('/content/drive/MyDrive/EM30-R-mito-train-val-v2.zip')
WORK = Path('/content/hit_s20'); WORK.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = SRC / 'remaining_analyses' / RUN_ID; OUT.mkdir(parents=True, exist_ok=False)

# Latest completed frozen-validation run (the folder that has report.md).
runs = sorted(p for p in (SRC / 'frozen_validation_1_1_2729').glob('*') if (p / 'report.md').exists())
if not runs:
    raise FileNotFoundError('No completed frozen-validation run found; run Section 19 first.')
FROZEN = runs[-1]
for p in [SRC / 'scripts/hit_common.py', SRC / 'paired_feature_table.csv', SRC / 'validation_v3_predictions.csv',
          SRC / 'validation_v3_model.json', FROZEN / 'audit.csv', FROZEN / 'frozen_predictions.csv']:
    if not p.is_file():
        raise FileNotFoundError(f'Missing existing output: {p}')
shutil.copy2(SRC / 'scripts/hit_common.py', WORK / 'hit_common.py')   # same loaders as the verified run
os.chdir(WORK)


def ensure_shard(zip_path, dest):
    """Extract a 3DMSL shard (skipping __MACOSX) unless a complete extraction exists; return its root."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.infolist()
                   if not ('__MACOSX' in Path(m.filename).parts
                           or any(x.startswith('._') or x == '.DS_Store' for x in Path(m.filename).parts))]
        for m in members:
            p = Path(m.filename)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError(f'Unsafe archive path: {m.filename}')
        inst = {Path(m.filename).parent for m in members if Path(m.filename).name == 'raw_mesh.off'}
        parents = {p.parent for p in inst}
        if len(parents) != 1:
            raise ValueError(f'Instance folders have several parents: {sorted(map(str, parents))[:5]}')
        root = dest / next(iter(parents))
        if len(list(root.glob('*/raw_mesh.off'))) != len(inst):
            print(f'Extracting {zip_path.name} ({len(inst)} instances) ...', flush=True)
            for m in members:
                p = dest / m.filename
                if m.is_dir():
                    p.mkdir(parents=True, exist_ok=True)
                elif not p.exists() or p.stat().st_size != m.file_size:
                    z.extract(m, dest)
    n = len(list(root.glob('*/raw_mesh.off')))
    if n != len(inst):
        raise RuntimeError(f'Incomplete extraction: {n}/{len(inst)} meshes under {root}')
    print(f'{zip_path.name}: {n} instances at {root}')
    return root


def run(script, *args):
    subprocess.run([sys.executable, str(WORK / script), *map(str, args)], check=True)


def show(report, *figures):
    display(Markdown(Path(report).read_text()))
    for f in figures:
        if Path(f).exists():
            display(Image(filename=str(f)))


DEV_SHARD = ensure_shard(DEV_ZIP, '/content/hit_s20_data')
print('Frozen-validation run used:', FROZEN)
print('New results will be saved to:', OUT)

### 20.1 Mechanism test: measured label offset vs the volume bias

In [ ]:
%%writefile /content/hit_s20/mechanism_offset.py
#!/usr/bin/env python3
"""
mechanism_offset.py - direct test of the boundary-offset hypothesis for the occupancy
volume bias.

For a random sample of instances it measures, for every occupancy query point near the
surface, its exact signed distance to raw_mesh.off (normalized frame, positive = outside
the mesh) and fits P(labelled inside | distance) with a logistic curve. The distance
where that curve crosses 0.5 is where the occupancy labels place the surface. If the
labels come from a slightly inflated copy of the mesh, the crossing is positive and
roughly constant in the normalized frame.

The measured offset is then compared with the offset implied by the volume error of
the same instances, t = (V_occ - V_mesh) / (A_mesh * scale). Agreement means the whole
volume bias is explained by where the labels put the boundary.

No rtree/embree needed: distances use exact point-triangle distances to the k nearest
triangles, and inside/outside uses the generalized winding number.

  python3 mechanism_offset.py --root /path/to/shard --outdir out --sample 60
"""
import argparse
import os
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.spatial import cKDTree

from hit_common import find_instance_dirs, instance_id, load_mesh, load_occupancy, raw_extent, unnormalize

BAND = 0.02            # normalized units either side of the surface
K_TRI = 32             # candidate triangles per point for the exact distance
MAX_BAND_POINTS = 3000 # random subset per instance (enough for the logistic fit)
BINS = np.arange(-0.012, 0.0121, 0.0005)


def winding_number(points, tris, chunk=48):
    """Generalized winding number (~1 inside, ~0 outside) for a closed mesh."""
    out = np.empty(len(points))
    for s in range(0, len(points), chunk):
        p = points[s:s + chunk]
        a = tris[None, :, 0, :] - p[:, None, :]
        b = tris[None, :, 1, :] - p[:, None, :]
        c = tris[None, :, 2, :] - p[:, None, :]
        la, lb, lc = (np.linalg.norm(x, axis=2) for x in (a, b, c))
        det = np.einsum("pfi,pfi->pf", a, np.cross(b, c))
        den = (la * lb * lc + np.einsum("pfi,pfi->pf", a, b) * lc
               + np.einsum("pfi,pfi->pf", b, c) * la + np.einsum("pfi,pfi->pf", c, a) * lb)
        out[s:s + chunk] = (2.0 * np.arctan2(det, den)).sum(axis=1) / (4 * np.pi)
    return out


def unsigned_distance(points, tris, centroid_tree):
    import trimesh
    k = min(K_TRI, len(tris))
    _, idx = centroid_tree.query(points, k=k)
    idx = idx.reshape(len(points), k)
    cand = tris[idx.ravel()]
    reps = np.repeat(points, k, axis=0)
    closest = trimesh.triangles.closest_point(cand, reps)
    d = np.linalg.norm(closest - reps, axis=1).reshape(len(points), k)
    return d.min(axis=1)


def fit_logistic(d, y):
    """MLE of P(inside) = 1 / (1 + exp((d - t) / s)); returns (t, s)."""
    if len(d) < 50 or y.all() or not y.any():
        return np.nan, np.nan

    def nll(theta):
        t, log_s = theta
        z = np.clip((d - t) / np.exp(log_s), -50, 50)
        p = 1.0 / (1.0 + np.exp(z))
        p = np.clip(p, 1e-9, 1 - 1e-9)
        return -(y * np.log(p) + (1 - y) * np.log(1 - p)).sum()

    res = minimize(nll, x0=[0.0, np.log(0.001)], method="Nelder-Mead",
                   options=dict(xatol=1e-6, fatol=1e-6, maxiter=2000))
    return float(res.x[0]), float(np.exp(res.x[1]))


def one_instance(d):
    m = load_mesh(d)
    pts, occ, loc, scale = load_occupancy(d)
    s = float(np.asarray(scale).reshape(-1)[0])
    verts_n = (np.asarray(m.vertices, float) - loc) / s          # mesh in the normalized frame
    tris = verts_n[np.asarray(m.faces)]
    tree = cKDTree(tris.mean(axis=1))

    # cheap prefilter: distance to nearest centroid, then exact distance in the band
    dc, _ = tree.query(pts, k=1)
    edge = np.linalg.norm(tris[:, 1] - tris[:, 0], axis=1).max()
    near = dc < BAND + edge
    p_near = pts[near]
    dist = unsigned_distance(p_near, tris, tree)
    keep = dist < BAND
    p_band, dist, lab = p_near[keep], dist[keep], occ[near][keep].astype(bool)
    if len(p_band) > MAX_BAND_POINTS:
        sel = np.random.default_rng(len(p_band)).choice(len(p_band), MAX_BAND_POINTS, replace=False)
        p_band, dist, lab = p_band[sel], dist[sel], lab[sel]
    wn = winding_number(p_band, tris)
    signed = np.where(wn > 0.5, -dist, dist)                     # + outside the mesh

    t50, width = fit_logistic(signed, lab.astype(float))
    # volume-implied offset for the same instance (same formula as audit_3dmsl.py)
    pts_raw = unnormalize(pts, loc, scale)
    v_occ = float(occ.mean()) * float(np.prod(raw_extent(pts_raw)))
    v_mesh, a_mesh = float(abs(m.volume)), float(m.area)
    far_in = signed < -0.004
    far_out = signed > 0.008
    return dict(instance_id=instance_id(d), scale=s, n_band_points=int(len(signed)),
                offset_t50=t50, transition_width=width,
                offset_volume_implied=(v_occ - v_mesh) / a_mesh / s,
                vol_err_pct=100 * (v_occ / v_mesh - 1),
                inside_pts_labelled_empty=float((~lab[far_in]).mean()) if far_in.any() else np.nan,
                outside_pts_labelled_full=float(lab[far_out].mean()) if far_out.any() else np.nan,
                q_min=float(pts.min()), q_max=float(pts.max()),
                n_query=int(len(pts))), signed, lab


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--outdir", required=True)
    ap.add_argument("--sample", type=int, default=60)
    ap.add_argument("--seed", type=int, default=790)
    args = ap.parse_args()
    os.makedirs(args.outdir, exist_ok=True)

    dirs = find_instance_dirs(args.root)
    rng = np.random.default_rng(args.seed)
    if args.sample and args.sample < len(dirs):
        dirs = [dirs[i] for i in sorted(rng.choice(len(dirs), args.sample, replace=False))]
    print("Mechanism test on %d instances" % len(dirs), flush=True)

    rows, all_d, all_y, t0 = [], [], [], time.time()
    for i, d in enumerate(dirs, 1):
        try:
            row, sd, lab = one_instance(d)
            rows.append(row); all_d.append(sd); all_y.append(lab)
        except Exception as e:
            rows.append(dict(instance_id=instance_id(d), error="%s: %s" % (type(e).__name__, e)))
        if i % 10 == 0 or i == len(dirs):
            print("  %d/%d (%.1fs/instance)" % (i, len(dirs), (time.time() - t0) / i), flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(args.outdir, "mechanism_offset_instances.csv"), index=False)
    ok = df.dropna(subset=["offset_t50"]) if "offset_t50" in df else df.iloc[:0]
    if ok.empty:
        raise RuntimeError("No instance produced an offset estimate; see mechanism_offset_instances.csv")

    D = np.concatenate(all_d); Y = np.concatenate(all_y)
    idx = np.digitize(D, BINS)
    prof = pd.DataFrame([dict(d_lo=BINS[i - 1], d_hi=BINS[i], n=int((idx == i).sum()),
                              p_inside=float(Y[idx == i].mean()) if (idx == i).any() else np.nan)
                         for i in range(1, len(BINS))])
    prof.to_csv(os.path.join(args.outdir, "mechanism_offset_profile.csv"), index=False)
    t_pool, w_pool = fit_logistic(D, Y.astype(float))

    t50 = ok["offset_t50"]; tv = ok["offset_volume_implied"]
    r = float(np.corrcoef(t50, tv)[0, 1]) if len(ok) > 2 else np.nan
    cv = lambda x: float(x.std() / abs(x.mean())) if x.mean() else np.nan
    q_lo, q_hi = float(ok["q_min"].min()), float(ok["q_max"].max())

    L = ["# Mechanism Test: Where Do the Occupancy Labels Put the Surface? (n=%d)" % len(ok), "",
         "Each occupancy query point within ±%.3f (normalized units, scale = largest mesh extent) of" % BAND,
         "`raw_mesh.off` was given its exact signed distance to the mesh (positive = outside).",
         "A logistic curve P(labelled inside | distance) was fitted per instance; its 50% point is",
         "where the labels place the surface relative to the mesh.", "",
         "## 1. Measured label offset", "",
         "| Quantity | Median | IQR | CV |", "|---|---|---|---|",
         "| label offset t50 (normalized) | **%.5f** | %.5f – %.5f | %.2f |"
         % (t50.median(), t50.quantile(.25), t50.quantile(.75), cv(t50)),
         "| transition width (logistic scale) | %.5f | %.5f – %.5f | |"
         % (ok.transition_width.median(), ok.transition_width.quantile(.25), ok.transition_width.quantile(.75)),
         "| offset implied by the volume error | **%.5f** | %.5f – %.5f | %.2f |"
         % (tv.median(), tv.quantile(.25), tv.quantile(.75), cv(tv)),
         "| pooled logistic fit, all points | %.5f (width %.5f) | | |" % (t_pool, w_pool), "",
         "Correlation between the measured and volume-implied offsets across instances: r = **%.3f**." % r,
         "Points well inside the mesh (d < −0.004) labelled empty: %.3f%% (median per instance)."
         % (100 * ok.inside_pts_labelled_empty.median()),
         "Points well outside (d > +0.008) labelled inside: %.3f%%." % (100 * ok.outside_pts_labelled_full.median()), ""]
    ratio = t50.median() / tv.median() if tv.median() else np.nan
    if t50.median() > 0 and 0.75 <= ratio <= 1.33:
        L += ["The labels put the surface **outside** the mesh by about the same distance the volume",
              "error implies (ratio %.2f). The occupancy volume bias is therefore explained by a boundary" % ratio,
              "offset of the labels relative to `raw_mesh.off`, not by the Monte-Carlo estimator.", ""]
    elif t50.median() > 0:
        L += ["The labels put the surface outside the mesh, but the measured offset is %.2f× the" % ratio,
              "volume-implied one, so a boundary offset explains only part of the volume bias.", ""]
    else:
        L += ["The labels do **not** place the surface outside the mesh, so a boundary offset does",
              "not explain the volume bias. Look for another mechanism.", ""]

    full = 1.5 / 256.0 / 0.9          # depth offset in the normalized frame (1_scale padding 0.1)
    L += ["## 2. Comparison with the documented 3DMSL generation pipeline", "",
          "From github.com/bioailab/3DMSL (`occupancy_networks/scripts/dataset_mito/build.sh`):",
          "each mesh is scaled with mesh-fusion `1_scale.py` (padding 0.1, so the largest extent",
          "becomes 0.9) and fused into a watertight mesh with `2_fusion.py` (defaults: 256³,",
          "`depth_offset_factor` 1.5, which moves every depth map 1.5 voxels toward the camera). Then",
          "`sample_mesh.py --resize --bbox_in_folder 0_in` labels 100,000 uniform points in the ±0.55",
          "box against that watertight mesh (`check_mesh_contains`), with `loc`/`scale` taken from",
          "the original mesh's bounding box. So the labels describe a slightly inflated copy of",
          "`raw_mesh.off`.", "",
          "| Reference | Offset (normalized) | Measured t50 ÷ reference |", "|---|---|---|",
          "| full depth offset along the viewing ray, 1.5/256/0.9 | %.5f | %.2f |" % (full, t50.median() / full),
          "| half of it (mean cos θ = ½ over the visible hemisphere) | %.5f | %.2f |" % (full / 2, t50.median() / (full / 2)), "",
          "The depth offset acts along each camera ray, so the outward shift of the fused surface",
          "along its normal is a fraction of 1.5 voxels that depends on the viewing angles averaged",
          "by the fusion step. A measured offset of the same order (roughly 0.3–1× the full ray",
          "offset) is consistent with the documented pipeline.", "",
          "Query points span %.3f to %.3f in the normalized frame (the pipeline uses ±0.55). Each" % (q_lo, q_hi),
          "archive holds %d query points; the data paper's figure of 10,000 does not match the files." % int(ok.n_query.median()), "",
          "Files: `mechanism_offset_instances.csv`, `mechanism_offset_profile.csv`, `mechanism_offset.png`.", ""]

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        mid = (prof.d_lo + prof.d_hi) / 2
        axes[0].plot(mid, prof.p_inside, "o-", color="#2a78d6", ms=4)
        axes[0].axvline(0, color="#0b0b0b", lw=1.2); axes[0].axvline(t50.median(), color="#eb6834", lw=1.6)
        axes[0].axhline(0.5, color="#888888", lw=0.8, ls="--")
        axes[0].set_xlabel("signed distance to raw mesh (normalized; + outside)")
        axes[0].set_ylabel("share of query points labelled inside")
        axes[0].set_title("Label boundary vs mesh surface (median t50 = %.4f)" % t50.median(), fontsize=10, loc="left")
        axes[1].scatter(tv, t50, s=18, color="#2a78d6", alpha=0.8, edgecolors="none")
        lim = [min(tv.min(), t50.min()), max(tv.max(), t50.max())]
        axes[1].plot(lim, lim, color="#888888", lw=1, ls="--")
        axes[1].set_xlabel("offset implied by volume error"); axes[1].set_ylabel("measured label offset t50")
        axes[1].set_title("Per instance (r = %.2f)" % r, fontsize=10, loc="left")
        for a in axes:
            for sp in ("top", "right"):
                a.spines[sp].set_visible(False)
        fig.tight_layout(); fig.savefig(os.path.join(args.outdir, "mechanism_offset.png"), dpi=180); plt.close(fig)
    except Exception as e:
        L += ["(figure skipped: %s)" % e, ""]

    with open(os.path.join(args.outdir, "mechanism_offset_report.md"), "w") as fh:
        fh.write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
run('mechanism_offset.py', '--root', DEV_SHARD, '--outdir', OUT, '--sample', 60, '--seed', 790)
show(OUT / 'mechanism_offset_report.md', OUT / 'mechanism_offset.png')

### 20.2 Imaging-condition analysis, rebuilt

The segmentation rule (half-peak above the border background, closed, hole-filled, largest component) is fixed before looking at the results. Geometric targets follow the simulator code. The focal height is not stated, so slices at 0–500 nm above the centroid are all shown; the best match is an exploratory identification. Check `imaging_v2_overlays.png` by eye before trusting any number. If the Epi1 row's implied pixel size is near 80 nm rather than 109 nm, rerun with `--epi-pixel-um 0.08`.

In [ ]:
%%writefile /content/hit_s20/imaging_v2.py
#!/usr/bin/env python3
"""
imaging_v2.py - rebuilt imaging-condition analysis for the 3DMSL renders.

Why the first version failed (imaging_condition_reliability.py):
  * The three configurations use different pixel sizes (Conf1 70 nm, Conf2 48 nm,
    Epi1 109 nm; 3DMSL data paper), so "foreground fraction of the image" measures
    a different physical area in each one. Absolute-agreement ICC across them could
    not work.
  * The renders label the mitochondrial SURFACE. A confocal optical section through
    a surface label is a ring, so Otsu picks out membrane pixels, not the object.
  * Nothing was compared with the known geometry, so real orientation effects
    (an elongated object looks different end-on) could not be told apart from
    measurement error.

What this version does:
  1. Records bit depth, image size and value range of every render.
  2. Segments each view by thresholding at half the peak above a border-estimated
     background, closing and filling holes, and keeping the largest component
     (primary, fixed before looking at results). Otsu is kept for comparison.
  3. Converts every size measure to physical units with each configuration's pixel size.
     It also computes a threshold-free size: the intensity-weighted radius of gyration.
  4. Computes geometric targets from the mesh for every view, following the 3DMSL
     simulator (github.com/bioailab/3DMSL, occupancy_networks/scripts/dataset_mito/
     simulate_img.py): mesh units are 24 nm; emitters are centred on their centroid and
     rotated with the simulator's own rotate(); the optical axis is the third coordinate;
     and emitters below the centroid plane fall outside the PSF lookup (map_coordinates
     with zero fill for negative depth), so only the upper half is imaged. The targets
     are the silhouette of the upper half, the silhouette of the whole object, and
     cross-sections at several heights above the centroid. The focal depth depends on the
     Gibson-Lanni model with the stage 1 um toward the objective, so it is identified
     from the data (exploratory).
  5. Reports validity (image measure vs geometric target), orientation effects (how
     much of the across-view variation the geometry explains), and cross-configuration
     agreement in physical units (absolute and consistency ICC).

  python3 imaging_v2.py --root /path/to/shard --outdir out --sample 300
"""
import argparse
import os
import re
import time
import zlib
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi

from hit_common import IMG_CONFIGS, find_instance_dirs, images_for_config, instance_id, load_mesh, otsu_threshold

PIXEL_UM = {"Conf1": 0.070, "Conf2": 0.0488, "Epi1": 0.109}     # data paper table; Conf2 from repo conf2.py
# (the repo's only widefield config, epi2.py, uses 80 nm pixels; the paper lists 109 nm for Epi1.
#  The geometry ratio in section 2 of the report checks which one fits.)
MODALITY = {"Conf1": "confocal", "Conf2": "confocal", "Epi1": "widefield"}
VIEWS = ("0", "x90", "x180", "y90", "y180", "z90")
HEIGHTS_UM = (0.0, 0.1, 0.2, 0.3, 0.5)    # candidate focal heights above the emitter centroid
BORDER = 6                   # pixels used to estimate background
AXES = np.eye(3)


def sim_rotate(X, theta, axis):
    """Exact copy of rotate() in 3DMSL simulate_img.py (row vectors times matrix)."""
    c, s_ = np.cos(theta), np.sin(theta)
    M = {"x": np.array([[1., 0, 0], [0, c, -s_], [0, s_, c]]),
         "y": np.array([[c, 0, -s_], [0, 1, 0], [s_, 0, c]]),
         "z": np.array([[c, -s_, 0], [s_, c, 0], [0, 0, 1.]])}[axis]
    return np.dot(X, M)


def view_transform(X, view):
    if view == "0":
        return X.copy()
    return sim_rotate(X, np.deg2rad(float(view[1:])), view[0])


def view_name(path):
    m = re.search(r"_(0|x90|x180|y90|y180|z90)$", os.path.splitext(os.path.basename(path))[0])
    return m.group(1) if m else None


# ------------------------------------------------------------------ geometry targets
def _raster_area(uv, cell):
    """Area of the region covered by dense 2-D points, via a filled occupancy grid."""
    if len(uv) < 3:
        return 0.0
    lo = uv.min(axis=0) - 3 * cell
    ij = np.floor((uv - lo) / cell).astype(int)
    shape = ij.max(axis=0) + 4
    g = np.zeros(shape, bool)
    g[ij[:, 0], ij[:, 1]] = True
    g = ndi.binary_closing(g, structure=np.ones((3, 3)), iterations=2)
    g = ndi.binary_fill_holes(g)
    return float(g.sum()) * cell * cell


def _rg(uv):
    return float(np.sqrt(((uv - uv.mean(0)) ** 2).sum(1).mean())) if len(uv) > 2 else np.nan


def mesh_targets(mesh, unit_um, n_points=120000, seed=0):
    """Per view: geometric targets in um / um^2, following the simulator's geometry."""
    import trimesh
    pts, _ = trimesh.sample.sample_surface(mesh, n_points, seed=seed)
    pts = np.asarray(pts, float)
    centre = pts.mean(axis=0)                           # emitter centroid (area-weighted)
    P0 = (pts - centre) * unit_um
    V0 = (np.asarray(mesh.vertices, float) - centre) * unit_um
    faces = np.asarray(mesh.faces)
    cell = float(np.ptp(P0, axis=0).max()) / 160.0
    out = {}
    for v in VIEWS:
        P = view_transform(P0, v)
        up = P[P[:, 2] >= 0]
        t = {"sil_upper": _raster_area(up[:, :2], cell), "sil_all": _raster_area(P[:, :2], cell),
             "rg_upper": _rg(up[:, :2]), "rg_all": _rg(P[:, :2])}
        rm = trimesh.Trimesh(vertices=view_transform(V0, v), faces=faces, process=False)
        for h in HEIGHTS_UM:
            key = "sec_%03dnm" % round(1000 * h)
            try:
                seg = trimesh.intersections.mesh_plane(rm, plane_normal=[0, 0, 1], plane_origin=[0, 0, h])
                if len(seg) == 0:
                    t[key] = 0.0
                else:
                    seg = np.asarray(seg, float)
                    tt = np.linspace(0, 1, 12)[None, :, None]
                    line = (seg[:, :1, :] * (1 - tt) + seg[:, 1:2, :] * tt).reshape(-1, 3)[:, :2]
                    t[key] = _raster_area(line, cell)
            except Exception:
                t[key] = np.nan
        out[v] = t
    return out


# ----------------------------------------------------------------------- image side
def read_render(path):
    im = Image.open(path)
    arr = np.asarray(im)
    info = dict(mode=im.mode, dtype=str(arr.dtype), h=arr.shape[0], w=arr.shape[1],
                vmin=float(arr.min()), vmax=float(arr.max()))
    a = arr.astype(np.float64)
    if a.ndim == 3:
        a = a[..., :3].mean(axis=2)
    return a, info


def segment(a):
    """Primary mask: half-peak above border background, closed, hole-filled, largest component."""
    border = np.concatenate([a[:BORDER].ravel(), a[-BORDER:].ravel(), a[:, :BORDER].ravel(), a[:, -BORDER:].ravel()])
    bg = float(np.median(border))
    sig = float(1.4826 * np.median(np.abs(border - bg))) or float(border.std()) or 1e-9
    peak = float(np.percentile(a, 99.8))
    thr = bg + max(0.5 * (peak - bg), 3 * sig)
    m = a > thr
    m = ndi.binary_closing(m, structure=np.ones((3, 3)), iterations=2)
    m = ndi.binary_fill_holes(m)
    lab, n = ndi.label(m)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = lab == (1 + int(np.argmax(sizes)))
    return m, bg, sig, peak


def mask_stats(m, px):
    if not m.any():
        return dict(area_um2=0.0, major_um=np.nan, minor_um=np.nan, touches_border=False, mask_empty=True)
    ys, xs = np.nonzero(m)
    cov = np.cov(np.vstack([xs, ys])) if len(xs) > 2 else np.eye(2)
    ev = np.sort(np.linalg.eigvalsh(cov))[::-1].clip(min=0)
    return dict(area_um2=float(m.sum()) * px * px,
                major_um=4 * np.sqrt(ev[0]) * px, minor_um=4 * np.sqrt(ev[1]) * px,
                touches_border=bool(m[0].any() or m[-1].any() or m[:, 0].any() or m[:, -1].any()),
                mask_empty=False)


def one_view(path, config):
    a, info = read_render(path)
    px = PIXEL_UM[config]
    m, bg, sig, peak = segment(a)
    st = mask_stats(m, px)
    # threshold-free size: intensity-weighted radius of gyration, restricted to a generous
    # neighbourhood of the object so background noise across the frame does not dominate
    near = ndi.binary_dilation(m, iterations=4) if m.any() else np.zeros_like(m)
    w = np.where(near, np.clip(a - bg - 2 * sig, 0, None), 0.0)
    if w.sum() > 0:
        yy, xx = np.indices(a.shape)
        cy, cx = (w * yy).sum() / w.sum(), (w * xx).sum() / w.sum()
        rg = float(np.sqrt((w * ((yy - cy) ** 2 + (xx - cx) ** 2)).sum() / w.sum())) * px
    else:
        rg = np.nan
    lo, hi = a.min(), a.max()
    a8 = np.zeros_like(a, np.uint8) if hi == lo else ((a - lo) / (hi - lo) * 255).astype(np.uint8)
    otsu_m = a8 > otsu_threshold(a8)
    row = dict(config=config, view=view_name(path), file=os.path.basename(path), pixel_um=px,
               bg=bg, bg_sigma=sig, peak=peak, snr=(peak - bg) / sig if sig else np.nan,
               area_um2=st["area_um2"], major_um=st["major_um"], minor_um=st["minor_um"],
               touches_border=st["touches_border"], mask_empty=st["mask_empty"], rg_um=rg,
               otsu_area_um2=float(otsu_m.sum()) * px * px, otsu_fraction=float(otsu_m.mean()))
    row.update({"img_" + k: v for k, v in info.items()})
    return row, m, otsu_m, a


def one_instance(d, n_points, unit_um):
    iid = instance_id(d)
    mesh = load_mesh(d)
    tg = mesh_targets(mesh, unit_um, n_points=n_points, seed=zlib.crc32(iid.encode()))
    rows = []
    for c in IMG_CONFIGS:
        for f in images_for_config(d, c):
            row, *_ = one_view(f, c)
            row["instance_id"] = iid
            if row["view"] in tg:
                row.update({"target_" + k: val for k, val in tg[row["view"]].items()})
            rows.append(row)
    return rows


def safe(d, n_points, unit_um):
    try:
        return one_instance(d, n_points, unit_um)
    except Exception as e:
        return [dict(instance_id=instance_id(d), error="%s: %s" % (type(e).__name__, e))]


# ------------------------------------------------------------------------ statistics
def icc(M, kind="A"):
    """ICC(A,1) absolute agreement or ICC(C,1) consistency, two-way model."""
    M = np.asarray(M, float); n, k = M.shape
    if n < 3 or k < 2:
        return np.nan
    g = M.mean(); rm = M.mean(1); cm = M.mean(0)
    MSR = k * ((rm - g) ** 2).sum() / (n - 1)
    MSC = n * ((cm - g) ** 2).sum() / (k - 1)
    MSE = ((M - rm[:, None] - cm[None, :] + g) ** 2).sum() / ((n - 1) * (k - 1))
    den = MSR + (k - 1) * MSE + (k * (MSC - MSE) / n if kind == "A" else 0.0)
    return float((MSR - MSE) / den) if den else np.nan


def spearman(x, y):
    x = pd.Series(x).rank().to_numpy(); y = pd.Series(y).rank().to_numpy()
    return float(np.corrcoef(x, y)[0, 1]) if x.std() and y.std() else np.nan


def pearson(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    return float(np.corrcoef(x, y)[0, 1]) if x.std() and y.std() else np.nan


def overlay_figure(dirs, outpath, n=8):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    dirs = dirs[:n]
    fig, axes = plt.subplots(len(dirs), 3, figsize=(7.5, 2.5 * len(dirs)))
    axes = np.atleast_2d(axes)
    for r, d in enumerate(dirs):
        for c_i, c in enumerate(IMG_CONFIGS):
            ax = axes[r, c_i]; ax.axis("off")
            files = [f for f in images_for_config(d, c) if view_name(f) == "0"] or images_for_config(d, c)
            if not files:
                continue
            _, m, om, a = one_view(files[0], c)
            ax.imshow(a, cmap="gray")
            ax.contour(m, levels=[0.5], colors="#eb6834", linewidths=1.0)
            ax.contour(om, levels=[0.5], colors="#2a78d6", linewidths=0.6, linestyles="dotted")
            if r == 0:
                ax.set_title("%s (%.0f nm px)" % (c, 1000 * PIXEL_UM[c]), fontsize=9)
        axes[r, 0].text(-4, 10, instance_id(d), fontsize=8, ha="right")
    fig.suptitle("orange = primary mask (half-peak, filled); blue dotted = Otsu (old)", fontsize=9)
    fig.tight_layout(); fig.savefig(outpath, dpi=150); plt.close(fig)


# ------------------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--outdir", required=True)
    ap.add_argument("--sample", type=int, default=300)
    ap.add_argument("--seed", type=int, default=790)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--mesh-unit-um", type=float, default=0.024,
                    help="physical size of one mesh unit (24 nm: roi_to_mesh.py meshes the 24 nm stack "
                         "and simulate_img.py multiplies by resolution=24 nm)")
    ap.add_argument("--n-surface-points", type=int, default=120000)
    ap.add_argument("--epi-pixel-um", type=float, default=None,
                    help="override the Epi1 pixel size (paper 0.109 um; repo epi2.py 0.080 um)")
    args = ap.parse_args()
    if args.epi_pixel_um:
        PIXEL_UM["Epi1"] = args.epi_pixel_um      # inherited by forked worker processes
    os.makedirs(args.outdir, exist_ok=True)

    dirs = find_instance_dirs(args.root)
    if args.sample and args.sample < len(dirs):
        pick = np.random.default_rng(args.seed).choice(len(dirs), args.sample, replace=False)
        dirs = [dirs[i] for i in sorted(pick)]
    print("imaging_v2 on %d instances" % len(dirs), flush=True)
    t0, rows = time.time(), []
    with ProcessPoolExecutor(max_workers=max(1, args.workers)) as ex:
        futs = [ex.submit(safe, d, args.n_surface_points, args.mesh_unit_um) for d in dirs]
        for i, f in enumerate(as_completed(futs), 1):
            rows.extend(f.result())
            if i % 25 == 0 or i == len(futs):
                print("  %d/%d (%.1fs)" % (i, len(futs), time.time() - t0), flush=True)
    raw = pd.DataFrame(rows)
    raw.to_csv(os.path.join(args.outdir, "imaging_v2_views.csv"), index=False)
    n_err = int(raw["error"].notna().sum()) if "error" in raw else 0
    v = raw.dropna(subset=["config", "view"]).copy() if "config" in raw else raw.iloc[:0]
    if v.empty:
        raise RuntimeError("No views were analyzed; see imaging_v2_views.csv")
    usable = v[~v.touches_border.astype(bool) & ~v.mask_empty.astype(bool)].copy()

    L = ["# Imaging-Condition Reliability, Rebuilt (n=%d instances)" % v.instance_id.nunique(), "",
         "Replaces `imaging_condition_report.md`. Sizes are in physical units (pixel sizes from the",
         "3DMSL data paper: Conf1 70 nm, Conf2 48 nm, Epi1 109 nm), and every image measure is checked",
         "against the matching geometric quantity computed from the mesh for the same view.", ""]
    if n_err:
        L += ["%d instance(s) failed and were skipped (see the `error` column)." % n_err, ""]

    # 1. render diagnostics
    L += ["## 1. What the renders are", "", "| Config | mode / dtype | size (px) | field of view | value range | median SNR |",
          "|---|---|---|---|---|---|"]
    for c in IMG_CONFIGS:
        s = v[v.config == c]
        if s.empty:
            continue
        L.append("| %s | %s / %s | %d × %d | %.1f µm | %.0f – %.0f | %.1f |"
                 % (c, s.img_mode.mode().iat[0], s.img_dtype.mode().iat[0], s.img_h.median(), s.img_w.median(),
                    s.img_w.median() * PIXEL_UM[c], s.img_vmin.min(), s.img_vmax.max(), s.snr.median()))
        if s.snr.median() > 1e4:
            L[-1] = L[-1].rsplit("|", 2)[0] + "| noise-free |"
    L += ["", "| Config | views | empty mask | object cut by image edge | used |", "|---|---|---|---|---|"]
    for c in IMG_CONFIGS:
        s = v[v.config == c]
        if s.empty:
            continue
        L.append("| %s | %d | %.1f%% | %.1f%% | %d |" % (c, len(s), 100 * s.mask_empty.astype(bool).mean(),
                                                     100 * s.touches_border.astype(bool).mean(),
                                                     int((usable.config == c).sum())))
    L += ["", "Views where the object touches the image edge are excluded from size measures",
          "(part of the object is outside the field of view).", ""]

    # 2. validity against geometry
    tkeys = ["sil_upper", "sil_all"] + ["sec_%03dnm" % round(1000 * h) for h in HEIGHTS_UM]
    tlabel = {"sil_upper": "silhouette, upper half", "sil_all": "silhouette, whole object"}
    tlabel.update({"sec_%03dnm" % round(1000 * h): "slice at +%.0f nm" % (1000 * h) for h in HEIGHTS_UM})
    L += ["## 2. Does the image measure track the true geometry?", "",
          "Targets are computed per view with the simulator's own rotation and centring. Only the",
          "upper half of each object reaches the image (emitters below the centroid plane fall outside",
          "the PSF lookup), and the focal height is not stated, so every candidate target is shown.",
          "Spearman correlation between the filled image area and each target:", "",
          "| Config | " + " | ".join(tlabel[k] for k in tkeys) + " | Otsu area (old) vs best |",
          "|---|" + "---|" * (len(tkeys) + 1)]
    best = {}
    for c in IMG_CONFIGS:
        s = usable[usable.config == c]
        if len(s) < 10 or not all("target_" + k in s for k in tkeys):
            continue
        rs = [spearman(s.area_um2, s["target_" + k]) for k in tkeys]
        b = tkeys[int(np.nanargmax(rs))]; best[c] = b
        ro = spearman(s.otsu_area_um2, s["target_" + b])
        L.append("| %s | " % c + " | ".join("%.3f" % r for r in rs) + " | %.3f |" % ro)
    ratios, rgs = {}, {}
    for c, b in best.items():
        s = usable[usable.config == c]
        rg_key = "target_rg_upper" if b != "sil_all" else "target_rg_all"
        ok = s["target_" + b] > 0
        ratios[c] = float((s.area_um2[ok] / s["target_" + b][ok]).median())
        rgs[c] = spearman(s.rg_um, s[rg_key])
    conf_ref = [ratios[c] for c in ratios if MODALITY[c] == "confocal"]
    ref = float(np.mean(conf_ref)) if conf_ref else np.nan
    L += ["", "| Config | best-matching target | Spearman, radius of gyration vs target Rg | median image ÷ geometric area | pixel size that would give the confocal ratio |",
          "|---|---|---|---|---|"]
    for c, b in best.items():
        imp = 1000 * PIXEL_UM[c] * np.sqrt(ref / ratios[c]) if ratios[c] > 0 and np.isfinite(ref) else np.nan
        L.append("| %s | %s | %.3f | %.2f | %.0f nm (used: %.0f nm) |" % (c, tlabel[b], rgs[c], ratios[c], imp, 1000 * PIXEL_UM[c]))
    L += ["", "The best-matching target is chosen from the data, so treat it as an exploratory",
          "comparison of candidate geometric targets. Different optical configurations need not share an area ratio.",
          "The implied pixel size above matches the confocal ratio; matching unit geometric agreement uses pixel_size/sqrt(area_ratio).",
          "If Epi1's implied pixel size is far from the 109 nm used (the repo's epi2.py uses 80 nm),",
          "verify the actual simulator configuration before interpreting this discrepancy as a calibration error.", ""]

    # 3. orientation: geometry vs measurement noise
    L += ["## 3. Orientation effects: real geometry or measurement error?", "",
          "Within each instance, the six views are compared on log area against the best-matching",
          "target. *Geometry share* is the squared correlation between the image's and the geometry's",
          "view-to-view deviations; the residual SD is the variation the geometry does not explain.", "",
          "| Config | median CV, image | median CV, geometry | geometry share (R²) | residual SD (log) |",
          "|---|---|---|---|---|"]
    orient = {}
    for c, b in best.items():
        s = usable[usable.config == c].copy()
        s["tg"] = s["target_" + b]
        s = s[(s.tg > 0) & (s.area_um2 > 0)]
        g = s.groupby("instance_id")
        cv_img = (g.area_um2.std() / g.area_um2.mean()).median()
        cv_geo = (g.tg.std() / g.tg.mean()).median()
        li, lt = np.log(s.area_um2), np.log(s.tg)
        di = li - li.groupby(s.instance_id).transform("mean")
        dt = lt - lt.groupby(s.instance_id).transform("mean")
        r2 = pearson(di, dt) ** 2
        slope = float((di * dt).sum() / (dt * dt).sum()) if (dt * dt).sum() else np.nan
        resid = float(np.sqrt(((di - slope * dt) ** 2).mean()))
        orient[c] = (cv_img, cv_geo, r2, resid)
        L.append("| %s | %.3f | %.3f | %.2f | %.3f |" % (c, cv_img, cv_geo, r2, resid))
    L += [""]
    for c, (ci, cg, r2, rs) in orient.items():
        if r2 >= 0.5:
            L.append("- %s: most of the orientation variation is real geometry (R² %.2f): the image"
                     " follows how the visible part of the object changes with viewing angle." % (c, r2))
        else:
            L.append("- %s: geometry explains only %.0f%% of the orientation variation; the rest is"
                     " measurement error from rendering/segmentation." % (c, 100 * r2))
    L += [""]

    # 4. cross-configuration agreement in physical units
    L += ["## 4. Agreement across configurations (physical units, mean over usable views)", "",
          "| Pair | n | ICC(A,1) absolute | ICC(C,1) consistency | Spearman | median ratio |",
          "|---|---|---|---|---|---|"]
    inst = usable.groupby(["instance_id", "config"]).area_um2.mean().unstack()
    for a_, b_ in (("Conf1", "Conf2"), ("Conf1", "Epi1"), ("Conf2", "Epi1")):
        if a_ in inst and b_ in inst:
            w = inst[[a_, b_]].dropna()
            if len(w) >= 3:
                L.append("| %s vs %s | %d | %.3f | %.3f | %.3f | %.2f |"
                         % (a_, b_, len(w), icc(w.values, "A"), icc(w.values, "C"),
                            spearman(w[a_], w[b_]), (w[b_] / w[a_]).median()))
    L += ["", "Conf1 and Conf2 are both confocal sections and can be compared directly. Epi1 is a",
          "widefield image (a blurred silhouette), which is a different quantity from a section, so",
          "for Epi1 read the consistency ICC and Spearman; absolute agreement is not expected.", ""]
    L += ["## Interpretation guardrails", "",
          "- These are simulated renders from the same meshes; they test how imaging configuration and",
          "  viewing angle change image-based morphometry, not biological variation.",
          "- Geometry follows the public simulator code (rotation, centring, upper-half visibility,",
          "  24 nm mesh unit); the focal height is identified from the data and the released images",
          "  may have been made with a slightly different code version. Confirm with the 3DMSL authors.",
          "- Conf1 files are the Poisson-noise versions (N_*); Conf2 and Epi1 files are noise-free and",
          "  normalized to a maximum of 255 per image, so intensities are not physical in any config.",
          "- The primary segmentation rule was fixed before looking at results; Otsu is shown only to",
          "  explain the first analysis.", ""]

    try:
        overlay_figure(dirs, os.path.join(args.outdir, "imaging_v2_overlays.png"))
        L += ["Figure: `imaging_v2_overlays.png` (masks on the 0° view of the first 8 instances).", ""]
    except Exception as e:
        L += ["(overlay figure skipped: %s)" % e, ""]
    with open(os.path.join(args.outdir, "imaging_v2_report.md"), "w") as fh:
        fh.write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
run('imaging_v2.py', '--root', DEV_SHARD, '--outdir', OUT, '--sample', 300, '--seed', 790,
    '--workers', os.cpu_count() or 2)
show(OUT / 'imaging_v2_report.md', OUT / 'imaging_v2_overlays.png')

### 20.3 Second-shard shift, coverage loss, and the `pc_extent_outlier` definition

In [ ]:
%%writefile /content/hit_s20/shard_shift.py
#!/usr/bin/env python3
"""
shard_shift.py - why is the raw occupancy bias larger on the second shard, and where does
the frozen correction lose coverage?

Inputs (all existing outputs; nothing is refitted on the second shard):
  --dev-paired   development paired_feature_table.csv (shard 10_24553_27272)
  --dev-pred     development validation_v3_predictions.csv (held-out test errors)
  --new-audit    second-shard audit.csv from the frozen-validation run
  --new-pred     second-shard frozen_predictions.csv
  --model        validation_v3_model.json (for the frozen calibration bound)

Reports:
  1. covariate shift between shards (standardized mean differences)
  2. whether the bias mechanism replicates (same surface-to-volume slope, same offset
     in the normalized frame)
  3. how much of the larger raw bias the shift in surface-to-volume predicts, using the
     development relationship only
  4. where the frozen correction's errors and coverage loss sit (subgroups, deciles)
  5. the pc_extent_outlier flag under alternative definitions, in both shards

  python3 shard_shift.py --dev-paired ... --dev-pred ... --new-audit ... --new-pred ... --model ... --outdir out
"""
import argparse
import json
import os

import numpy as np
import pandas as pd


def derive(df):
    df = df.copy()
    df["instance_id"] = df["instance_id"].astype(str)
    df["err_pct"] = 100 * (df.occ_volume_est / df.mesh_volume - 1)
    df["sv"] = df.mesh_area / df.mesh_volume ** (2 / 3)
    df["t_norm"] = (df.occ_volume_est - df.mesh_volume) / df.mesh_area / df.occ_scale
    df["log_volume"] = np.log(df.mesh_volume)
    df["log_scale"] = np.log(df.occ_scale)
    df["watertight"] = df.mesh_watertight.astype(str).str.lower().eq("true")
    df["low_occ"] = df.occ_fraction_inside < 0.02
    return df


def smd(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    sp = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return float((b.mean() - a.mean()) / sp) if sp else np.nan


def fit(y, x):
    X = np.column_stack([np.ones(len(x)), x])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ b
    return b, float(1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def outlier_defs(df):
    out = {}
    if "pc_extent_pct_diff_max" in df:
        out["largest single-axis deviation > 10% (current qc_filter.py)"] = int((df.pc_extent_pct_diff_max.abs() > 10).sum())
    if {"pc_extent_x", "mesh_extent_x"} <= set(df.columns):
        x = 100 * (df.pc_extent_x / df.mesh_extent_x - 1)
        out["x axis only > 10%"] = int((x.abs() > 10).sum())
    cols = [a for a in "xyz" if {"pc_extent_%s" % a, "mesh_extent_%s" % a} <= set(df.columns)]
    if cols:
        m = pd.concat([(100 * (df["pc_extent_%s" % a] / df["mesh_extent_%s" % a] - 1)).abs() for a in cols], axis=1)
        out["mean over axes > 10%"] = int((m.mean(axis=1) > 10).sum())
        out["max over axes > 15%"] = int((m.max(axis=1) > 15).sum())
    return out


def main():
    ap = argparse.ArgumentParser()
    for k in ("dev-paired", "dev-pred", "new-audit", "new-pred", "model", "outdir"):
        ap.add_argument("--" + k, required=True)
    a = ap.parse_args()
    os.makedirs(a.outdir, exist_ok=True)
    dev = derive(pd.read_csv(a.dev_paired, dtype={"instance_id": str}))
    new = derive(pd.read_csv(a.new_audit, dtype={"instance_id": str}))
    newp = pd.read_csv(a.new_pred, dtype={"instance_id": str})
    devp = pd.read_csv(a.dev_pred, dtype={"instance_id": str})
    model = json.load(open(a.model))
    bound = float(model["calibration_bounds_pct"]["occupancy_only_ols"])
    new = new.merge(newp[["instance_id", "occupancy_only_ols_error_pct"]], on="instance_id", how="inner")
    new["corr_err"] = new.occupancy_only_ols_error_pct
    dev_test = dev.merge(devp[devp.split == "test"][["instance_id", "corrected_error_occupancy_only_ols"]],
                         on="instance_id", how="inner").rename(columns={"corrected_error_occupancy_only_ols": "corr_err"})

    L = ["# Second-Shard Shift and Coverage Diagnostics", "",
         "Development shard n=%d (held-out test n=%d); second shard n=%d. Nothing is refitted on the"
         % (len(dev), len(dev_test), len(new)),
         "second shard: all comparisons use development fits or the frozen model.", ""]

    # 1. covariate shift
    L += ["## 1. How the shards differ", "",
          "| Variable | development median | second shard median | SMD |", "|---|---|---|---|"]
    rows = []
    for v, lbl in (("log_volume", "log mesh volume"), ("mesh_sphericity", "sphericity"),
                   ("mesh_elongation", "elongation"), ("sv", "surface-to-volume A/V^(2/3)"),
                   ("occ_fraction_inside", "occ_fraction_inside"), ("log_scale", "log scale"),
                   ("err_pct", "raw volume error %")):
        s = smd(dev[v], new[v]); rows.append((lbl, s))
        L.append("| %s | %.4g | %.4g | %+.2f |" % (lbl, dev[v].median(), new[v].median(), s))
    L += ["| non-watertight meshes | %d (%.1f%%) | %d (%.1f%%) | |" % ((~dev.watertight).sum(), 100 * (~dev.watertight).mean(),
                                                                     (~new.watertight).sum(), 100 * (~new.watertight).mean()),
          "| occ_fraction_inside < 2%% | %d (%.1f%%) | %d (%.1f%%) | |" % (dev.low_occ.sum(), 100 * dev.low_occ.mean(),
                                                                         new.low_occ.sum(), 100 * new.low_occ.mean()), "",
          "SMD = standardized mean difference (second − development); |SMD| > 0.2 is a noticeable shift.", ""]

    # 2. mechanism replication
    bd, r2d = fit(dev.err_pct.to_numpy(), dev.sv.to_numpy())
    bn, r2n = fit(new.err_pct.to_numpy(), new.sv.to_numpy())
    cv = lambda x: float(x.std() / abs(x.mean()))
    wt = new[new.watertight]
    L += ["## 2. Does the bias mechanism replicate?", "",
          "| | development | second shard | second shard, watertight only |", "|---|---|---|---|",
          "| mean raw error | %+.3f%% | %+.3f%% | %+.3f%% |" % (dev.err_pct.mean(), new.err_pct.mean(), wt.err_pct.mean()),
          "| share of instances above zero | %.2f%% | %.2f%% | %.2f%% |" % (100 * (dev.err_pct > 0).mean(), 100 * (new.err_pct > 0).mean(), 100 * (wt.err_pct > 0).mean()),
          "| slope of error on A/V^(2/3) | %.3f | %.3f | |" % (bd[1], bn[1]),
          "| R² of error on A/V^(2/3) | %.2f | %.2f | |" % (r2d, r2n),
          "| median offset t / scale | %.5f | %.5f | %.5f |" % (dev.t_norm.median(), new.t_norm.median(), wt.t_norm.median()),
          "| CV of t / scale | %.2f | %.2f | %.2f |" % (cv(dev.t_norm), cv(new.t_norm), cv(wt.t_norm)), ""]
    same_t = abs(new.t_norm.median() / dev.t_norm.median() - 1) < 0.10
    L += ["The normalized-frame offset is %s across shards (%.5f vs %.5f)%s" % (
        "essentially the same" if same_t else "different", dev.t_norm.median(), new.t_norm.median(),
        ". The mechanism is a fixed property of how the labels were generated, not of one shard." if same_t
        else ". Check the non-watertight meshes and generation provenance before pooling shards."), ""]

    # 3. how much of the shift does surface-to-volume predict?
    pred_dev = bd[0] + bd[1] * dev.sv.mean()
    pred_new = bd[0] + bd[1] * new.sv.mean()
    obs_shift = new.err_pct.mean() - dev.err_pct.mean()
    pred_shift = pred_new - pred_dev
    L += ["## 3. Is the larger raw bias explained by shape?", "",
          "Using the development fit of error on surface-to-volume only:", "",
          "- observed change in mean raw error: **%+.3f** percentage points" % obs_shift,
          "- change predicted by the shift in A/V^(2/3): **%+.3f** points (%.0f%% of the observed change)"
          % (pred_shift, 100 * pred_shift / obs_shift if obs_shift else np.nan), ""]
    if abs(obs_shift) < 0.1:
        L += ["The mean raw error barely differs between shards, so there is no shift to explain.", ""]
    else:
      L += [("Most of the larger raw bias comes from the second shard having more surface per unit volume, "
           "which is what the boundary-offset mechanism predicts. That is also why the occupancy-only "
           "model, whose features track shape, still comes out unbiased while the constant correction does not.")
          if obs_shift and pred_shift / obs_shift >= 0.6 else
          ("Shape explains only part of the larger raw bias; the rest comes from something that differs "
             "between shards (check non-watertight meshes and the source of the objects)."), ""]

    # 4. where the frozen model's errors sit
    L += ["## 4. Where the frozen correction loses coverage (bound %.3f%%)" % bound, "",
          "| Group | n | median \\|error\\| | coverage | development test coverage |", "|---|---|---|---|---|"]

    def cov(e):
        return 100 * np.mean(np.abs(e) <= bound)
    groups = [("all", new, dev_test), ("watertight", new[new.watertight], dev_test[dev_test.watertight]),
              ("non-watertight", new[~new.watertight], dev_test[~dev_test.watertight]),
              ("occ_fraction < 2%", new[new.low_occ], dev_test[dev_test.low_occ]),
              ("occ_fraction ≥ 2%", new[~new.low_occ], dev_test[~dev_test.low_occ])]
    for lbl, g, gd in groups:
        if len(g):
            L.append("| %s | %d | %.3f%% | %.1f%% | %s |" % (lbl, len(g), g.corr_err.abs().median(), cov(g.corr_err),
                                                       "%.1f%% (n=%d)" % (cov(gd.corr_err), len(gd)) if len(gd) else "—"))
    new["sv_decile"] = pd.qcut(new.sv, 10, labels=False, duplicates="drop") + 1
    L += ["", "| A/V^(2/3) decile (second shard) | n | mean corrected error | coverage |", "|---|---|---|---|"]
    for q, g in new.groupby("sv_decile"):
        L.append("| %d | %d | %+.3f%% | %.1f%% |" % (q, len(g), g.corr_err.mean(), cov(g.corr_err)))
    worst = new.reindex(new.corr_err.abs().sort_values(ascending=False).index).head(int(max(1, 0.05 * len(new))))
    L += ["", "The worst 5%% of corrected errors (n=%d): median sphericity %.3f vs %.3f overall; median"
          % (len(worst), worst.mesh_sphericity.median(), new.mesh_sphericity.median()),
          "occ_fraction_inside %.3f vs %.3f; non-watertight %.1f%% vs %.1f%%."
          % (worst.occ_fraction_inside.median(), new.occ_fraction_inside.median(),
             100 * (~worst.watertight).mean(), 100 * (~new.watertight).mean()), ""]
    new[["instance_id", "err_pct", "corr_err", "sv", "t_norm", "mesh_sphericity", "occ_fraction_inside",
         "watertight", "sv_decile"]].to_csv(os.path.join(a.outdir, "shard_shift_instances.csv"), index=False)

    # 5. pc_extent_outlier definitions
    od, on = outlier_defs(dev), outlier_defs(new)
    L += ["## 5. `pc_extent_outlier` under alternative definitions", "",
          "| Definition | development | second shard |", "|---|---|---|"]
    for k in od:
        L.append("| %s | %d | %s |" % (k, od[k], on.get(k, "—")))
    L += ["", "Recommendation: keep the current definition (largest single-axis deviation of the raw",
          "point-cloud extent from the mesh extent > 10%). It flags the most instances, which suits an",
          "advisory flag, and it is what the released code does; analyses use the trimmed extent either",
          "way. The earlier count of 1 came from a prior implementation; report the current definition",
          "and its count.", ""]

    with open(os.path.join(a.outdir, "shard_shift_report.md"), "w") as fh:
        fh.write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
run('shard_shift.py', '--dev-paired', SRC / 'paired_feature_table.csv',
    '--dev-pred', SRC / 'validation_v3_predictions.csv', '--model', SRC / 'validation_v3_model.json',
    '--new-audit', FROZEN / 'audit.csv', '--new-pred', FROZEN / 'frozen_predictions.csv', '--outdir', OUT)
show(OUT / 'shard_shift_report.md')

### 20.4 MitoEM-R v2: all eligible instances + resolution sensitivity

Framed as a **pipeline-portability test**: every representation comes from the same annotation, so this is not independent validation. The analysis settings are the same as in Section 17 apart from `--n-mesh 0` (all eligible instances) and the resampling step.

In [ ]:
%%writefile /content/hit_s20/mitoem_external_validation_v2.py
#!/usr/bin/env python3
"""MitoEM-R external-domain morphometry audit from the original ZIP archives (v2).

v2 adds: --n-mesh 0 meshes every eligible instance, and a resolution-sensitivity
analysis that re-grids each mask in-plane (8 nm -> 16 nm and 24 nm, the grid 3DMSL
meshes were built on) and reports how volume, surface and sphericity change.


The labeled validation section (z=400..499) is read without extracting either
archive. Voxel descriptors are calculated for all complete instances, then a
size-balanced sample is converted to meshes and surface point clouds. These
representations all derive from the same annotation, so the experiment measures
pipeline portability and representation sensitivity, not independent biological
ground truth or the specific 3DMSL occupancy-generation error.
"""

import argparse
import json
import re
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from PIL import Image


SPACING_UM = (0.030, 0.008, 0.008)  # z, y, x; MitoEM-R native resolution
VOXEL_UM3 = float(np.prod(SPACING_UM))
IMAGE_RE = re.compile(r"(?:^|/)im(\d{4})\.png$")
LABEL_RE = re.compile(r"(?:^|/)seg(\d{4})\.tif$")


def catalog(archive, pattern):
    found = {}
    for name in archive.namelist():
        if name.startswith("__MACOSX/") or "/._" in name:
            continue
        match = pattern.search(name)
        if match:
            index = int(match.group(1))
            if index in found:
                raise ValueError(f"Duplicate slice {index} in {archive.filename}")
            found[index] = name
    return found


def read_slice(archive, name):
    with archive.open(name) as stream:
        image = Image.open(stream)
        image.load()
        return np.asarray(image).copy()


def pixel_bounds(labels, active_ids):
    """Return per-label XY bounds; scipy is fast, NumPy remains a fallback."""
    try:
        from scipy import ndimage as ndi

        objects = ndi.find_objects(labels)
        for instance_id in active_ids:
            box = objects[instance_id - 1]
            if box is not None:
                yield int(instance_id), box[0].start, box[0].stop, box[1].start, box[1].stop
    except ImportError:
        y, x = np.nonzero(labels)
        ids = labels[y, x]
        size = int(ids.max()) + 1
        min_y = np.full(size, labels.shape[0], dtype=np.int32)
        min_x = np.full(size, labels.shape[1], dtype=np.int32)
        max_y = np.zeros(size, dtype=np.int32)
        max_x = np.zeros(size, dtype=np.int32)
        np.minimum.at(min_y, ids, y)
        np.minimum.at(min_x, ids, x)
        np.maximum.at(max_y, ids, y + 1)
        np.maximum.at(max_x, ids, x + 1)
        for instance_id in active_ids:
            yield int(instance_id), int(min_y[instance_id]), int(max_y[instance_id]), int(min_x[instance_id]), int(max_x[instance_id])


def audit_pair(image_zip, label_zip, image_names, label_names, indices, outdir):
    samples = sorted(set((indices[0], indices[len(indices) // 2], indices[-1])))
    audit = []
    for z in samples:
        image = read_slice(image_zip, image_names[z])
        label = read_slice(label_zip, label_names[z])
        if image.shape != label.shape:
            raise ValueError(f"Image/label shape mismatch at z={z}: {image.shape} vs {label.shape}")
        if image.shape != (4096, 4096):
            raise ValueError(f"Unexpected slice shape at z={z}: {image.shape}")
        if not np.issubdtype(label.dtype, np.integer):
            raise ValueError(f"Labels are not integer instance IDs at z={z}: {label.dtype}")
        values = np.unique(label)
        audit.append({"z": z, "shape": list(label.shape), "image_dtype": str(image.dtype),
                      "label_dtype": str(label.dtype), "nonzero_labels": int(np.count_nonzero(values)),
                      "labeled_fraction": float(np.mean(label > 0))})
        if z == samples[len(samples) // 2]:
            scale = 8
            gray = image[::scale, ::scale]
            binary = label[::scale, ::scale] > 0
            rgb = np.repeat(gray[..., None], 3, axis=2)
            rgb[binary, 0] = np.maximum(rgb[binary, 0], 210)
            rgb[binary, 1] = (rgb[binary, 1] // 2)
            rgb[binary, 2] = (rgb[binary, 2] // 2)
            Image.fromarray(rgb.astype(np.uint8)).save(outdir / f"mitoem_R_overlay_z{z:04d}.png")
    return audit


def scan_instances(label_zip, label_names, indices, min_voxels, min_z_planes, max_roi_voxels):
    records = {}
    for n, z in enumerate(indices, 1):
        label = read_slice(label_zip, label_names[z])
        counts = np.bincount(label.ravel())
        active = np.flatnonzero(counts[1:]) + 1
        for instance_id, y0, y1, x0, x1 in pixel_bounds(label, active):
            count = int(counts[instance_id])
            if instance_id not in records:
                records[instance_id] = dict(instance_id=instance_id, voxel_count=0,
                                            z0=z, z1=z + 1, y0=y0, y1=y1, x0=x0, x1=x1)
            row = records[instance_id]
            row["voxel_count"] += count
            row["z0"] = min(row["z0"], z)
            row["z1"] = max(row["z1"], z + 1)
            row["y0"] = min(row["y0"], y0)
            row["y1"] = max(row["y1"], y1)
            row["x0"] = min(row["x0"], x0)
            row["x1"] = max(row["x1"], x1)
        if n % 10 == 0 or n == len(indices):
            print(f"Scanned label slices: {n}/{len(indices)}", flush=True)
    rows = pd.DataFrame.from_records(list(records.values())).sort_values("instance_id")
    if rows.empty:
        raise ValueError("No labeled mitochondria found in the requested slice range")
    rows["volume_um3"] = rows.voxel_count * VOXEL_UM3
    rows["z_extent_um"] = (rows.z1 - rows.z0) * SPACING_UM[0]
    rows["y_extent_um"] = (rows.y1 - rows.y0) * SPACING_UM[1]
    rows["x_extent_um"] = (rows.x1 - rows.x0) * SPACING_UM[2]
    rows["bbox_fill"] = rows.volume_um3 / (rows.z_extent_um * rows.y_extent_um * rows.x_extent_um)
    rows["touch_z_boundary"] = (rows.z0 == indices[0]) | (rows.z1 == indices[-1] + 1)
    rows["touch_xy_boundary"] = (rows.y0 == 0) | (rows.x0 == 0) | (rows.y1 == 4096) | (rows.x1 == 4096)
    rows["roi_voxels"] = (rows.z1 - rows.z0 + 2) * (rows.y1 - rows.y0 + 2) * (rows.x1 - rows.x0 + 2)
    rows["complete_for_morphometry"] = (
        ~rows.touch_z_boundary & ~rows.touch_xy_boundary &
        (rows.voxel_count >= min_voxels) &
        (rows.z1 - rows.z0 >= min_z_planes) &
        (rows.roi_voxels <= max_roi_voxels)
    )
    return rows


def balanced_sample(rows, n, seed):
    eligible = rows[rows.complete_for_morphometry].copy()
    if eligible.empty:
        return eligible.iloc[:0]
    if n <= 0 or len(eligible) <= n:          # n <= 0 means every eligible instance
        return eligible
    eligible["size_quartile"] = pd.qcut(eligible.voxel_count, 4, labels=False, duplicates="drop")
    rng = np.random.default_rng(seed)
    pools = {int(q): rng.permutation(group.index.to_numpy()).tolist()
             for q, group in eligible.groupby("size_quartile")}
    chosen = []
    while len(chosen) < n and any(pools.values()):
        for q in sorted(pools):
            if pools[q] and len(chosen) < n:
                chosen.append(pools[q].pop())
    return eligible.loc[chosen].sort_values("instance_id")


def build_masks(label_zip, label_names, selected):
    objects = {}
    for row in selected.itertuples():
        shape = (row.z1 - row.z0 + 2, row.y1 - row.y0 + 2, row.x1 - row.x0 + 2)
        objects[int(row.instance_id)] = (row, np.zeros(shape, dtype=bool))
    if not objects:
        return objects
    first = min(row.z0 for row, _ in objects.values())
    last = max(row.z1 for row, _ in objects.values())
    for z in range(first, last):
        current = [(row, mask) for row, mask in objects.values() if row.z0 <= z < row.z1]
        if not current:
            continue
        label = read_slice(label_zip, label_names[z])
        for row, mask in current:
            mask[z - row.z0 + 1, 1:-1, 1:-1] = (label[row.y0:row.y1, row.x0:row.x1] == row.instance_id)
    return objects


def voxel_surface_um2(mask):
    z_area = SPACING_UM[1] * SPACING_UM[2]
    y_area = SPACING_UM[0] * SPACING_UM[2]
    x_area = SPACING_UM[0] * SPACING_UM[1]
    return float(np.count_nonzero(mask[1:] != mask[:-1]) * z_area +
                 np.count_nonzero(mask[:, 1:] != mask[:, :-1]) * y_area +
                 np.count_nonzero(mask[:, :, 1:] != mask[:, :, :-1]) * x_area)


def downsample_xy(mask, factor):
    """Majority-vote block downsampling in y and x (z untouched); keeps a 1-voxel pad."""
    core = mask[:, 1:-1, 1:-1]
    z, y, x = core.shape
    y2, x2 = -(-y // factor) * factor, -(-x // factor) * factor
    core = np.pad(core, ((0, 0), (0, y2 - y), (0, x2 - x)))
    small = core.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4)) >= 0.5
    return np.pad(small, ((0, 0), (1, 1), (1, 1)))


def mesh_measures(mask, spacing):
    from skimage.measure import marching_cubes
    import trimesh
    if mask.sum() == 0:
        raise ValueError("empty mask")
    v, f, _, _ = marching_cubes(mask.astype(np.uint8), level=0.5, spacing=spacing)
    m = trimesh.Trimesh(vertices=v, faces=f, process=False)
    vol, area = float(abs(m.volume)), float(m.area)
    sph = (np.pi ** (1 / 3)) * ((6 * vol) ** (2 / 3)) / area if area > 0 and vol > 0 else np.nan
    return dict(volume=vol, surface=area, sphericity=sph, watertight=bool(m.is_watertight),
                voxel_volume=float(mask.sum()) * float(np.prod(spacing)))


def point_extent_error(vertices, faces, mesh_extent, rng, count, repeats, trim_pct=1.0):
    triangles = vertices[faces]
    cross = np.cross(triangles[:, 1] - triangles[:, 0], triangles[:, 2] - triangles[:, 0])
    area = np.linalg.norm(cross, axis=1)
    valid = area > 0
    if not np.any(valid):
        return np.nan, np.nan
    triangles = triangles[valid]
    probabilities = area[valid] / area[valid].sum()
    raw, trimmed = [], []
    for _ in range(repeats):
        t = triangles[rng.choice(len(triangles), size=count, p=probabilities)]
        u = np.sqrt(rng.random(count))[:, None]
        v = rng.random(count)[:, None]
        points = (1 - u) * t[:, 0] + u * (1 - v) * t[:, 1] + u * v * t[:, 2]
        raw_extent = np.ptp(points, axis=0)
        # same percentile trim as hit_common.trimmed_extent (1st-99th by default)
        low, high = np.quantile(points, [trim_pct / 100.0, 1.0 - trim_pct / 100.0], axis=0)
        raw.append(np.mean(100 * (raw_extent / mesh_extent - 1)))
        trimmed.append(np.mean(100 * ((high - low) / mesh_extent - 1)))
    return float(np.mean(raw)), float(np.mean(trimmed))


def mesh_comparison(objects, seed, point_count, point_repeats, trim_pct=1.0, factors=(2, 3)):
    if not objects:
        return pd.DataFrame()
    try:
        from skimage.measure import marching_cubes
        import trimesh
    except ImportError as exc:
        raise RuntimeError("Mesh comparison needs scikit-image and trimesh; run the notebook's dependency cell") from exc
    results = []
    rng = np.random.default_rng(seed)
    for index, (instance_id, (row, mask)) in enumerate(objects.items(), 1):
        record = {"instance_id": instance_id, "voxel_count": int(row.voxel_count),
                  "voxel_volume_um3": float(row.volume_um3),
                  "voxel_surface_um2": voxel_surface_um2(mask)}
        try:
            vertices, faces, _, _ = marching_cubes(mask.astype(np.uint8), level=0.5, spacing=SPACING_UM)
            mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
            record["mesh_watertight"] = bool(mesh.is_watertight)
            record["mesh_faces"] = int(len(faces))
            if not mesh.is_watertight:
                raise ValueError("Mesh was not watertight")
            record["mesh_volume_um3"] = float(abs(mesh.volume))
            record["mesh_surface_um2"] = float(mesh.area)
            record["mesh_vs_voxel_volume_pct"] = 100 * (record["mesh_volume_um3"] / row.volume_um3 - 1)
            record["mesh_vs_voxel_surface_pct"] = 100 * (record["mesh_surface_um2"] / record["voxel_surface_um2"] - 1)
            extent = np.ptp(vertices, axis=0)
            raw, trimmed = point_extent_error(vertices, faces, extent, rng, point_count, point_repeats, trim_pct)
            record["point_raw_extent_error_pct"] = raw
            record["point_trimmed_extent_error_pct"] = trimmed
            vol, area = record["mesh_volume_um3"], record["mesh_surface_um2"]
            record["mesh_sphericity"] = (np.pi ** (1 / 3)) * ((6 * vol) ** (2 / 3)) / area
            for fct in factors:
                sp = (SPACING_UM[0], SPACING_UM[1] * fct, SPACING_UM[2] * fct)
                try:
                    mm = mesh_measures(downsample_xy(mask, fct), sp)
                    tag = "xy%dnm" % round(1000 * SPACING_UM[1] * fct)
                    record[tag + "_mesh_volume_um3"] = mm["volume"]
                    record[tag + "_mesh_surface_um2"] = mm["surface"]
                    record[tag + "_mesh_sphericity"] = mm["sphericity"]
                    record[tag + "_watertight"] = mm["watertight"]
                    record[tag + "_volume_change_pct"] = 100 * (mm["volume"] / vol - 1)
                    record[tag + "_surface_change_pct"] = 100 * (mm["surface"] / area - 1)
                    record[tag + "_sphericity_change_pct"] = 100 * (mm["sphericity"] / record["mesh_sphericity"] - 1)
                except Exception as exc:
                    record["xy%dnm_error" % round(1000 * SPACING_UM[1] * fct)] = str(exc)
            record["error"] = ""
        except Exception as exc:
            record["error"] = f"{type(exc).__name__}: {exc}"
        results.append(record)
        if index % 25 == 0 or index == len(objects):
            print(f"Meshed instances: {index}/{len(objects)}", flush=True)
    return pd.DataFrame(results)


def metric_line(values):
    x = pd.Series(values, dtype=float).dropna().to_numpy()
    if not len(x):
        return "unavailable"
    return f"median {np.median(x):+.2f}%; 5th–95th percentile {np.percentile(x, 5):+.2f}% to {np.percentile(x, 95):+.2f}%"


def make_report(audit, rows, selected, agreement, args, outdir):
    success = agreement[agreement.error.eq("")] if not agreement.empty else agreement
    phase = "validation" if args.start >= 400 else "training-slice pilot"
    lines = [f"# MitoEM-R {phase} morphometry audit", "",
             f"Analyzed labeled slices **{args.start}–{args.stop - 1}** at 30 × 8 × 8 nm voxel spacing (z × y × x).",
             "Image and label ZIPs were read directly; no full-volume extraction was needed.", "",
             "## Image–label audit", "",
             "| Slice | Shape | Image type | Label type | Instance IDs | Labeled fraction |",
             "|---:|---:|---|---|---:|---:|"]
    for row in audit:
        lines.append(f"| {row['z']} | {row['shape'][0]} × {row['shape'][1]} | {row['image_dtype']} | "
                     f"{row['label_dtype']} | {row['nonzero_labels']} | {row['labeled_fraction']:.3f} |")
    lines += ["", "## Instance selection", "",
              f"- Unique instance IDs intersecting this slab: **{len(rows)}**.",
              f"- Complete instances passing ≥{args.min_voxels:,} voxels, ≥{args.min_z_planes} z planes, "
              f"no slab/image-edge contact, and ROI ≤{args.max_roi_voxels:,} voxels: **{int(rows.complete_for_morphometry.sum())}**.",
              f"- Size-balanced sample requested: **{args.n_mesh if args.n_mesh > 0 else 'all eligible'}**; selected: **{len(selected)}**; successful mesh comparisons: **{len(success)}**.", ""]
    if len(success):
        lines += ["## Representation agreement in the selected instances", "",
                  f"- Mesh versus source voxel volume: {metric_line(success.mesh_vs_voxel_volume_pct)}.",
                  f"- Mesh versus voxel-face surface area: {metric_line(success.mesh_vs_voxel_surface_pct)}.",
                  f"- Point-cloud raw extent versus mesh extent: {metric_line(success.point_raw_extent_error_pct)}.",
                  f"- Point-cloud {args.trim_pct:g}–{100 - args.trim_pct:g}% trimmed extent versus mesh extent (same trim as the 3DMSL pipeline): {metric_line(success.point_trimmed_extent_error_pct)}.", "",
                  "| Candidate absolute volume margin | Fraction of successful mesh comparisons within margin |",
                  "|---:|---:|"]
        for margin in sorted(set((round(args.rc_margin, 2), 5.0, 6.0, 10.0))):
            fraction = np.mean(np.abs(success.mesh_vs_voxel_volume_pct) <= margin)
            lines.append(f"| {margin:.2f}% | {fraction:.1%} |")
    if len(success):
        tags = sorted({c[:-len("_volume_change_pct")] for c in success.columns if c.endswith("_volume_change_pct")},
                      key=lambda t: int(t[2:-2]))
        if tags:
            def spearman(a, b):
                d = pd.concat([a, b], axis=1).dropna()
                return d.iloc[:, 0].rank().corr(d.iloc[:, 1].rank()) if len(d) > 2 else np.nan
            lines += ["", "## Resolution sensitivity (real EM masks re-gridded in-plane)", "",
                      "Each mask was block-downsampled in y and x (majority vote) and re-meshed. 24 nm matches the reported in-plane spacing",
                      "of 3DMSL; z remains 30 nm and these shifts are specific to this voting and meshing procedure.", "",
                      "| In-plane spacing | volume change (median, 5–95%) | surface change | sphericity change | rank stability of sphericity (Spearman) | watertight |",
                      "|---|---|---|---|---|---|"]
            for t in tags:
                lines.append("| %s nm | %s | %s | %s | %.3f | %.1f%% |" % (
                    t[2:-2], metric_line(success[t + "_volume_change_pct"]), metric_line(success[t + "_surface_change_pct"]),
                    metric_line(success[t + "_sphericity_change_pct"]),
                    spearman(success["mesh_sphericity"], success[t + "_mesh_sphericity"]),
                    100 * success[t + "_watertight"].astype(bool).mean()))
            lines += ["", "Volume rank stability (Spearman, native vs coarsest): %.4f." % spearman(
                success["mesh_volume_um3"], success[tags[-1] + "_mesh_volume_um3"]), ""]
    lines += ["", "## Interpretation", "",
              "**Framing: this is a pipeline-portability test, not independent validation.** "
              "The voxel mask, mesh, and sampled surface points all derive from the same MitoEM annotation, "
              "so their agreement measures how much the processing pipeline itself changes the descriptors on real EM data. "
              "It does not test independent acquisition agreement or reproduce the 3DMSL occupancy query process.", "",
              "Candidate margins above are exploratory sensitivity checks. Choose and freeze a primary margin "
              f"with the adviser before a confirmatory external analysis. The {args.rc_margin:.2f}% value originates in the "
              "3DMSL sampling repeatability calculation and is not a biological threshold.", "",
              "## Known method artifacts", "",
              "- **Mesh vs voxel volume**: both come from the same mask, so any difference comes from the meshing "
              "step. Marching cubes at level 0.5 on a binary mask cuts corners, which shrinks small and thin objects "
              "most, and the coarse 30 nm z spacing (few z planes per mitochondrion) makes this worse. A negative "
              "volume difference is therefore expected from the algorithm, not evidence of disagreement between "
              "independent measurements.",
              "- **Mesh vs voxel-face surface** is dominated by the staircase effect: counting exposed voxel faces "
              "overstates the true surface area, so the mesh surface is expected to be smaller.",
              "- **Point-cloud extents** here are sampled without noise from the mesh surface, so raw extents can "
              "only be smaller than the mesh extent. This differs from the 3DMSL point clouds, which include "
              "sampling-noise outliers.", "",
              "This is one rat cortex tissue volume; instances within it are spatially dependent. "
              "No mitochondrial function, cardiovascular outcome, or independent animal replicate is present. "
              "Annotation errors and the challenge's minimum annotated size can alter the observed size distribution.", ""]
    (outdir / "mitoem_R_report.md").write_text("\n".join(lines))
    print("\n".join(lines))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image-zip", type=Path, required=True)
    parser.add_argument("--label-zip", type=Path, required=True)
    parser.add_argument("--outdir", type=Path, required=True)
    parser.add_argument("--start", type=int, default=400)
    parser.add_argument("--stop", type=int, default=500, help="exclusive z index")
    parser.add_argument("--n-mesh", type=int, default=0, help="0 = every eligible instance")
    parser.add_argument("--min-voxels", type=int, default=2000)
    parser.add_argument("--min-z-planes", type=int, default=3)
    parser.add_argument("--max-roi-voxels", type=int, default=2_000_000)
    parser.add_argument("--point-count", type=int, default=10_000)
    parser.add_argument("--point-repeats", type=int, default=5)
    parser.add_argument("--seed", type=int, default=2026)
    parser.add_argument("--resample-xy", type=int, nargs="*", default=[2, 3],
                        help="in-plane downsampling factors for the resolution-sensitivity analysis")
    parser.add_argument("--trim-pct", type=float, default=1.0,
                        help="percentile trim on each side for point-cloud extents (3DMSL uses 1)")
    parser.add_argument("--rc-margin", type=float, default=2.17,
                        help="3DMSL repeatability coefficient, in percent, shown as a sensitivity margin")
    args = parser.parse_args()
    if not (0 <= args.start < args.stop <= 500):
        raise ValueError("Labeled slices must be inside 0–499")
    args.outdir.mkdir(parents=True, exist_ok=True)
    with ZipFile(args.image_zip) as image_zip, ZipFile(args.label_zip) as label_zip:
        image_names = catalog(image_zip, IMAGE_RE)
        label_names = catalog(label_zip, LABEL_RE)
        if set(image_names) != set(range(1000)):
            raise ValueError(f"Expected image slices 0–999; found {len(image_names)}")
        if set(label_names) != set(range(500)):
            raise ValueError(f"Expected label slices 0–499; found {len(label_names)}")
        # Check the train/val split by folder name without depending on the exact
        # spelling (mito-train-v2, mito_train_v2, ...). Print the folders so they can be
        # confirmed by eye.
        def folder(z):
            return str(Path(label_names[z]).parent).lower()
        train_dirs = sorted({folder(z) for z in range(400)})
        val_dirs = sorted({folder(z) for z in range(400, 500)})
        print("Training label folder(s):", train_dirs)
        print("Validation label folder(s):", val_dirs)
        if not all("train" in Path(d).name and "val" not in Path(d).name for d in train_dirs):
            raise ValueError(f"Slices 0-399 are not all in a training folder: {train_dirs}")
        if not all("val" in Path(d).name and "train" not in Path(d).name for d in val_dirs):
            raise ValueError(f"Slices 400-499 are not all in a validation folder: {val_dirs}")
        if set(train_dirs) & set(val_dirs):
            raise ValueError("Training and validation slices share a folder")
        indices = list(range(args.start, args.stop))
        audit = audit_pair(image_zip, label_zip, image_names, label_names, indices, args.outdir)
        rows = scan_instances(label_zip, label_names, indices, args.min_voxels,
                              args.min_z_planes, args.max_roi_voxels)
        rows.to_csv(args.outdir / "mitoem_R_instance_features.csv", index=False)
        selected = balanced_sample(rows, args.n_mesh, args.seed)
        if len(selected):
            objects = build_masks(label_zip, label_names, selected)
            agreement = mesh_comparison(objects, args.seed, args.point_count, args.point_repeats, args.trim_pct,
                                        tuple(args.resample_xy))
        else:
            agreement = pd.DataFrame(columns=["instance_id", "error"])
        agreement.to_csv(args.outdir / "mitoem_R_representation_agreement.csv", index=False)
        metadata = {"image_zip": str(args.image_zip), "label_zip": str(args.label_zip),
                    "image_slice_count": len(image_names), "label_slice_count": len(label_names),
                    "analysis_z_start": args.start, "analysis_z_stop_exclusive": args.stop,
                    "voxel_spacing_um_zyx": SPACING_UM, "seed": args.seed,
                    "n_instances_intersecting_slab": len(rows),
                    "n_complete_eligible": int(rows.complete_for_morphometry.sum()),
                    "n_mesh_selected": len(selected), "audit_samples": audit,
                    "settings": {key: value for key, value in vars(args).items() if key not in ("image_zip", "label_zip", "outdir")}}
        (args.outdir / "mitoem_R_audit.json").write_text(json.dumps(metadata, indent=2))
        make_report(audit, rows, selected, agreement, args, args.outdir)


if __name__ == "__main__":
    main()

In [ ]:
import json
MITO_LABEL_ZIP = WORK / MITO_LABEL_ZIP_DRIVE.name
if not MITO_LABEL_ZIP.exists() or MITO_LABEL_ZIP.stat().st_size != MITO_LABEL_ZIP_DRIVE.stat().st_size:
    shutil.copy2(MITO_LABEL_ZIP_DRIVE, MITO_LABEL_ZIP)
rc = json.loads((SRC / 'validation_v3_model.json').read_text()).get('repeatability_margin_pct', 2.17)
MITO_OUT = OUT / 'mitoem_R_v2_validation_400_499'
run('mitoem_external_validation_v2.py', '--image-zip', MITO_IMAGE_ZIP, '--label-zip', MITO_LABEL_ZIP,
    '--outdir', MITO_OUT, '--start', 400, '--stop', 500, '--n-mesh', 0, '--seed', 2026,
    '--rc-margin', rc, '--resample-xy', 2, 3)
show(MITO_OUT / 'mitoem_R_report.md')

### 20.5 Save the exact scripts with the results

In [ ]:
(OUT / 'scripts').mkdir(exist_ok=True)
for f in ('hit_common.py', 'mechanism_offset.py', 'imaging_v2.py', 'shard_shift.py', 'mitoem_external_validation_v2.py'):
    shutil.copy2(WORK / f, OUT / 'scripts' / f)
(OUT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print('All Section 20 results and scripts are in', OUT)

## 21. Equivalence-margin decision memo (for adviser sign-off)

**Decision needed:** which error margin is the primary, pre-specified criterion for the occupancy volume correction, and how to report the others.

**What was actually pre-specified.** The calibrated 95% absolute-error bound for the occupancy-only correction (**2.661%**) was fitted on the development calibration split and frozen in `validation_v3_model.json`. Its hash was recorded in `protocol.json` before any second-shard outcome was computed. It is the only margin that was fixed before being tested on new data.

| Margin | Origin | Status to report |
|---|---|---|
| 2.661% calibrated q95 bound | development calibration split, frozen before shard 2 | **proposed primary.** Second-shard coverage 92.1% (95% CI 91.1–93.0%) vs 95% nominal; report as mild under-coverage under distribution shift |
| 2.17% repeatability coefficient | Monte-Carlo precision of the estimator | secondary benchmark: the best achievable agreement given sampling noise |
| 6% / 15% (by `occ_fraction_inside`) | Week 1–2 observed error, chosen after seeing the data | exploratory only |

**Options for the adviser:**
1. Accept the frozen 2.661% bound as primary and report the 92.1% coverage honestly (recommended; nothing is chosen after the fact).
2. Choose a margin on scientific grounds instead, for example the smallest volume change that matters biologically. Because shards 1 and 2 have now been seen, test that margin on a **third, untouched 3DMSL shard** (eight remain) with the model frozen as it is now.

Whichever option is chosen, write it down with a date before running anything further, and cite the 3DMSL data paper for the dataset description.